# Deney 02 — Script / Ton Üretimi (Notebook 01 çıktısına bağlı)

**Amaç:** Notebook 01'in ürettiği gerçek `hot_moments.json` dosyasından bir an seçip, insan onayından sonra script üretimi için aday LLM'leri **kalite + maliyet + hız** üzerinden karşılaştırmak.

**Çalıştırma sırası:** Notebook 01 → (insan/LLM an onayı) → **bu notebook** → Notebook 03 → Notebook 06. Alternatif olarak `00_youtube_automation_master.ipynb` tek başına tüm fazları çalıştırabilir.

**Tek kaynak politikası:** Prompt oluşturma, LLM yanıtı JSON ayrıştırma, sağlayıcı çağrıları ve `selected_script.json` şeması `local_pipeline/script_generation.py` içindeki **kanonik** fonksiyonlardan gelir; master notebook da aynılarını çağırır. Bu notebook'un kendine özgü katkısı **çok-model karşılaştırma döngüsüdür**.

API anahtarlarını Colab 🔑 Secrets panelinden ekle: `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`. Anahtarı olmayan sağlayıcı atlanır. Hiçbir anahtar yoksa notebook yine de tam bir **prompt paketi** hazırlayıp dışa aktarır — sahte script üretmiş gibi davranmaz.


## Google model router (bu notebook'taki tum Gemini cagrilari icin tek merkez)

Bu notebook artik `genai.Client(...)` kurmuyor ve sabit `model='gemini-...'` yazmiyor.
Her Google cagrisi `local_pipeline/google_ai.py` uzerinden gider:

- **Colab Secret adlari:** `GeminiKey` (eski ad `GOOGLE_API_KEY` hala calisir),
  `YoutubeKEY` (eski ad `YOUTUBE_DATA_API_KEY`). Deger hicbir zaman yazdirilmaz;
  yalnizca `configured (GeminiKey)` / `missing` durumu gosterilir.
- **Tek anahtar, cok model:** ayni `GeminiKey` projenin yetkili oldugu tum modellere erisir.
- **Kota PROJE bazlidir, anahtar bazli DEGILDIR** (ai.google.dev/gemini-api/docs/rate-limits);
  ikinci bir anahtar limit artirmaz. Hesap/proje dondurerek kota asmak yasaktir
  (`ALLOW_ACCOUNT_QUOTA_ROTATION = False`).
- **Rota:** ilk basarili cagri rotayi bitirir; hicbir zaman tum katalog denenmez.
- **Ucretsiz katman veri politikasi:** gonderilen icerik Google tarafindan urun
  gelistirmede kullanilabilir ve insan gozden gecirenler okuyabilir. Ham telifli
  video/ses varsayilan olarak YUKLENMEZ; yalnizca yerel olarak turetilmis
  transcript/skor ozetleri gonderilir.
- **Kota defteri** (`results/google_quota_usage.json`) YEREL TAHMINDIR; sunucunun
  yetkili kotasi degildir. Gunluk sayaclar Pasifik gece yarisinda sifirlanir.

Bu notebook'ta kullanilan gorev: **script uretimi** -> Rota A (`gemini-3.6-flash` -> `gemini-3.5-flash` -> `gemini-3.5-flash-lite` -> `gemma-4-31b-it`). Karsilastirma hucreleri de ayni router'i kullanir; her model tek tek, acikca cagrilir.


In [ ]:
!pip -q install openai anthropic google-genai pandas > /dev/null
print('Kurulum tamam.')

In [ ]:
import os, sys, time, json, glob, hashlib, zipfile
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            import site
            site.addsitedir(str(venv_site_packages))
            break

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RESULTS_DIR = Path('/content/results') if IS_COLAB else PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def get_secret(name):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
ANTHROPIC_API_KEY = get_secret('ANTHROPIC_API_KEY')
GOOGLE_API_KEY = get_secret('GeminiKey') or get_secret('GOOGLE_API_KEY')  # Colab Secret adi: GeminiKey (eski ad: GOOGLE_API_KEY)
print('OpenAI:', 'var' if OPENAI_API_KEY else 'yok')
print('Anthropic:', 'var' if ANTHROPIC_API_KEY else 'yok')
print('Google:', 'var' if GOOGLE_API_KEY else 'yok')

# Paylasilan durum sozlugu (Faz 6) - tum notebooklarda ayni degerler kullanilir.
STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}

results = []
def log_result(task, candidate, quality_score, cost_usd, latency_sec, notes='', status='experimental'):
    assert status in STATUS_VALUES, f'Bilinmeyen status: {status}'
    results.append({
        'task': task, 'candidate': candidate, 'quality_score_1_5': quality_score,
        'cost_usd_estimate': round(cost_usd, 5), 'latency_sec': round(latency_sec, 2),
        'notes': notes, 'status': status,
    })
    print(f"[{task}] {candidate} -> latency={latency_sec:.2f}s cost=${cost_usd:.5f} status={status}")

## `local_pipeline` kanonik modüllerini yükle

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['script_generation', 'video_analysis', 'approval', 'google_ai']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'script_generation': {
        'sha256': '4a71343c730bf7563993694455a1890d2105b3984ffc2c4a0561a5c6cf728822',
        'encoding': 'zlib+b85',
        'data': (
            'c-q}PYjfK+w%_$Du<8d%U5Rnprgz3!?Tj{2Q{Osv)>gXH@p@!Pge<NpQp<-O)z|<1o(BK|BxO6<?&IFf#Fj|l05~{roC9~e-7{6@'
            'auF+WeEeSI%PiU!qidC@Ocqg^h&a8DmLgxvOod{x69?bSxzW686hzxyvgpVp6bA?MU0N39Ld|7aq#Jra-^jdB88-_$os+asi!{9f'
            'VB$upZ7!lB7wKIhHf3ByqivSnMxn}H@YhWms#q*kvRrRub`vGn;^UF%XJwMW=T5F}VIHwK-Ku1A^kR|}>nz<y%NOs`^cvPh-E)xx'
            'L|g#<#;L}tC=iwu>vAI#SbLhOPO(;-!D|7(1WaSIEyOZS@**pjxRRmx@AK1>QKs@OJcTAQ%K^ra0B%)gr!JN<jyrdm+`>~_Up4<+'
            'z$+0)w@R#4rbM2KMOv&yqOp=Qwndt#PPEy^YNL{ZHzR_XC>e@$CBkwWM@va7SY`4$f|b(T^ptqx5ZQ<x0j*OR0+KVaEf;Z=uNBNA'
            '*%fPO8whT+&7!2pUqH|N1x`E43v47NAC6$3c0s4x?RHjx6)~T$V3RX7p9`2`nibFt=1;TCJ01P(Z+V)S&qcIR+&7ek!f&Rh`97pJ'
            'zv8kx=d;P|<N5sO$??bOxfqN4j(~q%^$DgG5kBJXP;^6>cAReU!F(IbrCO&kARk|4>NZk$^GyA{jDU2zLxuqR0Xu}>MHjGHu2#`9'
            '!tO8&i4Rv2FiEeWVaU4<P-M6R5D%SBCsZr3D5E%>m$}O3OeFnW#cEloaK1@#HisgNmc<Zhr$FdPgNFjh2-eaC+JP~$(JE64NNFjz'
            'dUh@U3P6YLc&)PQC=oz78=&|bQSN~3q6}65^DSg{tzaD`^W;2N%R?ZT0y=(wuou3WU@VjCQeNw^{zsg^xX=@S4aDe=`1Fu5*zNua'
            'SDzD$s84cP2t<V8mK>nLtm!RCl%b$K2Odicjhj3mx<Fw3fN%x;ER&lTd0J*mHP2x+1unrnP)SH%VJD#Tivk+67#ajfln5*@H^5#y'
            'n04Q-cGaCE3|ZXQw&<dVk9(I7u+>icdb0qK+V9n;)f<Y|b~28x6L?lN91I>t&xWs8`4n{13}EMj!)HvNu?Ap!y}?5uW)^#gqP;_f'
            'U)_nv7|>7J(ZHev=uKVk3^%_yJ(*2UX7l68$-9q}chmEscsu=J^6_{!ufOULG@zRm3a@dqE6t@&p(w)kK^%y`T3kT02EdDb9WSZd'
            'S7HWKYX{0om{tVh7~~o#zaNgy&Sxa(fd`Sind7SVpu|+XIuud^1I$D*isEj%rha!>RA4g%Fioye3U`HEfocQ>nU$d0@-@kLt!>02'
            'PM0?@1r3n6E#A@Kpaw|~kN}77tR&YmUX5019H)2rP_P8!_XhkLG>irZ14ug&t@;CcmnjhLiN{CxdY>+O%;<B(4sef0eUls%Q7CUB'
            'k;^p*kn~rO>`ACVZ{D}arw3@3L^~BnH)0WG;+M3Xf#o31*FZ_Ro>3%r^4Gu{W|k&ExF{NdnBS)%d~pRWJUnL+3!*YZ#o~#}p|~!W'
            'aY-X0HpH^r#43+=u-gln$I>6{O>_fu2w^8-m^u;?xgjL<t2axS0X!ObP<rQ+x6?OM($FHL>ECyGeO+MSAP7+7;B1_)DhGv#q~{?G'
            '8vFso`X3)2{+j$X={Yk!uW)|&*W_NyL(96Xy0r18J^^*+eKZzS$l8kJJbti(9=Rb&lM(k3t4JBSuZ6wL!CG0RfYKz*HXxn=Sio#O'
            't6HHBR%l9G;R3IS6|UMBclBDJC9)INj@rs%O#ng3gs50TFz#>;8>C;H%PLNB24F@>!CD`>C1lX#@{XF&Hsc545(0N8VI^QKAV}iN'
            'Kui{-yg+c2iMEMvWeihC39u8AUynv;sS($q`9{E^<Q5ehsW}*l1SAfp5mx}V1A|oh&FDwN=7GLbw=rs2y(J|2m>>o?qi%N)z}sls'
            'XBix=sL!F@euj9#T`mC@ez-XN?%;rBTaoRY#s>M&4rob_d9jb;E(dzjO9iXx|D<-)EJK{nc3Y)C{*0KS&x7`H(3CJ!ubv|<RNww6'
            '4w@_4Hz|N=U%66?`)P8gl^Kr(?G7*o?#Rdi;|#?CTA#$m*2?NyM$mSU<%bUl40jn|kh^6406=0Y?t5BgYWTp$Ad1dtRykW#eHfCr'
            '1E>RC1SHaRS^^z{)eVq)Y=8C;G&ny47z&zZ&>loJ=o2#SfgW;bm4Gf_g)A4D@dczIy1TVv{veqHY7F=jSakzvhW?u<Cs1a}){64z'
            'gv=Pg39L6*fgta9@4JI)gOX6c%lPF3pzinLrELH^ts#uX=p_z|UH&lUr-z<RifdC274^F`Z_t3(Agar_ggugB<P6V53p2oN4amL0'
            '&AnjYFw*6~orw{*$i1k5b$6H)KBzeF!Zf_J9QV{r4HChz1AQjcO%4j}R_!D>cDXZSO`trE%TiPNg&H4(0z9h*B$uqe>R#OY)cSDA'
            '`zm)Ccxe#`U-9E)HaR~6%>n>`?tYXTu_WCU#W&)cg9C9RK?{Ph0ZkB3hw$w3ciBU8g2bww&R1fB-qI&xG4gI-z8@9o4bT-#YpL?r'
            'nuK#h!vz?c1ri+~2v#yZpB=rQ%%<l63K@O5R?8csKS;dya(09f9+Kl0?I@s9(8*bb(v}5q(D|xE#u>nk*((JH83DaHeg9!{c7&*S'
            '1M5FNeaDId9Q~w<GQcn(_oD^ID-=NTVhNx=wgKrY4pWG2w1vjNt(!70#L4vM=@}Sp)Rz?xsDLoVu6YW;R$wZNv@FnJH_WG19?OM1'
            '`UDG$Lc&(=4+Y#Uu2Y3tHA~CuHNd-K?^4SwyX|>@a3x-dtE^0#5g2$k1dflEYnVOYSX+vnuV_+i8{!V637C&Eyp_ok0}KKnl(<rq'
            'I^fO+ozoA~lgZKi{LR_XhuQr7>D%cs`rjA*?)A19y$Ze`At!=$@-jF$7>e)V=U^zRw|R<=`73()?E${M>`YE(Kc1a_IC|3tb{Weu'
            'RHL<wZpzVjqgNL6D+5}$047gVK@Gm81}|SRcv$4S)6;jy(?d1~(5o&YK<*iAiV_nY02(6GXZR=srU%*!#0R2cq9vRvki}90^#DL7'
            'VL-JtG61R|sId9?04O~MxfG%Y0K)a`!}}LA_<#1{t=<US$!m&0&1Dq)xl9Xrte|;zg^t1UCLfaE3BgzgdB7BYb*(m05{<qIUSTBZ'
            'ima5g>HE{4r*Cm}Fc3`y-u(@nHZb>daUlMVu!(D;g6yCOgI2(g0FmvWwFDl^%jHt3P%B-cZGH=Uh8_qAA04q<GM=D&zghup+Wc(#'
            '?&y4W_KS_%0V1v=q-B7+1rh@K%2OR104qDEuvLx%pbVn|jjBP=;r(io0|_D^CdR_IKu@b!f|E3){o$X3VZGI>&uKM^ftBk`fr#{6'
            'KAfHYeDrpDHaF75#-7rxN@N7qcN>eC=#q&&LbE+ouUj%+p;;R$DA2^tC@N{NL&21hU=FYiTa!{H+9agyYimHdB;BUK`UEE7a40=w'
            'v-r*A<n-j|&E(j^r+{Q1JCUTgjU*TpR*<GZEg6Vyz}Q!YSt{;$N0??NiE^e^O{hKm6j5}>NdBN>eDV6M#}+$?gtrP7i-2>aOJZDP'
            'WR`WOvmc>@0jae@76B;|K2{)a7m_?^SWjT~_=m~y@t-DdeyTS6^o$4Yn~{g^UJmW%8z)LHhP=&q4xHCFb0E`?49KF~f;&itAb2RU'
            '(7!INHE9e(L|~sF4Oy?wqr`jyHXPuJ02G>lg^O6NKndK3Gf=d^m3>gLatoSgw2HDE`V!ng{9rIa+sYAPBpb6UIz~Pk5i}MH($&)2'
            '40g8!ivm142_^?{bsjA!x&Z<aI3DoBUk%a;(vrf$)>;LF9dsb-(Pf$!*5(A$RigPj7JImmydg6i>i{8S-L*298rGR-yREtl49RTN'
            'P9;WH=khw1JFq`;?cviXU>J%1Lfyc`6}szrnt`9u-!s9%!=Uo^2d+1wCwKvF7}E|I_Gx^pH0=OdvYk&oL*5NH68l_&BFGf7SD)~B'
            '0K~I}t-%h^6v@!*#WNS8ve0e-%_h*HS!g^ar)8ti!s*rM&ZnK;|L@I&39=Ct<&I0(zRff>@^t{2D61OFLJt^ULThx5c)NWcL%Xur'
            's`EG|R^6Um=R?m$KoWvKc~8*c1^tf1F1-oFdv&`kG5C9-a1JzEXQr36n9)9i)FWF>sx8zz!~?JNZf+#QA-F|py!S|g|H)f0Nzq)G'
            'Tk%d@fad>&!wOA)xNJM9APGNxi^1s8+m9cPLF>(?Z>tah!6igAt#}>L`jF-k`Dh171v0*qJ97xayoRQa4yHHRE)ahKl|+DiY#^{%'
            'Rsxokv?|T6elNqBs4yNdCTP?}R{!tOZ#)AEKj)|rwC<*bgCVK?QI0Sp5{US8TuqxH#;pOgkpvf@kv8}n+glgM;LEDzyQo;BgCy~|'
            'X%$_UXe=&cJa;3NO_n08fd)%bXIF5b1&57fQK-!paf{$E;;lm%OMnKDA{qNJwhD(}!)<AAw}_<{%HklLJqfYF$8i;hbWR;=V#5fC'
            'W_KNFume^+{*^d4lww<>F^`KUyu2noDu_VzX&PyCXH_Pi#_;r+MA%|3^-1Z&3|*4jVlzN#I0#(1gj23@Ynx!YxU6DCEm%v`r=s6?'
            'aQ=kl1q1tA1CAf*Juj5h7^oqP+h@r9!VyS1mswj%2K<}sf;dgCIl?SO7_C-{;tyu5jxlGn>^pppAVQbsveKp$xM{WVFpSn>7Bp6D'
            'gh_2O0Skft-VXam9t=8$e)nxV2}KG*KDm+pP5?~q2Rg2~c^2w^E^!a3qoh>c3+g=Ab~As$mGLLWY12%nw*wDO3mU1_@W~Mj?sA0('
            'TxAjgNnsEl%p0KJi^>wH<QeH$OG9=ZH;Rs}?JH2qPI&s_r60+})&K~H^>Nj}6E+EC81|t{1%hX4R%?gx%vNrr^*;*rIlDmG3h1+x'
            'a>+~Eu7<`A4KG+yvrnA7iOOB_4xZj%pp29XC-sOW_&y`_Jz_;ECYozW6Hpw~M&dEVE%j-O@v!cDc9FfI3DypKsKMrC9>PcTNs{G`'
            '58r!M=MH;p+_HXvo%A6ow8VN37$c&X?h8zmF`+bZ#W=bmYk@UC<1pYw(RySNkt|<OJg1+c+f|^74bu`|&<g1&2~aHZGF!=|-YE*R'
            'U__`)__UbaHk+<gWmRca`$C`5v(<9`_9p@pMwvpL4|^#o9ts8mc%0z+$1&O(%Nt}$$GkVRJO<N~7c{!B$l#&n7@;_T#+cj_;E%o='
            'Z_c;yisu*LCI`+Y2TKemE8pm6zr>Yl8<m66tMc*1eK$)fsM19eL%&@g3&SIa=pHV8+Y{{yjHAM?jSqtV8d_qSGck?tyTxuxaPg>N'
            '&YRVJ7}lHw{HF=YU#}Pe3YzB@EoW0OFrXpDuq4QUKrb!O%W!0ZClEBm$JF6{*LH<<*w1*%j6{rUtDh&pZpa)ni@FYXKtuiQp^_&!'
            'M^<6mfWCApiIk%t?$gsiADYt<LJs8ka(-}dfd2>LYb!X!?@jZH6dvsYqNVff)!H-n`e6?d3(EuS0w4D+yy>`fjle)O>QE8%CL)DL'
            'F0p?7b=v{_;Nf|vLC@aIRTo7O@1Q;rYHs8~aPZ(tcpLHh7YKP@jM$NXBS~UJf{`FCLPLJiL^XbOaPWjL>PVl@mo>kwY~V3aHegc0'
            'f2cq(Twe)<AWF97b2#7W0@U<WpZ_q{_d>@1*SKGw^!yW8e?ND{bRag?aU+>TD=>ZZS3Ms(iSqMjO}u4))~v{5rMB}`GPcoE&%V@0'
            'Av(<CWV??>VsiYK$uH;TJXR-aBxw=>mh|~ymEx4?^Av4G8JjQQczU2AtGVXeeh>tME5bJ)wa5fbtW2{gALPm9GA+@NzeY<0#NP|_'
            '$I{hmC}@L0$jLm9R+RC9Dd}wPVNhknJVU^4WsGz|gp`DoX;(zQf>a<NP7*pW-c_8!oaW|8DX^!0R~-K#fEsBg2DFi(;!G$1pxvIW'
            'gHfw?xtFwvMl2_Nrm%xMq;Lu_TTm_MTN>F{4SE>&3?cNcptU|p(I>p%#?O8{I@f*z20(_!VCM8M&ddkgMSut7`P>VBvE84GJFu76'
            'm}jZOPdO(tNMmiUjAf7Et!0+xxjA<BgHfC_Ym#IwemgD7kYBcFLQV;SHr1glCIfhk>*fSF2AXBap;nP&m%+yk+*X`r&iwMRNmz7-'
            '05Nh=O%qe8Efbp$6(r5<{A<VuMkd~2q7Gw<OR{?AHwN$HmlY(--R8qk=yHX*H;6d^C4tiMqWO$V&;bS5{dt$-@LkNlY^q|@lzFRL'
            '#g(X*;1epYF)S|WJ7^w&zWT1-kb@G~vYzLsHPH~0&jx;9qf4v79?_o8K>u2O+0=`^vN1SxpI#1q?P-o)o3M;3HBk6F_IC*v8q7UH'
            ')(NnEol4gP{HQuM__4kAAgX$A)Gv>xcb4dD&I$NBmD32)6!kNv9OHp%^sBS-R_rlA_YLJQIP3ZXbnG_)xL0T`freHWUk@Ks;7^&<'
            'o=V5e6If0qY@P+8RfO>A5&)$`l_zJ>5HzM}=DVS9MLY`>)lE=65o+#8S!mxhjga@G=SDV<W1msNX-z+TJU+G=P0O{660I0NCUJD5'
            '!18DsabRln>0~BK5)<hjQbf{I*#yR_?ZJ@4N_R58;aBx^99&5~t)}f6rP}Bh8#(%J^zF;VDAK|rFVgLn5>^Z6bc1q~@;ngdq?f_4'
            '0tGuF<5H3lo>k|tXj93hN&e-`HmarI1VeQjr6t=JI?D|jaCYrysOEp3&LuVLP`v&0k1NsErxlDA<=T8SY|*lvM@fC$`n2lwSt`0J'
            '=6Y0!Jzt%-s=LbA(g1Y4_mcc2s{rd3fTIeV%F0oVAf1Q*CQYdE^t8^audSy2Jaids@gT$BQsaHT*4wvaqtPCtn|%c@ldOnVAX7%U'
            'T!AD8=R%R~Z@daE4f-ILumr%wi7}H?yYC<as*1#>3Bi-GhmW;Y*5A6So!@O^iK4u<pLyr@GE`;Ry4JJ#-tBD<?eyFFPS;<%eT~?`'
            'D%tvc{dcXsH2Li5XNb$`Nin1OMir7OdHs&Hmu|nRx`e#%`9gtyRJHzl8a)PwRM8V)xQ&N#mS1fg<&TDpH^&;l`ViHNnuVv6c+8$@'
            'DIhh)_w{#<D%OsVhr$T8p>Q~{QKhP@o}U#=<sNe`*fQnPx~3GzPHDT$3#WQ`t{Y+M{1f3N_ut=33FZjQ8>{^1FXykc(drjIgVrdr'
            '?(-n((~K<;B=vVHhIF#Q^Wb;_b8gIlmO||3fvfCT%Ix_92vk^RtD&CPr@csvt=})0!gI^gJ?ZVFd)3ICD$WULhV|;$D-cJv7!rxz'
            '1j{*X9OSu8RH!m;UYLS)%97DH34kV&nifW6gld8?30j$2Vj529Uf$prY^~&tvgNlrvgn@czL}hyoX&JLuhR4~?wJYnFlreUPK66b'
            '+W;n7RvjCExRtSx*>y=p-r@%k8f$20B9x&)xs%)0tKFf{8){F16Tg#mk%l`;Tf+Mha0)wBU{yuBNonFmDx6uaDd8YzOC1wFbn*mv'
            '-4t+O^B!=trM&ZcOwueue^8a^vb8T7-!xl&$o=DF^v$d9xlmf!HBwC|UHSq2OFHP*m7`&Goq#KgBFZpZMU>rC8Tk!)rS-AA)6Y;y'
            'u?@7RsZ3<N1G9JzzQNXM-Yt`xB)v=Ay8D{6omV!pYhlO)Yl!QY9q{u@SJl+6G-!6x+Na*G{c?hC<LZO%V9*>+UtRFTV5Zd*Lz!m&'
            'Kugx&?eLd%y=@JR*7m-k!KN{ougb@{W4l|;*=gA9tV-R9TH%AkJNnm2>pL^j1JuF-1jeSS>tshbFSpwmD;_|0z1Bf#7(W7*vmtc{'
            '9uLD&{Y0Rm*m3dk8v5rL{pM`%ZZ!g!|0M9P@vs0|b*qFyQ?ZWmOoHN9Ds%ZUKJ+F3{>t|D7p(|^70hPmLqf$ZW=wE-At^nK(c{qy'
            'ixiYjD95j{N=4s1!ocpZq{80cKw%H93x=+^j*=BBWy*~S+#nryr1L@k=&Z4>!YEYgKWa0znfjXaIB$4#UC)Ff0&~7MS+6e1aE$gh'
            'd-~QppcEUrw9sm!S@LLo>qRe(E&_)FdYOBH0+54!b2U&<7Rx~p<teA;^xggq6Ayqio_RaOM(x~CosMSXGbCDc-}i-Noo`;#2Yt83'
            'lx{Joa+a$rt!ynOW@qVSF~M>Mb+0iOoP)Qos>EFi{E><_)pSnG3zebhM+qiVBe8OgRF!Y2f!*Q3BMS<yl`;w)1%k|CRC1IAZ|Vhg'
            'hA_&#ix!U&(_E43`u2zn5OSgO9=H?f+PTr-)WZ><_D-flPD>v)U(FXgcw3Vq_9c-^?IlJy^GEs^Ud-1L3<R&?gU#r!WO5Zq;uTea'
            'rsh>stxb*3dP&zjv|0R!Hb2)*s|W!UQh;e2y%t9&=aUoo`DS`NeLp=v5`UWhbbR#7=@|xk@s5gJ7SY8fGK@B%bgwpYmfvDT3A};_'
            'CKAO_)~J3_uqpo@1b`i`aiFc=@U^*B#F*LY)0V1vxQgWyFawxWFM!^F^PMPqw<TH6Qncg8(fs^JZY&48m_n}BLTp#gVX)x|54K_9'
            '%+XML`oKYF<Lbb&uClTDTi#<G2Xmiwp5b~ud1cx2?<RX}xW}(8Gdj(%tv%eS35xvz21kN8KCN|$<|HSc>Fw&>ZYz5--mPiVD1v&9'
            '`^y>lR>C$;w4PqH-fQnzPCtA8>D>Oz8$PCcM&Cl_H~K)~GtTt%xrLeV7ar^3F@y)Ad9wG+?)y*k9w(D$9~<fgQ{KowwC_0ShGsUi'
            '{q}Kw_Xh|5gT;bG10dNmq=ny`FoyE@`Eus7LOk^|)0V?hlf`p-GUbvqyr_pVuJlz77>D4ZN)*64;_?R5ENp<NPp&}3F@(EyigkD?'
            'B2&i^OlZd53^Nis_$2Ekg%rxzf1^hr{u1d6t{#y9V^gk&?uxG0^A_q9?&4&)exe#koX^gV-pr1F5$DrCf1IAojwZ+O1+CDt>Fnd#'
            '$+^Aff{tM+cg)SL#rU$AXn)EnXK_wpIj&>9Q}+PYLaHCciasVPD!xIAY9DMJ2d(vvZk0jjlTtGV)e8B-Qhh5+?$@e2HR`&cm5k|@'
            '>sgkv@&;`_7v*xbUfgoB=SCUtKACE<&b2Ot5<a5;+Bu!K_rsbZ4C8fbs+^d>ouSM_UA%_22wvo2t+(9dEjEI7TvXDjBGz7%au9fF'
            'jEzX)YkS#Q6)CjWo^f@ijf8RyB3_t?wJJ^}=+)PAQwxLTo<JHSTcrZs&for|PsnqctW*Dz5B#*fTCEFu_PF^hw2*=pmJPY)h3{na'
            '5B!Z8w%C!eh0e&v1jxXPCPQ+Ai>cdk7p{-ny?kg?emW*xGx@sK3WnW8O?$C6#t^H$#<AQiLMb*dJ2TNX%Y5HdbcJ1Q-A~+V)sWq*'
            'ZmX$!-}dyu#ht*roxr@O((`T$TrNi!*fEw_Ri9s*u(^e_X~?&r*6(`JHp$CQr*RVHd!|m~M`Eg;2U{~c_xiF;^VG)4DmQIzyjcH1'
            'OGM3tuRlL@^3bWJs{jg~yx-Wr*_d5{)+>m=?6_xKe=z8@hTG|yL!W=#I5LOQ^%aZ0Pv`Hmvr@l&qblp5g*q`jUUwH44o?ezH(C&Z'
            'jrO7EfeDea7W6bxM)%o29@-;4&Sr1qv^;UL>+Jv&nER8GROm-zkNtq+W5GW@Hr(Uk`j{X4Uws(ufq`)Y5jIQ7@#>u`e6<*0;m=oH'
            'E%b@c$Z9J-#21s3Un(Wf_aufYaIYn)`=E9FdwbSxIU*GO8VI)M2B=(XHY#5Dg%jxN^%%Vf<XYW8<vIUD|Jtc~5Mb!sHRVMD9<&PQ'
            '3;aBGumd#<e_eUzjM`d1)<{e>4ZZ9+R|>gTHe6szIVUSp?Tk_Arkpho!1~*O4UOAmURKFum+lT$aM9$&Cr@}@k8d&T8f_R4THR^!'
            '#j&YD{dxf_gPT4>EePR99ESTmh@o@0f`8?X9rCw8O{Z=!Eb->2>QWlfzpu^zk&88cmpu%~b`W^M+-6Ca*I<OH7Syz$Y?%B;Zoz|j'
            'E!NQzgP}`#o#8do+bE_xMC(Gs2Cm0$)xKT2t{8mc3Wq(V{aB6X0;0@B!w%j$><w+&l5hAA^)e)L<70Z4*s7Bn#J-ob>t*Xcm8MI1'
            'z^Vq1E0lEVCK3$Ye}neo3yXZN?E!bOx`nC+<;o@5*4iEN{%jpi;9vY;0!px57#!$75_4kxKWJ2tR|&mUl&gMsMYZ8vK;;JZ8di-R'
            '@uH<lhAske>SQ`?x$s_PTdiVK=OFJ-(r;dw!7}o~klN+YH*0i+(2Ex~Z8aM+ti=$YOkwUkD^-8|SirAsHm!Nft+BpG)0dx4W8N14'
            '9PV*Id_SzQX`o>*JSlLwf5-fGPs$yCU-wMUz5W<)LFFXaPFvcTxfbNUEvrFKqn3uuCO1EG=R0`H_@@kQefgEmC4NqlSUvUEorm{o'
            'P~-hCu$ZgObSt3D6N@D0nN7+z4{S<!OTPG?eDHtJ7Jt+pf5eJ*Qvk;<Rw>bg>eE_yyESQJ+@AUB`9v+io^%EG(Kx72|1x7#X?&WF'
            'scH81<JphX6S^dYOa@dtcw1GayDMeWV2RGY%I|&r;qopMdrK(4HH)5~K`-Zu!*wlp`SUoT>i1xMrE|Gb1?Il3${Z%f_oveRVHg7B'
            'a#nju8U}!^sg>VF1(+Uui*Q<Kcnf9!Ov7Tgm3eOdFi8)Cf;3jB@M=Q5jg|a8dN`*EP_d;<93!%CQVQDEqg8t|?U^hr7qE<-E6&lf'
            '(@-f1QSY%|9s0NlY%OJf_Rfu2UPV!LE)&WvU^nkn4GqL`vlZ1D)sI(zQtdQ+G3D1Af&GhwQ|!jwb#%R+n=}|oHmkYZ-ht0!r$Mz_'
            'bL>&izB3@ded`g`xb>;ukvAh8HxK@FiF=xv04e=#ht@y&fPoyGru_qLE<+5OERW9Gqb!fkr(Z+S9XOm{z2+x%wk2UuH6r24K><e<'
            'R`2;q9Y}r?fn*B3mgboQ&fg!SaVv44uMSrf^ETAAuRa7XHXC@Bh2O&zD>N&xyrOE;simyi`}-)Z1-ie5N^cf5S5tD8diB(*bsI5j'
            '0R2zU+Kj3%*4l3Of1=jzJQ+b)b&GN%KpO6<@IfrIs(MVMaafA}Jr>j(oD3d7!6a&fu{9fuxl!~7oN_7TO#yx5(9KENgIB!!_J08m'
            '5~w}'
        ),
    },
    'video_analysis': {
        'sha256': '719335b9ce67dfea3d9d5c4055d03f42981a35651571f9c11eee8b4de554f553',
        'encoding': 'zlib+b85',
        'data': (
            'c-qZ<YjfK;w%~XD3XHmHEv6+!c6a*Db&q;0lQ=V}$)hVyy6>#lB~ub*aZHiAeArQPUA4b#zijRQ+VcVkfTXNsy0@mQW@3@V0dR0|'
            'aNg+m`|qNv2<J(pZsTQ?jlwicc12vM$7-FGqfNGn(o!v>GFp^zmZ~JXju)y}hk3MA^IdOpJfns)-EaoYxYan`?$Wsm)1{hBX1lB^'
            't9djFt1{b!2y3<pi!#djvvIHYCM%<PmffhyvAT((ZK2|_P}yCoHdRu_qXj^Umth&Hd6X{Jn=rqL(`)tNr5Y@<%{I*OR}tMtd6@Km'
            'UB|^X%GD-YMu{roPf;;c#Ue^0nM{_$0~}kn#leJ$diClxe39i*UO<B-bhD#>gJEx8m8x7vDqNHmd>JO!Sss_`4Gys$O`%33rw@P&'
            'eN@Xh8*QWTrnik3IArb@;U(Du!145o_!>uHVI2p9b6RCNd=et`B)*N*I?5weWW9M-u2m{nD-?iA2x{~Z#;J;M6R8DEU&SflIfR+6'
            ')FzD668?u^m(@0j7l6-UFHV=yHiG|4%ValN=JBe;FBaK0jtW?s_kc}AWd(GSB8C;T&B7%tPp-DrJc)~SRKT=$<r*6Ph-k^uWT)~7'
            'KMG}{L@B*h3{VgQcYs4^#qYLxbQ?!^YLUcSq&}QT9+g#|0-?p}XcKL+e5YXjLsgV{1uKz9<K8ovG%nb{;2}^zo6xk_W@!;4Wdr`G'
            'FPsvfcgE-~#{VdQpvz1x<KkxA>-YP;Ri16sY__Tp_-v-)%{I$R7%V_1nk{-gd6q}|ucDf7^K20ng?>`Tn}}g;!*ZR(a}DTc_{lGj'
            'j`1`7M*kjC<4;*?z^}ssLFk`2Xfuek+H7|!EL6Jf_0HcvegENn_RG^(AI{FzRDJF#_^%&*gnba>jKicqRQ)BaUy^O{!EBp^i)fuC'
            '%P7ZJyacm6`p+tc{oNljgm4S<yT#wC542RRR`DXn=C~O6a23Lm(rfr|SrOsSaOrmd;vPo!;pd;<zI%W6e0Ki!!@Fl^vtM35KYKen'
            '`;Yf$Z_Z!7ePfMwyxD$_fX18bmjAusfA6CChW-d|V}9}dEkl3t_8Cn3-PvCR(x6|IVP4`#n|QfQBK!-88~<5U1+brf(ChV<(F&;~'
            'OKzi@dF5ex9St4<tH`Q+5zUrW&b+FK7DFH;Xb%&+h9}cEuttoaj{lI7YXF;AnkXK_j0Raheh?CW5963lh8}Euc=G)uP@|tz2|F{o'
            'Si$ZrVZiv$<td{Uc|A=R_T#$Rgz0FN!!j;mn<iO$J)+&Icy_ogfZt|mjC&K{XXz3d0RlSv7#Fmc^p+#egPRb!Hg4QFmnNfmSitrT'
            '^XrPYoO%!2`WcW0@LvM<4E9t8dlWh|e5!y!sjDj++gDd=n^eGq;<N&u!Ek7?6mX_6xeIqik5LtFw$LSv-(uLjn(&Y6CvfPf<B5jU'
            '$DUy#&}no7+ZA4(y?rs(7^8pVm9z3NPBq}A?DvCHvv9OPK!1evfxBkNCntgVRIi--wY{F$0GHG>fkAKwe^L{CX+ZpGswSrn_2gk('
            'MCz9?siLzy&+<Xv8_ZA8L;)Q%d;JN4Cz~*d`*qjMhPnHp>9`d&h)F<?vnA`nH3qio&*LOcH_<Li9YCk*v;FfA`Mr#%%y$;X-A!t+'
            '*|K@<+5Be}0ii8t53B-ENT436A131oHI6<mqHU=L?{`}s{?M9AfLs{9*fZHaZ?FQ6mLf*<BiD&P=?6u)iwmGtjebSv1c20t_GjxZ'
            'e6zdiQ<d_OtFmQy9V_74sk#YwX?UZYvG5|qyDS$h$E!9pJ=CE)pJjLx--IxoFo*soM@!h@^J-V!xCCY`0wCO9x?#&Wf>z3#PL=Te'
            '=8iNgn6$N$L!#NtAB-BA(FP!)oa}{)2Tl`YH$(wdM0F#F)>}A?caAv3`V|D*zNpzJke`8#C%|k$^5<23kX(I^^|igE(KTRag;cZ&'
            'KUq9Uqk?FL8Pg{GIDqL7_aHf70K(AntV)-Ix*38AhUz=`AHS-p@H;Ir%xLVhcr}A_V-dl*w;X`ba}6Y4(D{o81J8D`jutl{Do@pQ'
            '1kz)fn-5XcYJdUp1lt<wmgY6)<iVgGN|nw;@|^L@<zTT{4wV^Deewe<2mwNzER{4-cWV^H*ICM11|F9SK8=q+naDB_8)5dqr-011'
            'Ky%@>2mn`Cpl@xel#~ZJnKwX)Y6&xf4UQ_wHr|3*|Lb30zB(hQx<eQ*oZLTWNkUppoEIg*LtOx+MSL2rKqb5j^JPII!4e*WesBYW'
            'g%je%3pm@aRkT6HFiNxPdJV#Rc%5b-kSq!~^a&m)7@(ukFvcLQpq2yr$cOhYMt_8UNaiK|Doj8iK)nM+6V}3TWGFDa{@X7EY6B}r'
            'DhHmp5o%MQQo>xqYnUNX{7uZ$3DvKlMP+vd^-J>s-PVyK5D&-$jJ`1b8IYU~>J`StE6OFP%2WHRmp`A;^C-_7&ueZGeJrQ%^9sZg'
            'P)IYx=CogxtI;2UvLf7O#k4QeZ#ztx91wuJSsPr4w;v8t4+<uQV;WmzF+ep2=P?<d1dZnN9Mxg_iSR0GcJCPgtRLzk%^`rSRyYs$'
            'Zq0@;Q+M<%+knUnC(O_M!0e&Lm_Np4A9_;`Y?0v4AmyIXpD4Q643MB~f#L@N3aT}8Ftfl9faHOXA=ZL!=HH3o?}FafVUZvr{SJct'
            'Jq7Ne?<r5hZb_mT*gs~=I5*swpP)WG1*sW1B`U)W)dcxPoNlX9B$T_jT*LSk>!iHV(14Jlbpv=i8Y6bdS%fz|g-3eFUtKl&25xl+'
            'Y6i2DqT0fj(Q>TbLbv%H9!l7M8r?<0nm_{@MVoE8qmM!52GxQG2F%Wb53>dEu*yZ0M2iyPzIUPJXg`4u&^m-Gj(XHW4K||q<Cyf^'
            '0yYw9%?f$p7N$>TDC-P^;o!8?JX&QrOO&X`HuRSUB2F!u5p+#yMFx#e9UTM(eBELU3Nwy(5VRfdFW}9atbBndsh0NtW(%M5Dyd+q'
            '!1qDg*D&w<PIsm&IGoEt0RPO;mUoEK<M);o9o+~=(65+|v<wmFt$2aw=Ab6+(R_|bfq!hA9Egt_P))PEa`93F#v!v@q3xgr4DF&<'
            '^FyEmG<DG*K%VxPfZk|O4sU|*>LoO9knHFc8G1gBlk5(K4utri9j66|yKbOg(l?hj^6Cj$0<+m}AWNgU=N}nq2l#04;5dTdLF7Bc'
            'v+CQ3CPir(xvEmVDr5B=6_Oj+sIXzHPhkpjLzv@Y=|SK34R~2JAu<#;>*)^FO|+YWXggnqgsxLH@UTX_VV1(D^a5FfUJ;mN?;CZF'
            'j4;|{c=U|!vV5u5QI#`i0SO=Z8OV3nG1`_>%fhaW%Xm<ZduOlTz6Zq3etrAyxn+$j)*yQ!`sok&9RHq2WghP8zu2sZ=ZOB7ZGj-W'
            'm23IK*b+gsBa2~ty(SACsOEqg{Hx3>{+tZV_(wR;=vaiA>E=mxOXj>apif&KLER+NBVvWRL$eV5HBYjPhmwca2n2dr=WqsVaPc}!'
            'x%VQYd1gtq;eSE)r|02qSl}Pm@u!3gj{tiS(Z{H+Xh>zWPU0IH$W2H+MgIUfVNLeYGQ`QEt&$E-5Y+N;QO?NR0K_mO7r|h)Lg{uk'
            'kJD4MaDx&O-dSvT$b>k9(@=g>v-<Zw3!*Z{pHn5aLEAcF9%^}sY?7Wn1INj;3Iyc~18Z1={njzIC_J*Wl{W)mA!VjQIaqVl8$ZDY'
            '|LgeVhwmRhefo^-QF0Cnj?i~-{KL&(KdBAqDeBj!zp&I@0s+wHEbs%@gU{1qk;f#1J!YqiIS$o3<N}2PH3;_;aR3r6pjnlojI->C'
            'Sc~E;$Q8U8O6Y5>k5YFD^wHoxW!~d1j{eep8nnP1Z^Im&XPX;j*!-i=ay?pJXW0$?Z0qI=8uyH08N^-ZAgSpPNOwdVeYA!#&gNm7'
            '^2Qn^*>!?`1R`|AX1^xCZ`4Z^;6~&_BeN=qJU~-0mKc~2E*9FG<$Quu8Afluzq%R~#nqJs>@KT7X=UY$7k@H2olI1<MI-M3!9^cK'
            '6yXCg>Z+6PwLm0p9}65ZL{dG~d{=^g_$%saBhr}Au?V~sMOLk90viQgQY77?_O4j~je&|hH?Sc#Th?M}?xa~~q6SBch?HF5`l}o{'
            'xdGFMS@&@}<pZ;&0z~u*^=@K7-(E-QwxmBtv3%?9uV84wE9DbsU)mpR!Y#s@oZg;J2&j1wo=*OI-~RM=MXV5n(P&B=W)R;!J_#P3'
            'oXW2?{c_qYPRVbR#Pg4^y{Q90K*8J=h4xigl>9jUI9qM#?_G8K{mEqO5YzMQF5Sc*5sT<9j<T_Os-BFGjZweKZ)08@q%&T~5M8(*'
            ')nbh#n-n%Vtuv4ZUOoN$+Yj$+^*%;7WKq_h*%TO%LY^c5LHZ#B$r`O3qmc#BI4-NQB<F69j?^5zwZzHtXjLKa0STKp&e4ao7H(vK'
            '7PurtPr2&<U<Thm!Wjd~N^-`WuhH{|UJtch19Jv0w;av$?5==Lfpp@POP^u7y9;+j)QmrDOLiKz_~byApjL$WbyS+^lk--FLMoO8'
            'a0v?nlof>=K~9bkud5I^a2c`hC0w8)YSVYPzy<bcgpS9M-;KiCEMCso<Og~s)A}a7iDsXoJYyQoZtD$!6eO-b%PyGv?09HNoOwik'
            ')2TV=L`j#w3zA3i$Rks%n!Tx#<Y#$3FcN=TgHsDbsP+kU-jPv=-CHFnhAZGPAO)s7?SLX)I9BIX4(k>P-#M}=b`ynYQeO(6BsQbi'
            'as?coO`OXt;-Mrc-T{~V1e6Cy<mB-W^&$}S94q6%MlzjMz~PGhRtG?r{qY#>j(kWA8Zw7{c<9sR;>qcyC0g(rG{u!NE5pQfYO<l*'
            'VPOqphjXTP+qCV^K=aZnS3Hju2z>=e=T%ZGql<grqV1aI1ug9+Sef=vsLduD+oYua(Be<|l3@}&7FjMv-%a3HyfjK8%LTvn(O#cC'
            '{vN|qB5fQaPE(frkAHxH@^+_Pe@uT1C)h4Fv*P#%a@k`qp15T1<b5k(CjXl)0Xnaci7(=tFpnQx0TT1IL`p$VDB$z3SazqlB+`jV'
            '*hek50D%~>zJ&8+Q9oX6tA3Ykk8w}HL}uk~TQ>kH8%@4E%UPVRvQyyqSpx4~gh0hzz+Bni$o%ikGzwMEfX}iOMR1MoIM@orZ0fTB'
            'sSNn$3@y^Ij?p#w{&@}VD0nFf7i;|jnds*uBlR2&)e$fo{U^e}f5Y4B8BivSael-TI7+h_^cpSJGk&U$Q2i~APSu42uLegSz(4qX'
            'h)&OPcXWSw?^?@f*Ym_?GBM3EC8+E*t{rbP4lkQJZwB?+>cyP$KSO81ruGlUj>gpt4K_oCYll8Agg36qY2ci8^_svgME_Y8UPlsY'
            'GKWK&1~PpD>SMS>TMs~Hjle*`7kH9v&@5R@KlkBif>sHf?v@jMQ}j>OlgZ?Ms1S`aBkdL*I1K!VsY!-g6K0-O1rfuP>CbVsSdD|<'
            '>WJ28MBaC4G(<yhsf*`&|F?M%YzAV(9*0|um>bA%aj2lw2wTwDtD@^MJ_8;Bn&}|W<FLm-=2`DMXBsn<?HW_DL0C>4Tp9O!-G*#K'
            '7`lR;(N009*2Se~yoJ%=%P~7x#b=?U{TL&2TKyQZ@E`$Kk@LcgNF(fXqg~%@R-iEUyz;?c4}{&a&6cvY+uFgau2%RSn$Ty`2cf;?'
            'Q;Su!bz;4y5TEe``uyGlbl;$!JqYPU1}<9{E;{glP1@1WM{0wweZup;vbb&wXzOjutots1ZfWm+&<29YR91>`R=>ZuB30^x&Nu2i'
            'M-W3$9=<ip?9%}{(7t*EH?-bTuotc|{A0E={A>pAnKrXF;%Q3EgpZDH9S}pJuXZ}S@9mwnh$I}AJfj%pVqn`Abx?FIcEDza*6gZF'
            ';-b80i7dRd0}KlF^X?oWKPOl!ybdv<Pu1Owt!|E_gCm^J5oqCCRQitiEY@;FNIjz^abOu8h1dfg1V=m;W9R&)8FFX(@TQ@?4pYD?'
            '9IfbhXVXb*qp=AR*Ug{G2%pP~A%-sE1oQ(I2z0yIB0(Pf+2HZi{~i@zfe9~t@b_b&h=GP?Xx#zdV8r|5>xytqk?G6m1|U=*zG`TH'
            '$nVu)E*_0jB|{!6J-$F|6|9a-E$j_yWq?EwIk^y%vkfB~zPE8dX!&FuE|)Vsnt?sZK^w$z3mw~^5z$P&Hh|Ar{dy4S^;)T!#J7@1'
            '9o2$hPYjl9G2XSh0RtB(O%=5zHlP7YGXM>eeIp1^_Py&!x)_-L^r2uLAj=m~gCnEp)J`qrTnazX5~#?KQFv20T1uI1XFw3>cb=-_'
            '$q<bevn2pP@yK!<;M?(&JsyZP00Qc|Mi&=Ddvbh2tYsN5O2b87QBHxSWq|lqkZqT&O5_O$BwmO(1U3A8M6xId{^@p{F2g)Wd)HCE'
            'X*{ACvVuT<HP#C7Ly&66<MHLaIDg4FH{U7t%)_41xe`mXyrh{b?=nTuDBA#)AwaN?9mA7^hkzQNXGO>hi6%MR8}e|B`4oeOKwsoj'
            '(wIdy-V|1gRTNSZ1Uwz=rU0+T^0fXlxR|gYNJz4qqr?w_c}AlA01y>&_&c;40L*~9FY+$HJP*VI?%gm>JSjC^enhe(z}JXVf_ia!'
            'Iy%0zOexr&Y+6%mR}~k~`qJ@NgLXU!=f$9IFj64DZ!Y?JE&G=N#y0!Qz=MG=aWwXDJ<->GWrVXDxIV#6{~RjV&q*_~g&TOn!yl?2'
            'f_rBYh+Ag}7%a-$tk)hP<&283XgQ5>R;NuY*qnlAfJ3Bz>j7`@Z=zxNW%r`}lo(Ws4(nSyP&Bgm2dot?Ye5@JK7f5fYAxY_hd|zx'
            'b$q=xYs;UGF>Dj*%Y6ua1?H%uW+Mcl{-lnh(eX(`x`QF0p@9fi28q1#i8FtB!=Ny3cThtPj7N5heppTnn$2-3!#s|PLDOw2!iWC7'
            '*XgRV+bB<fl0em+8D)*7Ddc;X4>V_nT@Oc^vg|c9(LP2;KWtEdW;qB!OpMaP^#Q|5{V<+9G%yj{;n?P>S*<7I@18u+d<9#19xiUW'
            '0Q}pNj#QQ*9T&CKc}D97Vj|`3Ew!~>2aQpt8i?wf4QI2Zq@R<l7nc$+c1}}ahq5de`}qVG2w8EuNGdo3*l);DU928&!W}z#Hqey7'
            'k>dpChB2(<2~lYj=9o%IiGi}1z^pmml01qq1%!k^2!Q6G2xF{%#STico00WH1FS*8{Lyx3@Tvf>_-$20OBSdpfzO1xl7}Z*5}>Fk'
            'M;elvN9z#HH8~?mTw44+utV)h1$MefvV!t0*IB8({b8yy6G(TRWlPMqQ<4no*x2l#e;t0!te`>D!Vq)}%gh9R>vqUWM5nYor(Mkv'
            '*WB2JXv5L^g#)E$(LJ?rqp!Sa$Y#H=w0St2M*3}I`&crgXR2yrAx?-rKO=R)>MgJ>WtQ1ox`ruw<A>ST;qKlfv~>Zd@a^wC1>INI'
            '1Y{S_>;!-de=Q$53nk~FZdsGSg-sT9D@4wvta#0QDSLy$&py*a*VuYAXp%uqO?q=V<U}Yk+s?@5b#H7C71_ZUQJv9TnD#y}>oY*F'
            'dGn{Ki#;u=C2$hr+Wd3Sf127DzHA-%P7vQRp@7Z2&54qgSEsc!2D=jyXq!dANkT2tBbX-h#$;SLq%~?8{hzcL({YAc0%e%Q*J-pg'
            'blL#c^Fp&q7)^AI^XZ#!9je&%TO5&oH--TH$?&+_PTnBcGZeo;zgK5pHI`!J?6I#v7H`ca<XWLd?qCfsr70eaCE5B8ud|2z?O|y0'
            'N6~?#Z`AW>h4a0|_>o&6y^!QT8t-FfHas-qlqdp__`uRAow_X72sD;R#_DCsDL0&nPf|l^L=q7j@Ql{7g%(Hn3~o>iV8eMf^bXsg'
            '*MdSIGwaFFIbZ`fb!KW=T>IM$^k}q(_IdszLH}qO=Ug!%Oh$yD(e1JAK6c0CC}ckOw~gI>dtmLYX#7qaIR~6A<g|6(>~HLMYd!ZH'
            'I-8Ho(e?xDOPl`SJNqYpb>B>L^Om?ya&wjuAM1q^-H1qVX4Op?)S;H`N{v=DNyFT)!*uBhovMwO3vV7fma1<3gSU{YZ+s`&e>nz1'
            'xHhfH9!=<<Qlq|FfqGbV+B;wo#b$(5yWPp2c5yl*#bE}sMXz|f0r0yg<4Nr-aP_oj7>jvWu@h5UMGa!CM?x`g96`x$e*!ZiFbrjM'
            'dvb(Cy9R~Bj&Qf?vS?R?UF${c#Z>Pl7^`R5W*(;?8wg)s;*yp-A>a|HLiHoE(80Ega=qc40DSXZy;1O#S>b#z2mqOT5`CnZE#Yv;'
            '@oPIc`Yyuocu+VeqvOeh2dOX)p-5s1AZ0R^Q0ePDik2~)8=QAkaAYi-{BT!DAJ8#oGg-p8o||w0fvU?qG<`63L7*5r8o>N+hVF@i'
            'MDbXd-q22^X};x&VWdh};7MGgivuM<@K5n{hmT-|2-b6qb9?jly}Cy1V=W-JZVVRSi6ZlMCT}U!z8D0q7D|*flUdiMOU)y+l2;o>'
            'S+d8Y&u32{`qZpN{B+;qKa*pdwHV_TTCALY$PQ?QBDD?zZEY6-R4XUO#7|aE5Y3%dr>?vbEwrfYD$6j=dgwumRjHPH+7vlQixd~P'
            '9iz~4SorY<@Z8cd)lb%(aI4U3OOIB6(i1X(E}bk`F9PF&o!yuAs9I;Dhr*v5FRv3^QBW}Ccm)_kdvxBvcslwY;po%<`Cp^|`@fGa'
            'zwHlc2qs6zjm$uAZWCkp>cWXMxOB6V_K&;qt&a3SA3N#P>_#7Aut#gG2YqDs-&DR`EVGMZ6BajECV?Fbp#Rkd81>)uW3yO+PuuLw'
            'KwEhY?y^xLMH4RUI!(qSWjTY7$5xRPXGec&e1&B_IxxPXHWW4ED-7AA;T`?)?)$cdgfH73x5Pw%bXk#4nfM5<uHU~17;+RjU#iCr'
            'a{Y!R($wvWYGd(=KcP*qUQ;4Be9~gUc%UC%b3Q}+B1NDNYK!Srm{AuEYC0LJf1d<_$2ifMWUK@JdhvPe%|mYtH`~qLG6q9PZx(XZ'
            'arXkx>a*RR$R-N8Iq!NNzQqMPMlOQE%m@YmlSWcRrHrP1d|3Vi*}JpQCv;@t9&sXPdOCQNkJy`N=W-=?b%s7w3>)-iD<>Z6hh4ZD'
            'woTR!@wIZqK!=1uo*I|b#oYz`W|n@33+L?oOSZ4U7x>DZ=e;Fi6|QdTktE3BlVNj1kgVLY!6xJ5CwtH!A9NEAyq*l(*B=(~iLKa?'
            'KQmoMABV<Ek;0a&HYD^699&pd7GI_^Shaatjda5B1@&`@#08@0P@O!8+M2^(^!*;;5S})M(P%}F=~EsUZCze4UiC)}7&SG>$mII~'
            'M_fmGIO-4Vlod#tIeutiDMlTqkaCxvp8;D~qKY(j4CEu(nk(1%Z$7mh0+!(olX{{;+he?b&{^3D(Eh5W58r3%Opp19?tF-D)_=Fq'
            'C5)EUz5#lfgN6@wz|Ln|KKoW2(iT>@T2hZgL5wBA4Hu~Eds6Z`nc5vvcG9J7&$kc^Y7(UBaEha~q=jrJj4?p5t=HnD6{0Ys038HV'
            '#nJ$dyV{QFeC$e7<cXXY0~c!sUQp`@`W3@(mPh%?SWfq;-a$j~V}OW03&~a{x+VA3C~@NsJTsAF9MTq6ZQBB5b~?_R!ND329u-^9'
            '>wVSTdc4iHgLdhO<lCmfsR^v1;S9TsIn$2kO_)uC&b0Zc*{sc+Gi`h5Hg%WYLLn;T#5qw4u!sP)(c`X04gKBXz2cR%zaVk<Ay_q6'
            'YKNr5lDpveMgt3wSIqER)i7;70JJ%_PLN_0?E>iU2m<LpbQ*u7p4vHqb2N)pOD=~CLR*444x<s;_hV3Q=3!YTXfaY;!C<VOqKx7}'
            'WQhgf<>cH#xdqKN#(%)MpA|YAbl*rPg~Eyjm>=rFBPYJaj;TsASM$A%!lcldjGW~h>V#J~lc9GcTn`?l3Z*!Nc^Q`&tE<jS;`3M~'
            'nA1>euFDxh_>hW}da$^lCyWJn9Wc9M+_g08u#o6-S(GfPd2y{4izXFx?<g`(5sa4DZI&6QpX&xkzG!lEd2wV*mPeQOAm+7zB0lpV'
            'C`TOahD9@f7?>l(c0v<b_Cau-%nR_E(7siJexJOp9cx8;QD6O+`rF$#AI@JtReyi`HwAp7+=ZWFRe{(6y5T1J6!e2VJkhT6EOvnU'
            'B=!UH97g$2Jf6rs_6f_5k`AL|qpXJ=c5gv(T__p`meAT4Oywl)vFByBilby{0{MlN^*gG0gp}*1{_SM4b!@3*Z?5Ba*gS2I3V0OO'
            '#FmgC%ILF<!xU<XrRF5*UAM$(AE^Px{Od=QWO@{s;)3-YuC9zt`)(a$loq^7;uOn7I5oZELLqa+z${pauu4j)TrG2#dh%Q>EYkIh'
            'xww|9Qp~i!jsRuZY6WU`Ar-{KCFT=PxQH>wGh_v2wqZ;dm8(EDMFD15<ye=_NA!)=iz-P*k_B#Ft+7@&^KT5FN+P!vacba-Sldp@'
            'P2}O7RSHeX8P?aC`s|yX7g>UZFHA+UM|h3_V(bZFG?=cj+$a!MBVE_<9T@BlO|QYBxdt{Jey)Fbi3ZYvt`VnqG#|tu>vOavYIIX9'
            '(Nx2^w*-Ts|4JY4zg0G&MfAFii}k4S&7qZBDBZn9QnQ9Vro(A48S=+NAl0~>_HTev`a@lZa=PkI9)G3}?qL_ipO8BWCQ&I^5@RsC'
            '*s>E?y-FpY>2YA^xYt`&2e1el9s|zn*0az2@5K>{6-Sq+lau8=6>_iRqWFWp4U^7P$G~j5u)+hmq0j;9xbnmh$)#OBvte8RXEWZt'
            ';w+8MgJB~n-Ydo8mSgd!=zaK&dL3>}DY|4gR=<kygn9KtU6uwfl3@10u1G1zfgHjEhu^$@d;Z>36CoB8>hcsQGwzTVQbCYVR|w&t'
            '+9%^<A6kylrCgEcI+nN-eQGS@)hf!Vuqq9zgdS`51|U6PA~phB8mpuKek44a@{V;uy#j?|tf4dZ(0Oq2nq~o;(a9+sc2ad<PmLNl'
            'MDmb})OZ6wXC$p8#D*@~U@=iwTyvQEDF((?=}n4Pnfx&FOTIW|U$EK<gQ#oI<Wz-fC`TY8pRt5ua8~#gTHLZUa!!D%L3MPsx1Ekv'
            'pg$k6%`z>mrCPe`usAgCZG_9is%Q9B8#*dCOdAy-J9$iFOtoV)@$DZz5atJ$y27ol^>9qJ9(uM`2jX9aq6zVA)W23(;B$>deza?v'
            'E9*oe7{%7^V5KK?%1Ys-DC`UYF3%^3rS3Q<1oQa)hs0<6=)Ucw==Huib;OkdRnxM<cyBB{q^~_-!bI>Spr{k8#6_V+7m#oe90-e~'
            '+X8EmLd|%iL`Ij-pc=ZyOCN+7>1GEk>C7SWqBe7!1lcALpoW5R`leg12MZf&xvCP;c{g~J?F!g5Xb_7+lw*+<(WvZ(93EH3ITi#<'
            'Hsjul53gR$&YwMfBR3b!o_~1vlpICpXU{Mk=#LnN3SiC=Kbwq%#A^}<iPy0rOETqMNF}AUWY2f~R7<q*3O04c9)!pDR~Yo**K_<-'
            '@ROyuCK4`)I3YA~@C%D}#9LY>@cP56_b-2b_44c;4FAM>|N7~FSU08|JFnipK6~^2eD?F%JNX2IdvKV)l1*Kk7Os<Q&YCz~zC?FO'
            'i6ug!d#K1R$QsuyY@4d3xP}Ptt}X?_=^M3B4KfCDOUBh$okt+s&Ja{2HTJdOa%Io@FR$MI`RS|K*_*R>fBF0D*Rz*@`RjX%Zu;J<'
            'yh!Cq4y?N9RatzfMB*8DK`8q5uWw(S$x(JBd5w5PSQQO_e)gLD`p6YvPl*!rGj6YA%BfH*e`BUmv@EC`S`_(2%$tTg;{94@rO2w#'
            'o^UoL<$#hI92CAPFR)+2Kx^GZiIK5uC+fP1cao{1r3=<}lw2@oiz=@Ead@r7uP%4kCLjKG)sXx;eE;98A^j$5&KDfYUsp;=3hEYk'
            'J;;|-QbMwy*)}vBAeAk)la(o-)lW*EG?R><Wg&Wk-n~B8{0fhVE7SZ+g^N5ZP-N9x>({5h4A~$Miy)<U>hlPsM^dL?hb?t<8pyOd'
            '&GuAbVlN+5<y|(y9VQHbrRflD7rae**c9hQf?c83Rl3Y^yW`z$(RGM5E9Kt0t?&rk+RPF{+3V4WbEZcyuMsDO!$j!TuLx^2@o-ok'
            '#5^+kgQah|WesFIsaOtuid8z(Rm2si|9U0_&Tzg9Z=*UxfMe-3UV{fOkTIWjUl&6`MNSNH?L?&2j5}Ww&%yzY>DGlts&zk~_vuy?'
            'W>NOV8>~99D2d$bIx18B)D94fm&Cs6uW%vgsZ&G0jGT{X+PPb3A7e!b8f>eR2P>UxlL`>lXbwUcoGL@9Se>fR4Cs;i<2}A~3#`*%'
            '4!J_=I$B^n!Y7sGDfR|B&pQ5OUlT=iG~=vtahxAx4XbiMojhvLgJpEbi$|&r1$s;#ypaM;Vg`S>E4j~sJ%C1_8!|v>czVEH6uf&R'
            '>Jk$n=M`$U$mMm8GrWL_9sdxFLHB_N81O^W`|bpt-QS)3+Yf)VZsIXlx(Fftq>h^dkz*3?9pV9usbu=&$EF=dVKhpbHb*UgU7Rw+'
            'OTbaF203^zIv&=rI6x_cV{&@jz?L(DrmEGBH`<%p+HSHPcwHaM>LV$T<In9<Z(ZHmL{Sz0nb*-&^t*y%USlW-0^f6^JN$O)Ov1^C'
            'ua9A!8gC`DOY(x9%Wj?9oI7uQu1DV4!qB@34y$*7^_d+i=Y<(^J%rIQMpjsD0g2jj>qnE-8T^1YIWq@xW}eb%I8+a%w$~K^Ujq<8'
            'ES;KiVv?|k<{fQ$mZO2<q=&jL5(|W4`Y^>X>g*7fS8!15kmJ<|hVD~Dd0^YA3%e{Cg0X(0HI=^c1FB{1jlHoEyy{1RIx(znyr*;Z'
            '19=t^+Bi&igCOYc(y8~NX&U2SDe^Ti(hk{XW=3HWji>Mzu>F?G!)gGC64lZWt3>Y8gAW5tW(l2+^L<xZ$j>nd9(>U&tI=XsxWl5`'
            'wQXT&bZ1JyZ{#tEDE0!hQT9KhTBAW0Taw!_iUtHFk?gn#c(O4Xi^;8LmRXAvJ57tgmDw_hRc5KWJV{03xnSRu$-f6!@6Py&3Z<`h'
            '<BMuo`3E86%9!A0oD+97l!QKG+uLBU&kfy-h;DY)-(0cl3vNKvb;G{)I>9pB=o>LgEEb8^v(b%$`>ztT3Br>KtT%!4v3Dn+oE%M('
            '4H`SZDhlU9dT4g@EV+*KBwD4<f0B2hk6S3pa2;DOqoPKe<}-_-{iG(nwp6*L6+K;UxpWVT7<Dk8u&R0t)<r8xMuB*UXnRG5_EPz@'
            '0y2q*U@A*@Y)3S1!?fkX!^hlV&*c{_<ti!e6xA(Gy`fe?`*nyobpHGib-VW}K!4mpugSjl)vBb9`lm9!Z`JYekb0JFmkc)BwaNh2'
            '!qo9<&3WwwOkx0wF71~M_kC`J8M;*(<JIFc&DQ#zWi554DPvD+Wl+|WY<Yh>Qaq@YywOO)@~`e>T|nO_=}q6sbh0mng!Op{b?$Xj'
            '&|FX{g`^%tJ<8;h@src0IEx#-O#mBOW!V~nx3p|6EbqoiTP3P&dK1`I#tOx-*(AVy+&5<a)3h70O^II+Je7rZ7>$h3PiLd`b%)!T'
            'VQWp`pk?TQVoke!EofKzOyT}gQY0v#){e}1IAr;uU}ZC+?s#QWhiHn?d4t}=akQ`}&?g)%?Rapq63J8d+qR(FMl>-ur7(X&Lwg{x'
            '9_wjuZz*5~@o*@Xj=7#rT~V@y5Umy3TNkYn4X;Zm8kk;&*aIxSGlQ(zT}xuv9zD;Cnm6U`9)M_(coSMpZB&fTpGgbXcT<170<&KB'
            'i7u*F$CSwWiD}wsp-sol!2Rh*zd4#to!sB0NxTcw8{fs$^e?@9bN=%AS<}BHk!Ouc2Uo~?MsR-BsYvXoG{KK`YB~9KQIl}aGaZJD'
            'Ap$1AD^&k}!UeWM^`BK78J{4BUFetEd4r0|QPK1t5~s!Wt-HV~l-HEYz76%Iz8l4*HKeSV7eBBHZ_yx4u?3`k2>3PaR4%)TH<n<S'
            '0i}+l=<}0GfO)Jhxug|lMT+iS0v!q~OcC<UvcMz+I%l-A6Un%!v`dp0@8x-IT(Gn@b{v!LHX6xV*oFzjTu)yQtB&y_E*H9><XsAL'
            'q5DM(9eS)IA~FfLOX<!g@*x|SQ;AAp9g8EDI!{5Y6<L{|RA5tKBEmeA89Pq3Cxbq{5}!3m{jiVL;<E<S<6z<xb3h_5YEN69+#{(D'
            '^zt@XuglifelF@jh-M&u<6WKS!~32iU0_sHd67oNe-WpUK?A>+hqb7qOO%nu4q6gQjs%R-sz3_hA7u98PMqRUSSoUAeb@4tho+&C'
            'w^H>f-g-P~Xhs{H_F82$ZFQ0%gOceLs<cB&TGqW80@u>OPluD#AiW0smJ{8&<67)q5w$W0QxsEXKv{mC;rgNRyBX2-OLs^IOspZn'
            '4M0DDA?|nZ2gbADqogJF{DD}3N9v(z=stu}zF=*n4&gG&$;~0sqEB%hGK`BcPjD1{OKu%LP+FCde7VrsOw6fePoE()^5eV@JEhXS'
            'fL!5J&-<SePtBFh4+^OIOA=AhHbTNl&?%teV7W|6TL`X())s$jNKNN-;UBgDYi2I@apSdI7}z4n)4^pAhq&bIrsuH!M-SD0{aTx|'
            'T&6Q?x%w!<+vi5N-;H2i^Q?a$ptg&W%*#DjBOOw7xAQ)PgG){Q&W3i(OcZ+ykLADkG@5o;c)Y9;6U!}$Bh9ELCuZUW$g@^9NMVIY'
            'oV|Jf^4*zzrRR|9zF>iLN@LLCHPR<jou<qyy1A38E@cw<7hhLdRp>Oe!O8cN0%H^;tAh-_+M;g7Q;<9u(@oM+LJvh*n9~+7d(>mr'
            'Qngw|cjo#%Eq_Yjo-OxoGsG)^4YBC2A3A7BqLNf0vh|z+l0q=2I*#zwM`C{>>x3)d3QLqBPxZ}FAkWiX$9Tz>2_8nFTZ@epyo@P('
            'ShN>rNW!xwD_zSt(_$}1%oK9LBRWA#?eB>7y^j<HCIa_BWa!W&7fNj_S{4@5sQeVgk_^T5jyzEF^ulrD+`-v~(d1YpSQH|>mvGE+'
            'Z70ZYxWU<fygYyZ^37im`bgG%8*e$s2Sj&DM3SIAwE7SzjWLerz``B%RaBDxEB9n@YOh^zOjj8L*g{1m-3O{ij>H4{Q>{pboQueD'
            'jCcnlD-E9C)A$B(;6tNJ5Sxy$xN{n>P<I0bPHs|jXcDjJWFJ)OXnl5#=#N{eB#9C6qApOTDPL4@W$}Y6uY#9R5ACG5e_*&e?l_88'
            'q?9f6O0{fK#ySP*!uGl{+0%eaWUYPc%;{fAG?7e{YL$^Hj$t(mj6Cp^nBO39%{skw=%Dqe@y(&pg8!nfwaA1*nTv;Kmf^$ku(4?3'
            'xYp*?67P9s(Z!+}oC0c2W|Xb5!t7Z&8MgQRNbLzg4s;FeZ(#F%?SI1F>=da2qvHocgbTiU{e5NdNwJ2A)YS4xK6KAp9UiO$Qf(T0'
            '?2!EUF1+pQ07|k0Osy*ytUwVePSA`W!N>k#?Zk{6qk-Bt8v?c6>tlh!HQ_AT*t(>1Z_VDY=cYL5O}}w<YhWjG$dtQ7G(BkCST4pH'
            '*a4zLYflW-+jIIOXk6)vxfbSfUAD)d>gt3TWBdTHpeVnk^VA%vKjLZrkE*cmM@FuMVH}aVu43V)Xlnzh1TT)4?SVcp6I+161i(8%'
            '#=y)BJF|%st_;5~kV_X?oweaD7iKPE^cYMq7|&)YuKMg=-$3i<dSG$O!KzQ0q2WAClUUe=+J##QrNt^8$#sOH3{XBW_V79ebc74o'
            '3xm&g_xC#8aut(J&%eN!;?6nL6XRcz7XwcZ%f<TC2}^zbXExZu=+OFTT@USMvn&lZB_W1#&n#msE#A2M<|U7i#uXqSy%}uc0#j*S'
            '=55iK5R!M*3-FO%&vig|#@+bj-Wqo^^_~rTdsKjF%d?^5@R&OMx;-hB?&;w%20HDBR=-Wfu@ZsocK3M6*G@_JL$rvKr3rTU=Q;I>'
            '6V*ZIv&Ttg9DVP@gG2hWWS`Z;s<jo~hOiAdz>=;Z^*_*8?zvpF@!GzT&;C@#&TCXOhvVhr>YOUu7$CT4vjrS-GpTzZ%6K&A>(Y8x'
            '{Xf&v>+CR>He&26qNB5=FoXaYe6FK;F06}pFELEyhe>ch1}=0H--LF>2CBHC57(MLFtJ#Ee%27l*VLL->>kly0L7fytm~9?jQ}o|'
            'Oe5&#Li#MF31ID`htE?p*P%7y2Q+k>b@+9^<e?)HT<$rgbzRi@E@r#d?o4hOZd_O0teez&fAzF!*D==~yOx@kvQGAO$ZeJz=NfBa'
            '1AFI*A)?lOrp@4q4witR8{fvORtC{|0d6-(@zBk|Vdiep;7~{&^Vtj4xjNMC;Sg?OOksF5y7w(w^sLQfQ`c(N0>-;t1mjJR8_wL@'
            '35yH79PW~oDDIrpz6;NiOoS+OyZ;Wupc$dA!}n?^7b34;4xNYOHt+4N9M%${=x@y6pq;L~P;XQ0RAT?M2dmAYLc&X<N`Es!px*p@'
            '>hrf0fNoBpl`oODIo+2?+N@Yt+r%4gO5Y4_hQ*zeMf`Jn(2ow_(8J{49k&g*;T$a!Yj5SpakMk#M%#PO4<X{qn99p72Tl<X=YSel'
            'TNFw$+KX1qt^mwi1=Y)1c(vbasQW0<sY|*J^FrD>*OCwULTu0;B$DvHA#Q(-O2@+H+Z2<_jV>n=65n)^VSq)R-Kp@7uUxY;uS>Ft'
            'Y0J_<SevxsYBFp^!`|EQEQIXW=I;GSQ(LEF`P&y(i~tsGBFd7aIb9q{mq(N$X||KE#+lOLthG^1pH@+%^_1MTA8!KeLFMZZCCpLA'
            'JZ(wjXzh=qi3j_NAAFHA<S9h0vM(=*EI_RkNrYpqc<f7g^-4xJ&3wRg)jD3x<J`Vo0adbC^7l}w_hfGrx!cA{X0Bx%{3iMv*}Gc|'
            'ZDK)3Guu-bW)h0`1*@MWNVlqp^Dy}$dz)Jlwmn?TAbeP>;cZq(b4{{6Xz_OyTrA43xKtOzP&Y|D*G1;OU;+=8erqZPcJZa&VXiu~'
            '(GEQQ*{Z^1CK()7F^(r-*NmH*I$u4>krF%B6o@fSmbQE`(=Tfxcj`&md{i$E>@}<vgT5Y?^+%tZ%EP_2Ve5-2rv5EGQ@Wf-*Lj{k'
            'KYQ`?!>jia$Z>92UmUzo8`T}ORAD&d)b*dbr9r2Bw;Dck?Kxbi+A{$;v6{ABiO6>9g`FB|CUK%k*<S1>tmCp;(APde`(K*aPOFy0'
            '2%V?MdEI-qxNM#;0Xo5|duPd46qsd$M&C7S+}Ep!_U-0-?Xzr|8V3gNL(V<_88{S-fv|W%5J`=lXz-f2p02v7c6NE6+p!nqiEt&<'
            'ev8rYFr4<;Jv+l;LJR*s3-WB%k-H$(&yIb2k0qtpHel}~b~|Wk2(3DFT{W{?qa+1#Rbw;xsp+qqG}c8(y<z_Juh4GI_HwY|SV)Zs'
            'F25V!6*KWa3Q-wZq4^=Y6)e({%Oj4RYs6hKg(_;p8^(q&SOzVf)Z-Np_Hvu9XLAat<r7f`ZVo~RDu=Dk+P6A;WJVN}>s_m-8=@SF'
            'gBtj^fp}`=o=9j!o0PKQsgWaG#Q1&-T=uTBG-XN%NnnXi#8NL%tmSAmu3ABX#a67XFB`eB#jRzE+>l|46sO8YKB&+8!v*hubmEDr'
            '5EBZ!FetkmDrh@>pVQaDgA_v15?aa8##j?B+!Jry3b-r$xUC!7vOp^18j?*{CbFW7Ky;z(M|@oq0MsQ%sbH+NP87>Q+L}b;Tfx`f'
            'GzM7N$kX?OAGtAsseMs8tOlfq2rCguO=@X7ZqS`o9+gIG0y&&vyHWzj<s(ty0HkbT-#;>!rsN)0G;E;r-a9SY?2vR~w7uwVYix*K'
            'Q~mtmmX4jU1B(5Ptp&A>)e2xV-2w@f6|SVz(*71s(jV2YPv5<H`Q|UD_C0wRC=zY9<xUN-Vz|DGuVjwbx5FPb_+GEKNYLBl1c#W='
            'wKv%)+p26%-r(E>=a`!UG4M;_G8Yt81FB=0Rf~1Bq=E(JE)3Qru<khp>DHC?`c`FSH6D*m3)W7#g_dZRrb^!<t)%u307X^;h9uuo'
            'Tw=%9Ey@bE^}d$eoMIiGJo#Zpu6i;C;8y^f@wbexS^i&##Ts;@F@HV?>WVs4OoxT(&I&qwIYpJ;;L#c5-4`6=d}>#585~bezNZNE'
            'lkefIpZEKoVS)#vO*#b6N$8UI>*yo@HP96mY_&z=IAz|pY&pYAPdVktMFP;lIc0zl78Iy+QqPGB)SIf;(IUqCAuvFI8Lou!0X0@h'
            'Ij++-uv`{~fMW>Ocv)?>MXL@)<3aN{#{bO1ViCtv@>~unc$}`>VV8&%#*9Z?ndQZF&>y0Xeu^pS5<NM^;9nMw$2^KbzbaRwKlU9o'
            'dP^<i4O;NZN};HqCYNl`YnzDVayg;ux+w}FBySF79LPrD^?b(x%kuOejYfr~tcjjX=ah-s?GH_;RGb_2{FI7zh}XyKGc38W7f#z!'
            'Mhs|H0!Wz<MjD2?@=-$3c2{tQl<_KDloFqZQ5O0Uo?&E!r-Z9agclUzEW=PyYvlTZ=;$Lfc93e0A*vOI^*gsoNY}P!2_}nC{mmrA'
            'V-lHzm9)i?FdSRE1&)6kaphV#jZ?~UL1P%qQd>KgLi%1w3gDfakPD}xIp-Ec1QLdAir|kLyRVL*vw`d_I^KwBEFy}Z)Ag!AUvhxm'
            '8)B!$%B4Q6MRPb+;62XukGg=AG};%;$8L|=yR%<jp8b0EeD>$R1L_NvVSLV~AL|0Vi+B@b<S?N41~x5d#IZIvIo4ORS&-l*aIb9x'
            'xq(&hciW~3zG-=H4*&e@)!DQ6Fm64Mr*HnIw%Ml`=v1Md60Mj8+9!2o431$1uszEq1$fj0rtknanISmUcyOISK0RF6%iF)aH{{{q'
            'L<-h>J@1*MC+4?JdHUJu<KD8y{@BnHHqk9!f`}g`VU85X$vxhP0xP5|aClms55%z)__4_xA;Wzu_d`m(xD3RzAa#~z7SIc7Zv!$f'
            'srFkZ@-8)|eV~V4000915^p8m-~njx;zPTy3p=?>YK?Rsu~NWNWRBo)m&V8$L#KP*d3`TU%G8!ObG~qfaM^3Afp5};<i=F1(6oCG'
            '0H2+9e2{qqs`HFkv;CF~Dpps>-kqBUNHiORHbjD2OO<{HZc9@X!3X%y%#=g(I<_DjP!MqCBj1#P&x#!~U|@Pw*Rvjk?X7Ac$`wQ#'
            ')w-jEiu$<6_`y5e8!E+Cqs2dskYw}v$TXYW8~Pvedw<*aUxc63F^PHcr#Out8}$#p2OD^zU?br5j;Lbbl_R;m{W}WM$yr5rKZ2@s'
            'Y8M~qYhi&xW6;Rs%Ne$1hjlKp3-V#UY`)bR*{RaREGH7WlIV`(wDy3h+sHn>Fi_A2*m5MH<;>nw-Jdtjn)}<^%Dhwk#Mz*Y4}x|G'
            'b*Yio_WHHox_NJ!51{!)7mThU`0pDtpp4MQ!0D;~tyygMV|l#y24lB{^{U}(uzBKXv-Ez0(~+6DnX=Y*83cfGA9Q_fc$nS7JnXVY'
            'x16nHpylVzr$$Vq4_F**`yedq5AYYB)hGs|MQsd+AO-e(f_aata)F#Z!&*zoqZ`nS?X4D6`H?57>avR#fNsqfz<q&Q9CW}GHWC8T'
            'Xj~ptY$_DV+{lI_UKCag2OZ7CABj`@eD90`Rj#_%&+YV}7XZF+HD#~-3Ejk-Z@NAVt1^SNi5C{DJ>@Xs21)7Kn?~<x*N@FlTQT0J'
            'H1F48>%u5N3|3tfNde<Mj<<bn@nk8eXuBQ(>YftFL9xdPSMUVIFI+{n-9jsL7nin56c?ENh=!(t0XSqMN~03yMdQFyX;h?`qElp_'
            'k_z+2uH&SCX-|0R+`Q)>kVF4e9Ga7F*l{TBIfaJq4j;Pv`LKO8^k~y|N1`Oomu!)M%ky(;OE_0>)imNpHqlC$Qn^i}qI|QA)X`_Y'
            '5$i}rH${9?%i0aKnwN(ky8<~HFXTvZ@E%}cYO9E`JErE4+~$~^Q&P=pyHsSeQ+H{p&!)`NIS8k<Y7B{Kt*dLzu&7TXK|L^GYfo`|'
            'ZAe-Az2OQYJNJ6EsW0%@duQLa?E<83U2xJOLZiBvjvNvd_NvEbzjHMm!%mH>*R9U&4a{#wU;CJ7_SVLh59_d{<$-;+ad^`wG5T&Y'
            'Yc}N;rq|!yE+!+AAX4X}V9zecq+(`&Vpg1yfqVKzl0nD**~1}ePqp{miR6Zv4{-}Rm2XhfDwk+2R=Vu(Z<spJY*(|Gw$mv%#E#e9'
            'jZ;jhb$7V^89Qm0CFUXH+97?2eax;OAxG3MyXgmqoI{pjsL3<i#P>jqV|zx*HUR>w?L|?FnUg3-MHrkJ`BtM@QEfIMr3>v}OrNLv'
            '0A%R2;<z20SsserF^#<y<rp9k6-iXs2W*Em%N<jTRqN}mQ_%83TU3j+sbi^HOiX>d=v614J!O(G-feLDs<AM`fdePPN}y>TGKT`W'
            'x_^iT>(Ch+1C{HulPzPr<yvDkJMSBA3)hrtL3K3WVRe2??O*3|0SVO9B^@7*!+f|GH#F1q)QHYJc-V~)O&+c53v7A<|BUz+N8sKH'
            '8+`A1MRhIs&Nl*aWW*#^2#)x!z?i?QM>P`18;h6GRtm;TmBKO`X<_p*$A*?1UKTEQMv3b@)m&e_(NWq=Si`2(YS?0o8#c^q-p1m('
            'GTIDJ8<qNPy}HFxGpXHB4bWdKS5>rRFtoc0_2iL9PCr{c^VPMO@`|?MT602AR}m&G%AKmu)cGIsd-oO{9HEgUeB@lJpOpX3F6sK9'
            'q5bf23wT{IM7!;5>9=jfezi4gz|BH!zvX!KwRY{pwJ(;+3iMshGTt#Y_uy+WF+d%!;qdNZqk{_CjHzA#f9ikJbMpF#cepT{oT(jd'
            '%Xf_^ddtu+@yMsDpJa<L8Bv)cj^!QQp0peCJwCD5o=;HKI;wIMlNVO;KZ}cCxuYLBbI4L%1%9Dky?TvO42QXB7YWA-Y^ymOA8UIE'
            'ZAVD|wDn}MS~;=e^$%xiR<wBi!-m0U3j*N3+N_KX|AI{h+PLb`4>?JlP{1y8wNH^~{PiAn1=$;VkwLT(91NuSxN|UV<lH}PrE)*='
            '-Z@gCjr*YOYcGhq@yvHCn26E}<JdalN?ZLqk6k`>aQx<vEwu}pocOWFw+Z}YG!HR`)`8x#5-lAJ_`YU3>gf=Moe*y}Tw3nJ!>fnq'
            'ZyQ*=Z0ka*OXGcLG~3`uS58OKmgAE35Ks-1uw3mk2UdpH2VxzaB8qOtZa+c0%f!}xPn)?+M(t@zR4vh=jgKe{mAnFtS3ySvsF(j^'
            'c&%H`91ACXH=(P*IbA(1FzOW_pFF|G6mRN0Kb8uQ{b-ZXVt6lIzK2#P_<$G=yvD8)*l56tXUmw3pVD#yVDGKMoo>ywFD%*FDhTuC'
            '8Qnr}mzaz<1%J=ERow+Ui%pm$dn#%EZY4Uuv^r-~d%@d`_adgDyx>KmgpdcyW_E)Nm*KXA^|y)(=_@j{8;2~0P2J+JaGs6m2HGLU'
            '-(4rsNTNAM^@+}9;!MD)sh~JWxtb%FK%`}a^E+LvK{LjzkKk07Gtspofqbv4T<$C%+V^{_SQIG6y<FB^40ZhleZ4UiU>GV%@UToi'
            'Lz5+0*v1Gh1~rr_?RCMz2Q(=iLCH!I$4aIwzDy>wT~?J96*6JdEas6?;^8>n?$SBMd>3wAd{leheRwnb<>m9Ux3i~jp1%6~`OEVk'
            ')#TXQo2XceKx+fzE57fde8lxGammb>06B9EryMl$X@+iXjX;u&3ap4_tzEs=_WEH}%`FDBQ^2T>Xosa)Qz8ZiAt%>a9+zwH!sTz&'
            'm8RS)I-x)m+~Pnn)kY%aIl5aiXoJU+Y%MpPnZ5)ZReRDEz;G$&(K@`9@Kn5atq{TSM@?u2R!`*uu~^9!ufoL&qgk&mqxRt?6?6p^'
            'jPEDl7<?QraKDrb<8$FvI0&iiYoANMVL1TIQ6B#mB_@B*8k3e_*{F;q!VYeay6VxsqJ(w>Dsa@E2u%^<;yV2sXQSU>PK{j1`~_Lh'
            'Ur`FFd9nI~rk++w$t|4LS`o;dICX|oK1X`H0ZmJ-!%q}(+w!%XP?RoyK*ll$cv+bHXR0k7US-KNX|J|)Ky@Y0^1i3alI_L*Mm<d>'
            'Nyp0EiTUd5Rak}z=o(nKBVLSXTRw&@Hj-Q(DQQakbRBB`VP6qH(s_BnMu;npP-<n>SPlx4yfZmI!YG*d8dX?<rEB~WbA%=!?gFpj'
            'fOQ_VR2rtzCh8^KHNqE!m;%27%Bb#xwoIp~oY{1xOI`L*@pqQBPBTfEZ^SGuia47MIXz%{gPBi^ky~W0a5RNUc3nj%tW&{XjIkc?'
            't7%8J3a41h;fY$?9KBOVAO8OI-Ahr}Z)i<1aDADj%SsJyva|xJ50F-D<5H2c1gnz3f$-P5az$~I<p~$CJ@Pj7Q4ua9H1=RoC&0`<'
            'a1VY9+<l#5-ej@C$`R>~%5cX!m=hPZT!c42N<ju_eIv+iLT4=g&m*j31nAo#{?sNaQw~($A)xV5>u%?xmzaYdOkCn9*xX(n&JjBb'
            'X4zbCZ^f*G;Qtntw44GeP6;9B2WP0!5*<)eQkvb%niBGsRlfu0ri{qBO%PU$DUnhhABg%dt_k+FwZQh*0Bf91!SL|XUcI{ds^M@('
            '-UG6XwjvvYAOYxGfFe^oKAseY<>{LN5rhDvG$kO9XCiWGktzC!webrp+nf^?`zWhn$}#2#tus1Ej8qnvg}K^_3Nw*nqb&*`3sjV9'
            'UGpUpM$}u(F}g!Lai~rvn&%sRg>$v3de{?Uj~l!Ji9#wkbDJ`)Si(aw4Q4H)7=J|}Y@>&R+{7LQvi5*h_(U$gL1SB|!p6zK4I22#'
            'c45{!r_cr$FKM=V+)={DGSs(Du5o{St+QgZT(hH*H7307XV40Eq>lI9A>?#|*tN{H&B+B;`QI0~1CT)ZN`++}ukcm{R+>UA@U<O>'
            'bkVC4=C%VRhHg`!(+iC~p;Jy+Mz2-?Y3<x>1H`xjNtpIPA-5M4B5<&c+XCd>ByETrNdttlic-q!+sDjh3Pjf(>&;iaKGFN$LcDc5'
            'u8&H)Mg7uh(Mh(_`q(axLwAz?Q5cF_U1?9Dz^^Y5*l8x_T}}UQn_{Lm<ybX++N}Gqe`Z*48}tmk1TA;wKW#^gEv@RikK25mEhW49'
            '2bAOD10ARVm~vc%aA@s9lxsjL$800YDGB=js+Fkz6{}GF3l^Mum_&b%;#1b^t`bxaD#X)0KA3-jDpYviMSdNXBa$~p`l!XiH2aU<'
            'E_xyXs=taqMoYOwW@Oj`LJhvCycP4{=5h8=oh!Sw#q(R5ISx&`nZ@{9n*APCv3}<&B%QU6zq%sHL3NJ*#j2Ay*&r>^zFysl?ktXw'
            '-l4Tq7aWjUsSHS`XbgCWvGf|SpmL5zVY=NSOn0`qg3S^GnPP3V2-rjcl({T?L4r;4C<>DrkVh*}80gNdJX=I;MhO822_`j>n3i;B'
            'lA!i_6YV(ONa7-#G!EO_=mr;JGl@_oEbh&dz>yYkI_=Od4r&HvMVzXm&j98A$PE)l9o}RiIsJDeznF6fmxzKt?aI5jK_{Kjen(aZ'
            'M`Uy%iGspVo!}i;Cf9Kw`sk&YCBiEQmJZe7(OZ--I5atej$6|zf=%9u<;V8R4E1E2C1@KtFYb^}+bXX+nK1l$d-+&@nJr{qe1^f^'
            'Up5nBs6eTc7UKp|{y*-AZ+!'
        ),
    },
    'approval': {
        'sha256': '119794218b882b74fe3deb51c7ea08faa1cc706fe6145e48bc4a40116a202d9f',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;OTXWmSl78o}n1ehBtfH9O-Ad{dT{=~YL?_%RlGl=x^{P}P2o6bvAi%}N3a!$A-><u8fB^_f%4YZY!IprTo}RvZeVKth&$|ec'
            'G>O6>R=YGWPFFh7Sx`i2qSoa$NKS*@E=#}Q^(rXzu-E(Tr_bu{r^&7Q<>vC!)mV-Hb8>rk+dJ+2(>u4bw`E?axmJO?9e+6gcz!ol'
            'd7*czSO<k#V-cO?>iqrP_#+-^RpvUw<6H${Q3i3mSGm?nk6sxL^+E+%5iK!bs6GUF9wjR^%fl?%6;okaFV60=QqNSB6e*KkmPyEa'
            'qh))RQqJ#Ws@m!xkq4H%igNH8Y<F1Fvy?p$m1)6ND~K=d@m9i2GjCC$PEWDnWtxFfvKMKpFfHQ+Hugm;45VR0bw~5CMv@j7sjR?z'
            'B=7+d!n;6a<$MmAFe6aIt!&6qmrjao4<%x=IWcEtQ4bmJ_OpSS6-7=zVXZ}Pmg!`nvuRmGu{>SqIGVFqAoL&%HDq#qQ*i~UtjaCc'
            'qUqRoqS6Ed^%tG>^cKv5;+IH2PBZ=AWt8c~O!;Z1X3HRo>6dTSEK2fnxs1XHRHlod2=KHwTYyBI?uhhs7YCtUr!hF2^`YG?-KL_p'
            'RL!ggW|i6KKf<`oqc6nqT4#FDBi@*=$F&wUk%mZo<!%>8M&q|XouB^eci*dY$-nA9NZDbU6tq2TMRuYW5cdz0>&waYkLvuxhmSYE'
            'o?ofE8+CDWeK)?oJ7W4`|6|ByUkfkOVVW)SdJ|K#+%#VYU<K+SP8QKh!(9wtmW|$gPqykCLoM_l1;iPYizrn~NEg3+i56OZf!Jo@'
            'T8EpdUEJ@_K<X4Sg`{$^R<qC&0&bZ6Nv^^&%SaA}!GX$COt;1aUki-K40@oNl}SPm@9u5~3i_r;Y+3Y>C<|vnS)^M`580)o1Z+i%'
            '$^&<_)3nTcmAqgo1-S_%kz-RyfmvaYWqTqE4Z*t}gNzh(ofd0)NcvahE>43*K2W=I9!L3F=gbss1)C#pN_6p<W*eYN_9Y5|Yo6!z'
            'SPawYvMkC>Pp2x{?$V459iGV!l=ph(8K7wmKe505o~MaSr`;40Ai=z`zeBP3`)itL8Mg}xB4CGoz^}cYv92Gk#?zl+aJTAA`5r_O'
            'YKAxRi^Ll!PXtdteM!MR3@M{uiVV<Iy%PC*{a)`IKnzLO(osZ5P~~!bF}a=GTvwVO2n+!R7g10>9l?<Tddw&Wf?|_75n+SXVU&}D'
            'Mn$gUrSiAwLX*jqS%`nPDY<pd#+{NsB&!&Y)3Vr=1*icP$-;zVQqwY8_1WS;ukuKeIWHJ&;MBBfHKU!}dv<y6VKxLgak^!s0ks0k'
            'jJn|(>uT8vn_IL4)+OLRg#mJais2dp>=-kT({KY62Sot5m={>iVaW!S7h1$kvj}pS5XrB$Y!}goJ*f9$e0vKdnSQuBzn)t0Pk$bN'
            'hO+C8kkVFgCehL|(-2pd&VhoVa*$HZgPivOkWAw!+>eMeLgj=I8S#ghH&{b1fS(p6hrlVtpFOEt!6OR{HYzqqxkE_R6i9}>>+!GS'
            'k5k7u>4(YX_#+wUlea=Z@t%p}bF0E_uowKLh;Nr?EGC5lG$$X&@Y1{h6I)VDB=%^Ak`g6z5TFt8n3qv(unHS2lZvnbMAAUuO6Syx'
            'BcNFMih5*wh=Y=uE!JtSU3yUhv&<!Hz)DWcue=209@kO0R&(rN)5;gfU^C83#1a!q5g3?k76hcrNqy(nmx@vpti?{nu^1i}Edk5-'
            'o;4ir;obw$MZ_TFJ-ov{f0?iy88c@Of&g>%K8SPR*~jsJe1ecKrys}H(9{&c>T+@iQ)5u+Dg5yY9j-a`n;L5+vtYe1ynzg9<x}=?'
            'CCV4xkh@|p1)@+swXoC!9Nm1J{5ZKjznY$Zx}4nnMFs_0rtktSX6-P9wR#J^L~;_uC46ye1WbvQUqjm>3X8rv{RjUr60)8*Hj{GB'
            'ZT6(HN(B^mjIoM=3SopHn*32CejqJ*Iw_N_E>1@3o-81L7^st4i+KDr91frHP;#Sb<J~>pJv<MY;5m}J?XK7>Ll7{P7Z!SNuCG2*'
            'R#-4_iBSfwA382*82kaT(qJ=824-o&icEi@cQltmE#fD;M~7*Pn3OuTsSMDq=6_~ho+=%H9h(ZOWgM(3AK-Kg5JC2DsJ-^b-$oL^'
            '&i)~GJ}@{}yJ)|Iu><+eh_x<XWHHwiBtQ*o;~R?c@qCgC$BW8JvC+JYIgp)l?u9Lr+rM#l$g9>YZ(vT5fjbU3grbrKQtD`gXpNNr'
            'Wuq9eQQQMjAK*Xt52ERsb8@PxURvqG_hfTkpTfe^b6+0T6YLA76D=K5GSawV-&QW9#!<CX=4r_J`NDtEkNd)@Rn}CZ?wM*bTOpy#'
            'Rjp7O-yk5f9L#j**}y-gEagxzl7F-W)0ji)Rf05>Tn3;L20LSS*3zd9*8EB=uR&im5CU&;iR>WFqE(b&i=6XM^KxE9MXYnLnx^3s'
            '>NxcCIN_8KjN=kOql4>_%Srv1WkJ&#YUjwy#3c7I7TGZShCwg|(ffgK9#Q4NI3sbe$BX7vDg;TW{ezJMc8z|`&o}CVb9%`%v#oJu'
            '!Bjz7L<7oA3W})Ax<#76%~Bj09WppIVCB74Dld?by^?n8`K|^JW;T~0A-$}gmsFH-JX25A1dm60BO{smxuYEktA72Y?~FTC=IC$>'
            '&iD=k^#t(zj#utVO}8C2ol3?%n=C;p@F|tmKK(U9RLwXqz`s>m@V<;=<s;cb#T%0U_p3I-TL0lx?g34M?odj9x_f{6cg5BF(1NxR'
            's;MJ~vbfI+6Yh9kf2flXO!{6~EKmRL_4`9CAoTryljqR(@TOFV`sw`l1HPSJ2lZ^`ziY1YFV3%Tt|u4gSL)wyQ3djn6n6X^RVjka'
            'M(=a35%L{U>AY_;h`MQ*r&25-<e`?10PkOI$<Lj!sxl1X$6%kUJ*BI8^`5C2p=El7S{`_o6PH+7QNYVc97t;*JYs<&Z%T3UG0Lf%'
            '$fazisovKMK@={-qkLn^J~HS45|#3glK5oND9|9MD2`A0fm5rwAy!ne05*|;hl_H%%WG7JGzS=xQ`aEK!zem4$q5N@N+RSwB!1Vs'
            '06`<o@-yEXP!t(?sB@vo0C~e1^d%;Fp!*k0#<2E+bx=FVr+&j<F=iqHdubZ68F1F-N%(z?kY=Vp6JDG;5_1Gq04NjzH8+tgT$jm)'
            'zzK@Z{5aUo7s1Gk98#(3fB)Tg|MoqkHHU9B!AsZ}mOIj?&y*SxGw%-tWSehXs(=fBgZ!8MexN$)16Oo5$?xbR+J1%UmT*w{V+VhO'
            '0TrofozSM4R<WkS*XGyEfZ1HD@xw6*<g1pAWJ)i$S_Zb8VrZIDx%y*hx-CU%;h6DtOQ|s)84o9wkKE1tZF2V$J&rPJa)IC6%&sf$'
            '#%!m9&&|vCldG{<mYHX&mo}G6N`6c;z?maOR}fLV;#U-HdYom98K(7H7cWTJBJR)_B?R-hR@Yt^%Gbk{cz!_QM=Eg23Ahmf0B94e'
            'G#Me1z7haba9J<85I2aZWgz)M6G0GyNw+oxw>y2cL1(lPXCMC3n@0#)TfN7Auor;E2C6Ev5)~xT{9E>{{JGx55Nm>GZQ9R$n`B!i'
            'D4A@x6>APE$ZjR!FEy|FZZ6bH$Sl2q@g&MyFv31=`trf{57Lx2=R<;I6@^r~suN4~gxq6Qs(re7haxK2V`LBp8#RZOdWS@#Jk|(T'
            '8{l&mDIhsw!d4^y8a4+XY}76&X)31~{lB^;!$z_Q)Co~MLF5PxKeQVJTKMXQx_@v9eu<E$S5dr(vZD+eH0;h9>^3?q#q@6KO_`<;'
            'qeZ(=bKZFevT|PjX%R_>2Z^2#X=vSL2nt8`f@~nv#)aXw8>qAYu1oK2(*R&)(dJDC(3B_VjSeFnf^_=xZZUFKo$)mA$n<wC)H>Ka'
            '5X7ZcDgm%V%E0f)X-;`nDkfmfr#{Q2=EEI2lgK^O4>h=@0*7F=dVLO+nMx=R9*628%`$1cNhb+TPwjCU&dYHV-z69a1-JO^H2Y)6'
            'aZocA<BWoL&^9@KD&_k9Lp3%m_2E!@crel!rxB0AXd3yaZPYX)Ydtrtbc4!ctG@+hL8W<VqBWAxO`4SXHt4cYN_?Y2>4mY!!2z5F'
            'aj_4+MheAP@;2<+fdDdQ>P_4~c~r_hih#s#>4>R*1{lxep{^eyLVHf3)nu6uWRa9&_&iNx-_9|Qr<9^Q?Tdc@nRCEL?qHG{9=ZsV'
            ')`Z!xgW0f)+3<B{yZ8;?#&1aclA%1j5oC?nJD{OH$E%G_x;&ISAGX7)SBD_$Of!*|4j29Uaeo-)QWU%vsFpxo#6=X=8I^KrzDK};'
            '-Nt1SY=cFFWTPgFG{ie52VTwLz;q`mld~IqOLvkJz-c|L*#mOOxf`z^cW%66(<c4w+;2^--fzcVokdNpA!mhiO?Adpp!Ah;qj4Nz'
            '>y7I4l5fZ)S}*vrhg8Z{buqv*vRzw>bFZCA@gXW6?WO;%!*z-@0(FFwxT<Rg_-$F#SZtM3<+I5$rPIH<)k2$r1JOTD1IA9A(h)^T'
            'j@rr><Z-{*MRKl0V8%5vo<=vw#!9@CO1pl^MG?7CdltW-1gp_Qx__`uAgOltPU90hyl*o@&GpWzvDRR}MZsa0W>?1@Ko`1O<5VmG'
            'Uu9`?V0T25VJbd`9Jm>cd<j(KY_uVG&JXy40NKy+Bi_-=+hO^?z;QE50#Jfg9PAMh)yY$Xp%cr{N&opo?a~-J+61T-Gvu`pO&nBl'
            'ZyS7V1xn99a($@pguwoRfXo$%b}}XvCY_yeXNV4oQ1E;nUoz^O`rU6@1|7lebEM+MLCRygK3aJb^~mR;bnnYECpvsuKd0<qM-Ud6'
            '$M&pJR&(6r8{iS&PGbKhl&jCMsG_$0Y0f1d*K{way1h`HxJS7|uRi&%Gc7tIh4x-NIc-E9dUJ0Y!w;?Q^XqogvYnQj(s*w6HWuk0'
            'o84)7Y$fmM>_y#NpMO^4>mSEgR}*R{vPmXTH(zFh);DQh#xV-8QAa+u^qx)z>SXwP8YRMzIhB3rKM&PM2^~ZRMw@VkMrj@@xgesd'
            'ecPSlZ4wk6$i4R^O26nd*dY6_*iZ#behVHKbp8WUaB1k3+5+9xUIP_4`WOvQt?pXK-vi}@B75JgbvI4k9!a9A`)u>Qp#sZ922DTH'
            '#TM@{qRFj3O_$*eFEa{45)jT#EX5PPM+uYXgYHNgQ?nyH&Xl=u@XBpA+Y)RZ*av{Y-<V0|(|o=MVh_3`8ZP?7j#d-3x0AgQ{j$u1'
            'SU|uWYSj1t+?zQH3A$Wm;@bo&FGK19EX$ZGe8U>S5r^#R+9lZ8!V;*9yq8wCZ1xd=r{((R$i4B%FhLhinO>ZxbSiCf2m@u3YbjqY'
            '59~6&L0aEK?2}s3dBK--ilI6u9mPTqoUf1o=XBC-u4iTXcP`Y;4&Gx38Q0l~V8{tm?!E%Vyao&Ld%hy*TwY*mey(3S;DmPGes8Gf'
            'G(=egN3ISzZA{)C6ik<?$|8$a>q4#q1d?4fFBbi&D@Fgzc40Gbc769aUC$Y&zNT53#GoiZx1qKJp%MX<+64Or1@3T*f_K<l>ln85'
            '-d1IMN9J3GhUz8>$#Fx??YwmvA~Hs(is21#w+PB^rw*$$FBnggs*%6YE4sN#segvBpio^43Js(`giT^Vs5fBYkX*JXS7lNrEQqMt'
            'ku#OgH|oON5BXvI{sxXvg=jv(mo$THNE#^Os|+lBkej!X`fubU);*|al&UVo#u@;SJFsx#oH>**%965@;p+*~|7BkAhK34<YIyqP'
            '_;PY}IT_iL0Wmx3t8jw2iK8l7R^faRk<PjTxEt{8G&n~im?M^V1ZorBC9rc(Hma|YBFeMcyQBHh-eX66w~4Mq_^zq9(%_7~BD~Nf'
            '@j6{>^+u7M@6#;jgVDoW((kJcZ8<qe9pqAtL`tr1-can2@nsSa#l_i*>txTG)Dg!W(jocI?+@ejQD<Q7Z)Xh-?DMw*#PZN&1PoZV'
            'YR(;U_I1<frA1C6<v$%^@P70!-@_P&!+uK*OV5HUjPUl;`Ss5e3l4h(V*knAhC_uKD*2WaG*lDeK{#AQU2u}8Nd^GY2|`(7dwVda'
            'G#2Z4*9CU4zzm*XF^^3V6zpaX)ZhPquO0ui*8WPJ*b<>zHXf@JNuC*edIOQQipHLs-@+sKUP}|}?Av?W&Y=3Ej`5vi=>Bz1SJp9f'
            '>B&iPwMww0F1bn15O1HH;QQRB$8)FkUEyY1rs876t%t$k8_@JdZWhorCZN8n{g`2xq>sM+o?%!PA$8VLpaZgR-)-e^5rB#OFC|&6'
            'rp%GgV5oEJc#qqnF{MqF!<mEp=9ex<i<_>#n*Ac?pk!#23i7#Ax<x5ZP**7PMVY)cteG2{K~ZP?EUxyw;HtJkA<1VR^n=*L$+LBu'
            '>KisLm7DmFU6|`=)?$3;0J=U%>9=XgJrsV;KMSpjThs4=xJ<IpESp}UX1D}3D>!rEM&*Q1)f(@J@Y`<0)}I08>h|vZYRpP-`yPBv'
            'maO1-rBRg_brDwIHKZe}^Q+q%y1ry{ngyj?;$rou8)}vE#}IN!h_4pVRRQ<%gv+g*a>0MO+B!XzhTj*ykdT(Ewev}oeAI>Cx4DkV'
            '3`Ma<c+ILn?ice%7miz-%SLi{*ZJK5%m~w~TfE)PSBkn0MW?$+>)m?|sH$oT#??s9ol%%c-QEIgd3w@E(b4zXT}vU~>$yu+Tr_&6'
            'm1`ZG6STmud0ub~URouI(0*~-tuo}WNrC91jw~m2Ky~KFEv8n8lEgu)xyelVm=6^c3V`S<V&NV^KcHf&ck~$A6=tK4ZGHHSjqBIj'
            '(w$b3mqD_LT3OXv=fO@wFCU(O%=3#@K$ne$T{j`L8tO;}U(=MyEPfncARU?f*T^zRN?5?ZCA}gO&bJ3i`qJ}l?vR{4SuUzv4miP^'
            'KC`J_o#hSHXOpp+>=R|DA@yhU_>KD2!H6=khzd@xZ|+q68Lep{3GR)WxW}y(LyobtYHHW%24J=RioyA4z!^{1_5axg;%nNiYs6Z?'
            'f~ulXEtzr-4F_)N=(JXer_D=MB4z6BF7ffA@z{a;E2t;X-GaP>&2uKJTu(EQ5mAQEa1lq1>&130PLeu{!FH<jV)M6tYumgp1oBrK'
            'z#q-W(1r79srv2w?&7EMrCf^?R^__O&8Iu(R!dY=4N|^di8?2@of2+p7@{VHmM_J{a|rX}6n0%*5VWo$^{@RHQOw>s98nX@$><_7'
            'b!+XR&a5HNnYooo-`2}rI_GP8{ydZ%m-{VhCQ)QYA#2u*WBh?pP@NXZ64GJhj};~5__l6LQX=Di5y^*92cO~}?RUSkHM|4+QS<9_'
            '>F-uwxN@JpO6A1q3=GEf?I+*BKImyST7WON8=YZwyV2IVRnKux_mhtG9A9f?SoCU#z?XZKsyYMwCv-C=*B9ff@h@Zb^ZDn?Nuxq`'
            'dKtVy|A|hAk8sw1p^sqzMRa(yZ$Sr%PVeHNLaywjvqH9S@4be*w2M_c?66pM%)uMEHq<rde^;y?5nX37`dW4SYyz&hIQ9K8-yUkT'
            '8m<*qoB!rP(3&m^O#vfQu5HZ~`eq?pl)G5p3vScj4;8dtqzQ@*WCo!gF(q<#{=%c``qrOGQ1llb<W$(N|1W@6T}_72my@kU`k0sm'
            'V+XNlKH8GA5iX78<0&~}q0=JgKM=_!!>F#39DXO@4XQ6u&g>@;T^|lqn=#*I&`y0B)Hnz|x}o|7Nw(knA1&#y3j'
        ),
    },
    'google_ai': {
        'sha256': 'e82125022c0237de8bc42f08ee360f4edd0c6264a5cb32a28d46ece49c8cb0e6',
        'encoding': 'zlib+b85',
        'data': (
            'c-ri}dw<(Fb};<EKLx6Jo?4udWIM@ZXPwo~tJq4S#+R%uPj=Sh(v(EmtSC}TQg)QA``O=f0YHEkS;-`Qw*7T}?M5a65I8tEH=IkO'
            '(Kzs;I10vI*b<qS&f4C+m-ym!9N&b#=nX{kwLcG{K<o(kG4~vi#LLW2#57Jsb{nK(k;MPukF(C!*84ZVis75Vxi~&Oyg2HM{$B^@'
            '!}G0n<$qf*eHg;$G3N1P37_-t{A4AfIP*twJQJg35N0CCQW4)r;`+vqyr6Rs27Z(^JDrYmExc&5b?YUQaXj%S;u?nW!|wJC&A9!v'
            '^SnI`z4UhbS_DxTM1DsMZ@o;+y@?OQieSEo(=-^tm$O(jVQsT0zTdjP4aT>E#!6v&zL+Myua+ssUL?G6w)DbqB@*8o-?l_^1YLc9'
            'p<B1Jz)!Z$-W~4@;s4Lx9l}~BH+}{$<GC35L39I8k_aX}S_v9)986$cUMQws)CLf@aB-qJcHlt}!UBZ1wMhI)FwQ)fF}z+*0{SWL'
            'y%nx3hhxC=K0WD+Md*zY$X>R;wI#$}M~q?8nV$+zj0u#YnfmZ7YX{N9U-%JDqb1bO@d}>f!u^Q@{hxGjFL-!@5P;w&NHf^BAH+D0'
            'roqh;CYb|4{Cyc`9t=12mSL8{5KlW|5~Sl8_BsV<F1%3?2GBGWcY!Cac~Tt)e<y^oo4nBg0>C?%Yet`+i~AtEh4J7&0HFV25e8$}'
            'RM?mJ9=pM-XC1K!#xq>&#D~s^7HJlY8HdFJriVBWEr0CL24YKinnEjrLjQa?IPMMmhjJOA54)H^`{x~4c$VO-cf`oUZ^Y9n{Dx;V'
            '$~Z{IOPJ0Gpgp55f9QxiFAOFGe1O;zkO)BqQ&}uCk*;9(#~GePK&;-$p-BC?7s&_E|EG=!<C|0-5<tg0z?ejMS?13d8A2zfZSj&U'
            'n8HZV<&PbadJzEkqp$k;NdgCV#AgrB03Kk3EKDTPYoBb1z1<E1HBJdtQh(tkF!PoMG)yw`=Fn{%16hFWL1;>{p?7t)wRIv-Ga%<O'
            '&G2vo<l1obLVu3f=S3?39nX9ONP(E_ujp(hnMoah!2Hrb8@w98ZuDOF1}Eo3ar)}j;9$@@60c8BUjxMfdUbeuaB<u}8TN*Q(-XvP'
            'U!3)OhsS-<c<ZeqZzd-3O|lHVxz})NXw#Sfx78NMbSwskDMH5Q6Mvg!i*$cy2k6%gfv7X_?{*}e@)p6)Bp#<bjPo!=;Sd1f^AY$i'
            '95BSoCet|}fh{2)rTQ;8fn6ewEkJ<6K>mE>PxL67gw`H9BF{dI1JhgLZozi*{M$r~MY>D?)qFhG%?;NW+~8824t$g*dky4x1qg{J'
            'FZ6)SA%%nS02_fCWUGarcD_H5P2&-gq8Ap^Sa>Po=;7dCcrMZykvh%1RVq;kDdd)*WzOUAj0q-C4mfs%bmxa};p{+<5Ds?-fe(X6'
            'Goa9J2uCKoM%wHzfao-EC37!qh<}I%5E=RxPC&4n)6d)B=C&c4a1Po8NC$UdGpMLf!sic8&(8V>guHiYho-1l&j{$vbRQ-N2Obdu'
            '$j5aH{)`vyLj9OAQ^c)@e~H?_hA*RxxAWw52*+;@=WN25iR1U)4R!zw9dSCH!qEmEbhCt>5ikxQ6EKz(0iF_LiGd1wcYqoBEPx0R'
            'o0e1_xXiB2kl-r`ksKH-#k0Q>Cw(~0;=F%yc+TV;1q$_{&IiSR=>T85C<;tm^B^ET5QLds10N1xyO=W(pte)u7(imwb|azyaRlD$'
            '0aFcAAGuuOkG(92qt0fqmpk{Q^CQa^ZJ=`gH39}f8xrw^4*|0liW;8#UIH_`0j2{x0$*pwx#5A4=Pji{Xu*u*@{4d_aGE^0KWQgE'
            'w`3Ogd5a?QfHQ1AxrQU}^B2<P0?!em6f#Q}kL2|WVw`kvkdXoWQlJcKHZVT2q|E=2ZB7~V*Hedh0Dl4YXBvEf)xtoqRdIrXp8SmB'
            'bTkd$qtp9J3m&HQ6)j<K5I58jSS&}dRk!(2ID;4ZmsrGrI)Li%QD7<O8Hidjkr1F0AUK4<9MK&|Yj_Lbhd#4E3D*{JIeY*o3*?Gb'
            'gZiQn1_L&5LL^S40(UFJ(=!hy5s;5eyz|BZ972%=bHA}U@bFZ$d0GIyKC*hGDL1e_+&B=(`$z6s|1}8YXTM%c0g;a+9EO9_lUIY+'
            '7iYbfz(WuFuX-0pLs$e#3?!T`;rt}P&4C5evE3KTMHqXNw51`5KT(cgVFBgI7W|t4F(Kh2e+Z{Mi4ltF*2vH9eLqs?1n4Xhs&q$!'
            'k!Z~%28j#06TmuKjYeaO=$7kFmq>bCSBVN<6ydmtid$Rq*~m-%=g-t%w{Tv=V5I*1k2H?d?>JSz64<JEuK)7YFGh6r7j6$v1Su^3'
            'rKT|<DM>wTQA;@Pna4jr#*yr}@G_i@>h})*+}iq1d`)5GJMjjTBfw^m*j6G1Iyi=%qSKZ7guXDdU_>+(NHP=aHnBYvB>>rBI|!R$'
            'sOd7IgAV;}NEm=OX-rrxreUyHAiBZo5V<^%<yo;x<7EPTWPIyIH>`({Nbw!&4S@YA4w!<V?2}>z#J=;DVX*bxzBnVqK%L9RxZc6R'
            '>BY(LeCOTS>A&?+egKS<XR@Tiz_REE4oK4d{0@MCoI?zV9D@SDO4~Iv8aIkviPJ9ibQ6i7C|tqeTcX?oL<R;-mXQdiC=4S65uCui'
            'vW_@FxZ%v{1fL_N1F4_D9D?En7=`pm(eoy7YZxr_P(kPI#=fv9P=JEz6iz&lHCXH~gTwxrgOi?uTFly23Y?U8N2&P80y<m0qodRJ'
            't^}3)my6S3&pkVp@^x3dLPknZ&Pi{&9L)pT+f+$+uvXG_CFEgBfcK!okZceT>6|R_%#)zzt#Cl-2HF#KyIq2g0e{^E{yh@%$H^k9'
            'dzu#FXqsD?!s9)VH6PW`gmgK21?V0C%0j;p?AE-q_3EtOcZY-inS0oS#RCHw9Q0x7P0~IaQfNV1zeF0B1|P-D%V5I$z!QNhCqW1}'
            '2V*9|42BgzwlaFOLgg&<wBFR<0W(mwU|&E2gQm+Ugnr`2N7*7s=00%UF>tNfG7O=6NQzd1!vbzVkDK@-sO@T~8y`^B$039<YCKy;'
            'vS0I201n~HQ#IbLHwyzGq;mmOZRM?k;*veN<hj2JqE2JW;RER1i{?09tP*5${=`vq0d?4T$0PoV5)BE3%v=~>q)7P!P(Gw>3}e=i'
            '5v(Z1W<aphF=*W==K<-yM~yrML4j2Bxu1DJi#;NCDe5>9pU-;l-Q)h@pyv)xU1Nip`Q><h4v2nn?*7s{y6B(7<4+6+(1PLnbF$bp'
            'TB0#Qni((fflDTr+c<<7;wwg4H}U_zgsqq~T08>C4T$~|YyvyCB+(nY<8<*Mntk}SJR5MFTs#je7~%nDcJQWu+;ah7=X?_Pwzjs$'
            'Ad{vNoqB_JMELut@;A?b48ery4`-j}&}f`_0UpfzTR<64+?KcnZpvK{BSo2Xkoj(H!H$T>37lyES)2fw(qDQ7%SmDPmJ`@KBz4?z'
            '2gD<xx`}&_?GysE%Ms~g)ct^zGa;!Hs5xjcxb{EzW1v$mqpwRu3n(jbr6|sA5n)KY1*i!c{%An7rjdD(mX4?asyY(YjE!=7LZ}zW'
            '1#riySVs?ewfJ9xfUVpJ-Gs(nH(Ga?EPWt@L4qfG>U|77wJKoXaDZ7-N&|U1^`LOlQ#|QFOubt;_dv;`2~eE`hzCfYJ5n@eITpza'
            'kS{NtdowXZPFEN$xDC;6#YASbWNrgYXyv8e3`kN3_$%>?l@~_A$FV1Rzn%=Z8PTeVUmTLS864poqG>*nqE6lhh{ji!gw9F+1mWiw'
            'FI@U8yx%8q3SqlRi~&3xlMIQNB5h(V)Jo@QVw<4k3d5k#*w%tAn{TD<_FG6U{SU~o4&VTe<O;~@#%eC`RFKa@q6r|MBFO_OJY1$^'
            'k-Ue)+#Z7>2AGUC5M+U200J>I3Xqqkakyk82mDGpTj%|Qv;NTS9SwTtCasg_3wT0fUo<7Vdkb0(kS4afxxo9}xBXumjwEg?K#1kY'
            'Ct9XE{>l%r9rXP5^kR7Nvd=8F*dNr{zw^h=PXD6cw$$q}k`Yk9LHt+rCzw$CgQL@n!&gVWv%Wr4gTv~MpwzfoJoBSc$BW?`;0ka1'
            'CwWKjBkCII!uQ9wE^2hxr}wuDXny{#-#>U`Pta<P7V<bM_ds>*l)nMwTyh>4_TaF8G92`dToBY=4USOKXh=5L*isMOliqP3hJcre'
            '-@$DLZqZB{|9;tP|IKTE+-?8yD*x$r+E<@;ThE_9{GSFKu7lH~-b+{R_nZFFySmNRC$hmP2q*)3OXLG@n+2M>H09ZH(6(~}^au%Q'
            'Fav4@Okn#5@v9H?<tF$TnuMw$dv9Z--eg~FYsA_X%{zbP2|vY;OyG@$W!p|m9QrqYvTqKT3xp}D>q@Et(Col!kj;3L1Sf~Y)C(uU'
            '4RQgVM8_OyTbZl{9CKJvu&tVrn0Ij8`xPX;XbIvDoOh6W7I3assYmA#F}`!-XTSlCAsIk0?khCY5Fgl4U+Im<pa;=2lsPpaTz8g='
            '`yw|$gEBvAC0JISJA|k)TM1m$0Aqw89vTCj_!aoT5I!IdQMN+K_wK0%(xbo}gBTo#bAVdD)CVoR9DtC7ij^0h`BN=E^q63w#K)Ed'
            'VB9fq*gK@=iodgxvCNd|LSL$Z$dTU0_mPw*t`mQefT*x@4O%_@%VeJQGL!*G+$18aE>NWHrzK4}zO+TNR6t6D__y=Z6U`g=k}gHn'
            'YphqKI85k!GrqAw^NLhVT4Xd7c*`U3wvucmbhU-{XC%}>_*0M6Xb^U8Zjo>Yk(aC#mjY!k7gy+q>!x9xrLK$WN@E7dkF=NgfQ!)5'
            'fBC74*I7#5oB}n$pDw`{X$FplqMz%+Dac&6nff8<yEx=N8b?|pAC(@6e~<v!64v-F-m#V?572+|tK7FM>gl1AOIbLF4o(3&;6L_@'
            'd_wR#|D&RjnjN&uK>P5U)>FssAtwrL)qS3u@31T1Hs!yCz6?+93je|eBa9fj{~vS#ts+}#Smzh21Z@(k;VgxQn910R51YmZoVpR|'
            'Z9Z7OP5F?3J7|4PKxN~T))A+87_Oxao*4w+7~z3|gS7-XnlyFPpF2ucHs)tcEypA}Pu5&Y>uTf>XrSy(;~3Y>J{s{GWNV0S8u`#!'
            '?2=(}?BZ=w{tf)gTIsnu-7S3#+o?B2!x$@-WZyZ6frx;TCQeQj^+CG}9`vf!ezJvKI(7RwuG+ZYATX%+3=^l&L><U|3jqs_j6=O&'
            '#Q&<MfZfuxPd(_;uY4kmsXUlO04b;>0($p^_zq67cQg0)Q30bXe$mD&@jHN25lYQz<4+c?rxV=$q~Y|y7$ToY$UGEO3H)!W03@5r'
            'zaIF^%ZE8I9{#N88BoV8A=;=GWG<13s-|=x&5gfssl;vwR8oI-r99^{Hn|`$06H~Mv?ux}Nu1(EX~&Q5f&@?zB@1k&2HJoE@qpF|'
            '1DyRuyCQgO!ylua7gZZVZ$;xkxbQ>ftHcrUl5XxrOJHTN9-oX7`Uoi6hD=or0PPzM2X+q*Q`1p8H1fcrB-@!TLl6qG@okf7SW%fK'
            '#jQ$*Hu$Uh>|Xo%Gmy*@P*7oGSlB@IRja7CUIKdx@4b0TY<ITdf81?BqSAJIn~ZgHKMYKr)YuV8$!H6tT%y1(x3p<6Msp+dU+F$k'
            'qY6ft|7T%G7s)j3W3_~I#`8!)UH~elq!*qHh~^+#1vhBh0HFgd9lVPu7|}lB4Hi#K%1Cs5t8M-)Q1R6w;tXuyEnj#MxKSqJCh~4a'
            'cvA$tytknN$@8QmuGRYS;(w7!<g*bjRg+9}l~)Qwq^n9EPOKVDK^5w*k~%;v3}ZH-XY#V*9Plk)tZCiRp!f{dF0mP8Te>e9>(`)W'
            ';|4o8aXm0KjdC2&yYJn3Xv-zBA|;yK#A7sb=*Xe4DD=QM;-?44sXrgn2AWi&xxWHkr+hs2#V0kRe@Px*fLwVuUS{xA&*GuMqZ_=o'
            't7m<wI$u#61?euDcvbV3gOD+xV!UF4M7*1&t^Az@Q#CN{{Hj?1vnR)ax71kC7y_6_CO+=Jj*v_nCM4tv1GccEn)nf|Z{o%WasZDd'
            'I(ied$0z;_=mSrfPbhAkd17&D)s&t((L8aA#$5BMX31*bq790=N?t)7aui@CLG+cbP~De=P4zI4;-KABQaFpo5HXX2d@Hviu_Uk1'
            'qN@ODQOc@uEVV%WSsukEuVu3OWz><b&f*(%t5f@_)G4~trwtk3g1D!Y#WIg}=muT+MT#aX;n$}AkN!9|-@YabA{`h36fzT&7)YP*'
            'u=O#&l43f_x5BBo)_OO5;u@!j!i?w%1CfZT5o)K5RFPOlsh{nH@pzUi%y{WfzMtfv(!fJAMRP<$`to{New_8~Klw5yUV3X&%KXgR'
            'Rzng{aF41E^8SVdOKr<CSprSM(eP|GM{wZ$$d8bEF@?0h?L0XDdmO*y@TsGwC73a2jT0AtBeplyj|y?7Bv<w(6Y}=ZRuje8>4xhA'
            'dTXK29$vC1cf^%%=_=bB>$^VkjOhGMo}71EXG$n3oNB4`9OnU_Mb8Wk0}H;uJ!r0J=FcBGnv*=pr?lb+b!WyJ<;a>nD>eCpeS6DF'
            '(A_rpaZ}q%oo%rrwxt$qJZ}lxa0*j^<1-1IZAscr=!}n*c8gm<NL0G>%4cY3-(}!6%;|kHIvL8B)370^r~L_K@Xmk4LDVev)XM0C'
            'UV?(E0nA%JT)60SmNN+(fa+8ft>2nxwTkvsvURh?I0$c0GEhcNfyAiO_!c)+9O1xz`Y3xhD}3=^pw@XCOmJpwiXpN!TB7kiO_{?v'
            'U;RePyk5>R^_aJ?>WUeibLCD|nU88Qza>}YqD3Z7R?Ob@Ie3X<A)T?jk!3ssc~@cVgMmg;r}1WneuR?1we{6&wC794M+Oni2E`(d'
            'Lm588&T~whq#M77aS&dLCR!k1L0jHRNzWmDo1EV-`X7)4N}m|y{xqDF0NCgCu{-u_M+qo4sG8m)z$+^D0=EPeban*EBb7JgFy#>a'
            'pLZfZyN{EZkkbgFJF>Kw$Us2?&*9rD(5)h;AFdXantKIKzzk<3blbKeV*ZI};>Gs@wD-H^$K?)!B3%JsfwpEO9nAfYPN%4}%b7n$'
            '#EcvJGH!3#1}Eh}_HK|<gjGE2h)<OS$@nx0M-LTKw0BY6DcH7INx^D1-1v$i3(#?35G!9pJ}%-r`8Qxj=zR>(77$AWiF{I45j#xM'
            'X@y|be^wf~l<2MwJN?sT0i@!1w~QS!&%1`r1~4fzEOINdJ8ZtujT2ve8;FH5C|=Zsli-EC)>Jp4=vX!P(8UTJ!RO;Z%$6K$^08B6'
            'N?K1AOMtTf{1)uz*b3_c76Q+%lg0dc=Q^YRO%@XlCa7{7J|{Vvq5!2=;&#dbbm$qVd`Xv4fZ<;h?KVcIjB}6%(R$>ki#Vc?qX>`^'
            '8X5lA((C#jG&9hD*omHEz`heOrS2}d7BA-T1M!62h{E88tU*#O@|IaV_jCl<SLcsC^0Rk`{lB6MS)&-Z(@+i0hi4ZD!wZl{u~{Ar'
            '=5jEYZuk1^^y1`@TS*jQ8=R}733S`x5PdcpS(fTY{o|M1*9!d6uY0dS8phY&4Px8aRlx&FxIzzxar65Y20WrOXA9xm5*hvvfV?H_'
            'P?@ybbXu^Zb08(%2D=hAOarIsHP<BvJ3<IQ?KT?lkwwrggUmLCe=__BcutLdb!EqY8y6OaIO@X0p(Reg@^wb9v%1%?Sn6$Dq%;a^'
            '*YhYmH|N8kftLKkcw_k!?(=<~`OF-JAlBGt$QjSsHvs<Ec*Y_FJ6pobhCGw5u{Pye!oBbw-i{W~z9Dp0L~mbUpi}$rOAiD5q?mf='
            'O92Q)`Nxil^h~vbbYXUkZn;U6H?-R{PE|{kPLO`24vPURBnLkd!ginhxa;oj?h@j5pX}ysN;s_JyjmRQ)zVR3Eg9q`zo5O9)KY%@'
            '@4gh?;+JxZK?+>XeKS^-q{K}qVI^EBR_dDpXRiu6Kkj~;o!<KzEBdcr0PpX?zP!n4tFSSl!^Tv&|C7#-M#LJC5l2+-?~{K*yk83p'
            '{7<r-x)43O0FZ&g(VUI57_(OqlGkZe;>eS)gY`um+5J|aC<wlX7k}#?4UhZ#h#K>t6un51j^s=yem#BLLcieE(uh3J11S`9jmX3d'
            '-IF~uLZY-Bn4)n7x*qNelx>;|_M}IX4&FVEgIN}cf_J?mHlauA^n$F^mZNU4LBsQwLlRK;DR07_u?Ks`9#~=y`f%6CHZH|IG)hRR'
            '7D|wQAs32E{^;dVFnY<4;-4H2SNlI^m5dpHoq5#gmr3##%!~lo$00?+wSv{&t~{7eO=!{2d;o-wEr~yJ2z9(IXjkXM{@bJe$#MU0'
            'k_s8|vY>{>055^TV+lIostDY|S)BV>gwDC!IPkvcbUNZ6_`lm|3!4S25FI2w7M<}c08go4L^EE%d6$9OF$dD-!BCFa4DfTW{jB|T'
            'Z`2O54XLEUPYLq9=LX21xzF&op@hu6GVUoo(@n>>{@k<dM=!{aQ$aAGbSkh}9RA>~5(-&h7?lnlt>ww{QQLd=-+n#3)2~3I0NwB='
            '@$OLCx0I5Wh!>bPhFpW{w_Ccz=c8zfwEJWG$?lWqkAsOcRRyQ#3QoHgWin2hf|l{kMyrBg6~Ty?3#$~koA)1{MC{-N9EnAMrV6G|'
            'H(r9n!WkpJcjqOkw+cW?5N~_$dPjpwDE`e);x-$D>6`?ql~ib7np@DbMF*pxg(Xk6A_v3&`hPt*x<I2qQ@JAhAI9O5LTNGfR^<h9'
            'TG}UPiQ!wKT`#}EM828yCbvs&)NTptAUfzELKR!IOOZ$~M9lV@83_$5PxQ`@dg2DdAV1ddJ>>2%%4hKPI>BrG<SW?U7p&*er7#Nz'
            '8UM!rJZkb7)PZ9xqio5+s372dy;&3Z|2xe%7nT0mXZ=7yVZ&R$BHvY~E{%^&Fa@Mxa5C(lo%atgp%q(B_lX$25S|q4*=LY!u`2n~'
            'h!W*^+WJoMlG?(Gce)*EF``(;qrv%5yV2X0By}-x>!ofKM<0PG@|_kZG;#yu(l&Q<$E|<??V;$(R`s%Od9Zoua)5(*O^0-7by>6z'
            'UWq@$R#2Lc^6Lq|mP-gySKE>~^3+wf=CzBVNEQY)2F7Dbr7FWTJKLfI9f4i%F|Y^B40bCM%pj5x7}um;(~cB!<!W|#8CWp%;mYgx'
            '=3Cq~3^BzEtvD6UckpdrJasza6rDU(7P~2|9vwW{^%GGwT~f%@bQy|ikfa&K_+p6X@bukzp1^>!$jm+Dt6~u>@FrimS53Vr#e?zH'
            'ZvcP=4*PBoeNB49!7r31pls`GRTvFhbq2FmNlbR+XI{E5FOJ^YpQz|sVX|sfimMJl0|m17?oaK#pYXam1XUsk<7~Mvio{Z;Tg>T<'
            '4E23|dUkPg@$2as3<jDUjB=9%;@R#qj6z<n;sh_*8&NFKFBh?2d%k8d?9M%f(WM>nF7y!}mJ4+Gfw5Nc5`9%B)DGH6#bc)gTufOo'
            '<F_z<yh=uL3c@NsgQ5I6`fjdR{G~`h#C=bADNJ}3hz$M#&Mx9`vP^Kwq+l(hWfCrD(4UqhGcSZdt|vGkg)@g8@lsnta_btUK=i~4'
            'h&_C`Be(Csyx-XnOVrjM%KM(?daFk35qp3duCMQknpMhhOSOBYFO8aOC3@(vHSC?gb*Y;XOne(>^O({kX!)#oWF&Lu_#2h;iz8@j'
            'nqs6PDK+EOqr=L33$jXN_02}haORS`3#MbTzt6``HoOZG56B}DasWcMrlrb7QMDwiVV=?gX5f3#a#7u`WLneBa?DaglI_Pp9r-*r'
            'BIwr=6<Q^=t^65@4N5L~=O^tZnig7o(j0A#6l9I;h~5ZuaER`o(YL@hRU*0P1RaDo!#9JIw}X?{?ot02OnkErfj(7TCK(3Liu)P;'
            'wm$r#_WFV>st;?h^xMp5!wFTMg>N*Y4X0E*a^GM!n@yzpP=BiLY+SF@UH#}b59<lSSN=*hG5N|LsU$>S`5RS4tQtY&MMf2N$W4~&'
            '5ek(cb)z7dXQJglF=jkc0yyTQ)v26iETkkqXZ^oi49@z8E^B;i$&aZN))R$ttICsFZdN=!<rdXwU2a!{ndOGn$W?Avi67-=Ra;yz'
            '8Ch(~xQeLYQkMCBEt9g0hXv$NzUHhPw<!^b82cE6a3gb<QXV^yWqa>?zlxVQPfqOUVZv>hw=KP;RH)dSGe9GjMHo05lKc2R+WVwU'
            'oN^Z_hA8uaBhy<br&U(RG3CJA2kF}TAl-pDO-XvqdLX)9%ESU0rHlj|=WLy%{^ES*apbQAYFPWIL1rMjjwvb+M<;9WJeagOfkBJ%'
            'i>J!D9=4iO9E{_Xk~yCD4^B@GDMy|E_oa_^21=<WgBX-GAWtW0r4%FtikFxwkfH%NM>Qv@!<&lCOk=04*fJhPWdOy5HUTC5wuK4G'
            '3YDZPTU?q2VMq?gRzC1D9J?}|I3)~A<$*u${nZ7T=H11RQ@@-WZZDMHNc4*CPF?0n`!b{m1Z9jE#@L|+YMjmg-1~8NR}9b4Yhscr'
            '=i2WC1sefup3WrVIK0Mm2?|-V+My(Ms8p$JJ{5@5sij7H&)ug#JzGPFtOk`d$ZAm}KjupXnwMqJFqOvxk|)n3^TfhiQR+}Kadg%Z'
            'u%Fz$KR#Jcy38ikM9XYZNv@0rYB>k{x#|WzDB&Qcl;*{7npi+B&Na$jib14P&2{RDp8VE>G$NHS6GP6Md^e{|N0^89A9f}+v}zdk'
            '9!a!h*dw|d3zorUatTiErjHZaj-ptFxf2a+0;M&s6M>izDY=PtWL#|m=Aw0}`Ye&P4!@2zBo&Q=tz>+PG#?I!{o{9})jpFDva!kr'
            'PfL^yZN+>R6!?b|<3nUX%^tZ$R!(JfBMXu$Ymj0pannN|uZNV-pi+YaEot(dD1$dYuKY<%T?ZX<aIwnM6;-;9hE@xseCsWpG)BqP'
            'sbk7Eu$yuqvAKUz6`QIbY$>H)C5OkBXgV^Cmy?)${YaQxdQ&AMu+GHXi-IZg4rwxc^7R8<awI(&yy~A1U5pmPL<NQ?ZZ2E8YDOJQ'
            '57#L79uAM(H>Ve8h}%z|b(9s^od$k5X#yi|QH*;_9mztt%V99i+HC5i9>8tl*>XWg9`n~TCr?#sq98E?Oj8*qf`f=xgR}FYvg8Ac'
            '2ci{>_pRLfI-?w-@-@5kr`J2z9T4E>^ke3)(rXR_&9WpIq4}01lIt5J%~{ga_tBK%Ky33M+s-xcLQEEuVl-Vw1yl&_*qA*+iivde'
            'HlC=Q-l{f)CQstdN7wgKYzn}j!quOLmrzwrjSv<BJ%#C)I>9TQZbNU3V@E^qYRmhlLm*fb1kuOkLr3wHiFaR$>?;~m6G^r>+g}*W'
            'e6d|)@Z8o^h4LhH6<Wb(KwZLT4NgLU8v~!V?m;pc(cDQvUV=lu({L`^dsjMP2d@KX<!VE%idNwH*l``-Zy4HPXXAkd9Q003PX-6Q'
            'BXMwg3<u|MpYsuuWLtBy#aC1;ROXe4er9}Eg#XiaJj2jJ?z_D&E`v#W1rI18#kTt4VqnhpLkG@@qBLHJ>=P%z(UO=ro>CU8+cGBh'
            'd;>%G`FzFSZtDxP`&+4alB`N9rwp-@wQT4fj<eaTOCHfIq*^M|86Ep|0H<s@C)r=*au3gAzigpLfnKM?#W~r>$O>}S`PRBMkvzqn'
            'HaD4(Oh!T1gxX+mIkYGpyibjI2Evkz4`}cj;K3zqT7&kgK^(scogMYam06aIWH7GE3-wT5oMI=mJQ+>?srt0eIiti}C(XpjI18{3'
            'PR$b7Eg)Y({#9cod_L?|;IoIVszCBupxK^YwGgzMX=uuDZH0359N}y2X;o)6V=29_N+Z|_q$(mo%h>%z6lJZQ()cgm14+N+aG)cj'
            'R2y+lBNrJc$6=^k(3^gsFo=ti9Qxswb+J~p=HqPor6|7NoS2R{WY$h;Xqd;e2&1G0MFm+2F{#yc;ky5tzMsSk@iF&vl5T<RKg&Ea'
            '^98C_R2{@m+L>|jOSBTbowQiJOp(p#WR2yWA%~l590^r!q-w{5lff~pH=Tnf#YvyN8;okt#OcY=uR3CPkp!q@O9BqU1y*oV#Z0Is'
            'QF-%>uwb13{cRlbV6b7pBW_fh#~hgBTYo&$;l6l2skDf<+D<V6v#f7|G8zr5W`<GR7{o?(PI#|C*0O0dY&u{h{x7stWhOfCcT<&@'
            '=q$6bRuoF;UKWya(Ym_{qkkkMaZ}{Aj!b@(ODqIVUiDN>8e9F%obYW<;Y;v(s5_1u5b&)Mt09J=S8=Ikc;(2fD!HFWS6t4kriGh2'
            'FFS8sDQ*!i(dYnp=Wafl%u2O(#M~pKFqrunshL(dkvKdYpX~J`RQHw^n!(gS0#1=H?omjS)sXI}rW_+MokcdOYL(GATXIAkglbrU'
            '?t#80oIP7wDzzP8n~PBDfKsdx0;e8H7^(SQB!^=&<jCiQvpb+yMVNIho=!QZD~RkjH=KfJ8DSVP2<tM^UN6j5&jC;r3TR*qApMnP'
            '1}OK+&r2n^i_Ti>LCzTJjxd)Hp1KE}EXVXDQYp(oUov*n5&u9yHF+A2nYgveIB}^)ywdH=ujj9|Z8lG$sWMr*c~~fCv?Nb1_3)}F'
            '!;~UKs~yAYdu7$Vvh}?<p0wJ_tkA0v!O4BBhs7S{z*GYg#g`?B=onRQFaInFj&{5%x3wM?d(^qhLGMkR>59|LNf-=ZVxl2q+dWj+'
            'EYFqyx+-Nil`@Hi=xmE)$Gkw=IN7g-Z6Zy}c);;0%C=XyNW8?cuFRakl$ir0h*?0T2)d-^T;@%RCyl02%n?PYM^H~=;3;=-m)~e0'
            'w)FH`k~Fj`$cJMM9!%BvCBdP%iVjv&h$aPVrFNLMIJxf2>sDPqF-J7^lq#s;_b~vQewgYO?G^*IOcWRz^4cm-a!Zek_{41)D4ai+'
            'H>N;lc(&Mpe-%fO;FVOUv7ZCHob-2z*ogGp%2&lHjiXr3g{6>22~RacGgsZ<@DJR{ls|75i7^7hRsC3;0!w+uH;mKedKrb9-*<vE'
            'R$0=`DEKUNZ-EZsTI$W76+`FvW+=w_!^$2?;xgDFKf0=Kt|YQzAAW9j%tcR3v2SEGDP)r_WwxwMgHtC8(;k0PiJ{G!3#~jKbYmG|'
            'IqIlkN}vtRT@^MLxVclki&Z8FGy6cb*a+?GURCwZ#Rs!v{?zDWBDW!+OF_YCs9TtlfQf$?BFAI^PaFNPb2ym#oRZefED}|@l@R;@'
            'DXKwao(nYcx#(NX)fl)^7+PAYDufyR^5bq7jw-frHK{8^0<QebkI<`C9{WJ7U=KsXX!rxgDTP?{H3>otS+1((Bg2*EAr`YhpHwVC'
            'mb{RzzJ**XRCsLU-GtC%j0J$oQGt>gA(N=EpX6ekoc3jDLy&A^S7i;dU)xaFO#iIbCT;qw=F?R_T7u0qmHL0HI&+WetJ58A%6uhv'
            'Sj&T}ZZlC=Q<6ZX%2v*wukiy&S1PkvZz6jW*SBV?zE}{bE0O>zTwTR8(fm{xM&<vj*A$iTP+m+Wg71oEF>Hq8%0K2`DngCnUjBe('
            ')#Z3UuA{Y*T)IL-V}3pJQ+zEG26!0NZrzK^3sS!IXS8ua>;-y~-Zs?}nGsyu^#CaqcijiQG4_-zfJ_})h%J#I)slfsjI0fNbct(o'
            'c4tkb4162oc+?E&Vm$vN&&0(Lu)mT^1+6twf%Ax6UR$@&IVu-Fa$;gH=E~VxpL(^lM9V(b$vy_rG_JRg@tpGZiBFaOn%u8jHotw2'
            'sZJF|z}9W!D~|Y6H?JYvB%pQ)7n%TMM_<sz-oyo=hE3~LHg&+XWtgQv2NFd=*BjWkKHqr9<H0tKQS2t<{xTc-2G;oelx#ZY`or`E'
            '>vgS?HRYpeo8-MQ)|<}Hdg4?4*K4^kJCAr3T+7tCfi5|Ows=%Whey%?o%mQrHlD)INL5`L`(c=S2PRk~I?d#jBpI*u<?&`^ZMN#c'
            'fLN}s#7&vls3)|hsLxi;U(LL`2HHAer#3}#t-pp&S8UnLEEFZnA7$)Cij0eN*5rL2^FDT|5@3g1*_zZDohCTjqaTf9bXDj!mf5uZ'
            'Q^Rp8MsRa@H=F#3X{VuvNCzGE0n%V}9>G-T^B9~B<-S>*PS>pJQ2D7c2jemLOv3*gjH|1EndUzm6d)Diza$GbZiB2`;YH<mq~;qY'
            '*(Og=@tqjQNwQpI+!>E$;uDoDs?m?$N^X5!Ikf6ptKwM>RO5sg@^lV4yzXYuQB`9X5fu0$u+A}IXiIlnC&sQ3#=p(Atd_;4$Wwn^'
            '>XfPKG5eY7OcZ?mM>lK5>%TQuf9lRt`a=S3J$>~`G_TJtPLzw{!Rg6KUwO8l>jJ^olv9^{zM0@U9nPw4Ug^nuycbm3cEyEI$n#s>'
            '!`e4|O!92QPU1zmKS4(3EZ267FdYsg(Jb8flg$%#P+bHk^~V8+NCpx7k};FMbRY94oZ>WF&at>NG7C<>jegG^oiFg8$#I0ir2HQ0'
            ';Z!4x*ns;|XA{eAO!$E)y>A;WiR_!Ax+e3cMI-kBCGT0Qon^6f%rXMfQe9=PRo^!6gG_sN?s|&8oxvl4?#`9@SrRa+tkVe7OAXtr'
            '+}4X<!n~REL2db!_o?ePsxBN&ZCImF9i5LnYH<%v<>c^IO&9$xcynYywDj#8Dbyvqyx7a#tF7W_HsW2?+r>-Lg${w}mVAwhd9|7)'
            's8GXhZf}qcG1nGiW*Llp_0;{;D0!lzv-*SG!P#(E%crV(n+*g1nv48AwNV6UF<)dWl^)gAZ%F7;e>qOkx5^sxp}s~(iL7jdAYUI4'
            'b;RlSIBR{G!<GRsUC9nt`3+pXx&E!~2_=3|Yn6o%*xmaNU*Fxke|C0y=AQM2efMZ^JQ%uX?~YNVqVq<!P`U@MYIj&_H&N|g4vvn{'
            't)Hs8V>2K|7^;DaI@_Do-7gC`vK!W)rmW1%>{dNHIR%vlL(oXjW7m}!7%0{6k!{ZVXTS8%D4{fEaFN#~^6hYN+y@#@DWZe9A7hqj'
            '*{}DiKm3*Yc~hT4=%&C{LiK)d@=Nb%a417Muw4+5is)7(uI4U=-f=uQKL;RSw+e14+h3f#Jvn`kVV3&VgUg@e3Z#b^S5(MiEc7Nr'
            'aFX#d$ixUVubH20r&O`nA1^T-aa*}{P&X`ZK0NE4oDX1DE`Rgw;N*}pD45V|2@6_|ISVdn2U_^}Hcv8wyamfjNq3P5!*6JRI$N)L'
            'M@KJv2XBktqw`dSajmMShw7<DftE2q<*e18_QtigU^S}j$<g3#-+hM>H)kjLq^wHfy}?KCa<~0QE+X#!*1r0*x7&J9?ul00=Ffri'
            '+XyQHTw=qYLE8NXqHmnYv-^*c?<Icnj}L7mH*ILxhBnaWxAgnV{q7a~^V|6kwPQCgfBx62=Fj^y?jmsc1C#}Z8J5#^F8}>se@lOB'
            'Y+wDbW{T;oZB#`!*W~B|)aM$jPYJ2Ha7A9MDv!%J!%6}Z4zTia-Yt!zbVexLvXxt4sK6zAVbWbKg=xv)qI4ByXeA$$6C`)XxTN#`'
            'aSsmD0pQcGpxeb*Er(cP?rlIG|1FGWZO79#h`~292ne)a5T>vcNC!A<aC3vfY^nkZ6-fd07?nKA;Ee%KGLa(%p)w{@HE9JLA`xdy'
            'As8|hWnW~J+FX_rn92g8f(eX_u1i5x>LbO<)V$QCw62&UKTU;xYWY<$l;Rl6N?LWZo3fBL@feJ3h;AG_v6v_kjxdH(q?mh&8z6&}'
            '8sYmmnd#J)vikN8U)AkUDw=eefD*&;xIl^&|H8CcadxXKtmFkSWsr^$o<nxL7!!0+F%DG|3wo7eT#QR+V)y0EC>!)m8ecn36~@;M'
            'WmA>9A!}5(x<ShfV@vuCPh^obGAq*14Sd*;rOQmCBfph5sb(YAQIRwQgAolbGL<{wV7Xc^$=C}Ne^IqfOA4|-Au&M>->RHJ{#L7x'
            '`8^_n48LU9bi(*v-Z9`zJM?GM(f6wa37prn-MyB0w)>Qp-dG^|rgYy3kPfI$cBIF`7Rv}JF%@{g+JDyj-?j>dLo6bIwu+p9mBy20'
            'UtB2!nD1*g7X=z?8Qm`BgTm2rDW>?HdoLA6jl#9FmJ~OyXZ`r3G&?QL?8zT#Ow_;7zs~hP+<MD2s{k!DePm+n4N>fe;v!$MS5FjX'
            '<6?+k>SExRtHegWAeF71$~u7+f@j^R6A+MAn~!M=C0mcTIWA!?=CT(*?mof)d5Zt@jB%L43nHMDEU>PJD4?@)PcOk0uguUQU$Is2'
            '#SM^x#VIv$EhrZUqx(2f^=9IG6ZG>%Hvr$CY=ntYYSUk3CRrMK#ML}Y&w**)2BRP|(n_y~Ah3HGyuJxUa>ijjmlC`3eOQlIRhbY~'
            '<Y%;M&b?%Y`Tp`rbD44Vv}w4l`FN9x*?jS*{3Sj7Qg=a)IV;W*!`2e@PPI0?qi~+`6k>;E6fS2p+(P;M=<C%-Ph)MaM{2cW>3M>E'
            'sa2$S0<`5GsMhVu%mV47gE`$7t8(1%Bo~k*2TI|G7qNbH?*$}0cmm`ZwDO_UgQ8i8**+pn72+sKlhc?aKA`am<w7js!g9wPtstoV'
            'jy)3Es5~bMhm-~og(j8~m569Y#dGvAxrkr_5%Q&n<@(ZBU#4-C+L3MX{vuq!q|5kGxg!)$#XnK5!>lIeBPA=FIN(R*A8!n>@IdIM'
            '?3sCcNycS)R<YYC7T94dGTw`A>7q@K8vBji#&+IGy<~$ysforq5D-$lr384pq<mu4+86|9mNTkj#oxpcNtrT@Yg?ks1mj_oRl7$@'
            '><;hZp&xoHR$r(hYowyTju723X+blPCD$Dkpx+FK@5Hq+(6zV)f`@gzJiN%piB51#tP-X_PzV(#X9$clhtHwypW-fpi1R7nVOIUn'
            'j{9^kZHQ^&-OQ=}TcqPQ<N@gDFkd^>h0I}_dhGObi&=dvl;e4)hHPoea7vlWrMQOdQZIQH8dQkJvPCK6oNBy-)P)G%FyPD74gd7i'
            'pP;>;{o3xm0?oZq;MediIf(pD`NFkia4r!2x%Z*D+rfnM<YFSnxrAY_obtW1w4gS1dB>J0smaCvs3NGYw+z7JMH|SY7tL5;6_dO#'
            'Dg*e-gtxi^ftK<Dgp>W`dtmQbPAKFy7j`E9EVS>))^tsRgH(O9T%nMC<mu$%o!B_`QLI#FS%>5sL#fua!<ixc+*aNTqjdrA4mzB4'
            'rkO~{{!F0#t?{#AaB2M9{B80>^XL8FI`FUavy=YZFtimMWuUQ`hAw;i5t`7u4r=v_=AL62j8*Cd$ww>JzF%=pD0KnQo=hU5Q`Q>`'
            'n2IAgbt55fHl0y5kmx7C=-&Yx1z5n76oK`B03_BHm0__FJaPokNirxugM3~j8f!72;@YSRCUOj?bmg(pfi32fDSI+*bP*x1_3?%&'
            '#M<qc(5MTt|8B`{GbB0=LO^c0*&(GJx}dFwvM-rKw<+fW+jBZ9>B9sIN{XiE$jS(qOv}joBWufZ@-j>tf?UH(Np%b{?m&CJG)Ktu'
            'R>V74!kVMY^-3I_p1wWjlpG!s+#So(DQWLcK27Pda&dM9+zMz!ZkN(|TN0I*I?dvQvQDk;F`?QE&PhbsEp!_9Le2&@YrvpYW*Cx6'
            '$w714_=J^Y=xBFb3Ve0lhkfx09z0Z8nCinCI6vI_YPxb-WXe#ElEp%?fc@Wp|NUA2u!kAx;b%j|{MOHe7G!^a(~s7Dj;%f!>Y&Jr'
            'Doy02Z^#avbUA8<-h4Cxv3p-M=lax(AIhI2+QBW-rreieqqUda`0;9M7Ok0cxYz_3UtI3X<9GE9ZtmWBW4y6zqq2g!8K?0<M;okD'
            'v_!!TrO$VVtkfW2q|oe7F#or_)qDQ+Z}o`dEO>fuJe|VV*%hrRljbe6@xFG<Rq56=J-RqoYZDj41Hl9Ai}oMHWQin49G(wbqWzPI'
            'G5O(rkSbm60{uGCy_funStk7`DCY`hU{U6a79#+Ot7tK#vd;+>0Edw|BP5q}^TiXu_;Um+iFe0TM{RF+e|MLrCNH1%(#&Xd65nCc'
            'q(togQMPlKat!8+(3jhi7nqTCX9TImZK;2vnHr{STgmaJT)YpWNqnEmyiFgp5&Js(hzq@@{7snjX0j&543(zVfRe?j{|0{#;BVcE'
            '0T0NY?^?%8l?!!KwQn|hK%j%Mw{sMyZZEp=LqBae4*FR05KW7T%VBFpEO~waBbwZ_lL6L-T6S^7&W?Ec92nUnZs%I!U4ckuF9V-Q'
            '!7U+~1d*|cPg~+?OYE7VgPZc!c4gxq47YH2o_3qhMVsGs?tOm-&(MeG|B63A(?2MWbcqwzm-hDbrzpu-{w3YM`eW*K;d?yP1P98{'
            'nx1>|H?5^~`1AI&c^sjuqWuv5e+^y4qKH_ArOpo@XMz?9V;<%@<ut5Eau(LJb(8qx`WZ-E0@8sY2O2PcGLb?2Nl%pMLi;Ce24D&X'
            '-$%oHCGo28KDR_GIM~}VJr<=QL2VvMvb-G^BOeo(;x%>)v#Ky9(A1kzR2q@miYO9|(C*{LLb}43<NDZ#T)oH-MO_?*^#!B-qj4B@'
            '9StH<?i9vjFSOcQhk%`%ALA}IE2bekbnqNAoL25L`VQy;3AmeXwR)AaE?^2QK4mI_9(yJf_fWIt{|@`88Ns7q-+Tvo1W6rn1E`m)'
            '0A)rYjpO+!09-M*u}m2EoX^UDqeln!+9A^9JWfZdCfcrD8e><TPdIXGLLw_`YLA+5r+ttE<_kd~0fFHhFbNV$#fT|6<5X>*N`{np'
            'sq)oF$F4XUFOvj~co?HH_9Ch9guX`!^=`dz+Q!diek3Ks9UcFoLsX2w5d58oi*y!VLJ0zAFsmm2k;<}c{s)YIjA!(xl%8QdUG9j2'
            '3sCs!H2A=A9Q+&BG;MUC-KUt$J&Rz{bRHVIdxzmku7?x}GFi?+s7GkR7o!xkA=gXCL7?i~V{M!$>pmgH;5SVERSFYfV5Rv%m@is_'
            '*Qwku+*hy*NAY-8^neF|$0^o*iP0%W&<(>~;E7W%Z1Rs&7t02=IDwak0%z#Asg=CslCtaTI0pU`QPy9ThMTuz8<b?2Ep(%$CvD8*'
            'rQ?j2XpJSs3~Pk5yc10L))_6kAT&ymks2KZ(J~hr0f-6uV&J_2Kqc@E9am*47Oo2`&0H5`h;YhqX{pI~_sHzAh{Lc8zkY1-0@-Tb'
            'eGZSz+%wR>W1<o}005YAYdQ!N!=IX3%g?lAU-*ptX?#X>g>EKW<S*^v-04J;Z*zH63{0-in6A1~l;cYxfxeN(%*u=V%o9OjBNSkU'
            '&umG!(r9cf1W*<$!4e$x_koPTbBTVyv*-`RLG}Cn8AfI}^=8d*x>aFp#skpD`KM&G`V12{2Cy0cY#Xpb{R?<90A)@r_Rpo=-LEC;'
            '`6$Z>_G#9XzKv!0R=8i=zU76bo}Sk%(b6i`Eb-4>jXl+ae8<5m@<x(^WK^9r%;k@-NZZ5LF9Z=6e^N6Nte!5YMU#@`A&tPa{Qw<`'
            'b)mFa%z309y6MK#rsAs7(DxTjeFPor@O&_6!}kC{7g?xd`BZ3>t=0JeF~?z?`c2i;*{Z>LZ5I7J6tF^&^EA^({lnM&v#KQ1d{ao&'
            'RVJ54kJJO(^;KEfQ`1HIU~qPDF&Mfp;Q+pc!G~|o`sZ&>kI;qqY3c33DZDv-e}dL8Xt4LA_+C8U)qLkKIE^FX1Df-^!z?pcI{*k?'
            '|9m(&?qOX?<O+Nr%vK(mU?Vrt=Cw7BPBBBJ>Z+>Mdgw(L%!M~X)*x-G=&x|6uk}=#T~@s5dYSXO;kcnU!XT!JxW+sgSXg4h!3EbH'
            'G02$qrEKlM3r-MzC~h{QyK{9rUKKnGCjv`$-1B97s`F%10S~^%M4PeU8mzLyu?HlYY33eTjVV7ReV_O%394$gdR)piOrS!`2q09R'
            '6Z4RK7o_r*!slSPn1Gkd_{w<I;_C|KS7_MeSn%J7(@HBo$o=j#stfF^i;V8Bi$`6<mzgJ%ryM-Y6V2PlQAgRs%uv*n=Ggs5wG}-?'
            'MJV(7An5X=!iZ$+>&gj|tFCQ(%9l#jMm71U2td|jqGOZt2Sh_Do-|pz!~asdX};w=NfJHmAj|q3!y6}?Vpj`w7%M}4D3t>qd&)>v'
            '4sB}u+CS?biDB=}@!+I`R?g)Z{y!2cKbv6;Bjp0YYK`+H7h;^`X>H2!#OS#vF!fD_iEywG`e&rRG1A}I&P1%Vy9(9Nf1$!82}*V0'
            'ggT94uDs7lN>s>Qr7)6{m{lQnp(_{Il!?B|Cp*u;^y#7a#*3CPEUmg2Z2FLs6GB{#Sq+1gG}c^cvLBxvXP+OGjf3rZ7&F+OpA=*5'
            '-OZB})c<E+Tk(gaM?p@$$`SareULaP__Irt$2DoHWw7E}yLA}g)QIzvZ1FhtO**jOQsQV9kN{H`IvWP)*HTI|YF}Owkv$Vlrc}5`'
            'i3((7*s78rHS%FStQ`4liBDCGR#;VK|1Z<^OY_8B>A545v8{SqtwNPsEAEDdWPh)0pwTT``PT~!pshkIIIHcKGXNfeD(o=VFBjLb'
            'dBWr6Whi*r*c}qkx+WlXfNZ(OjZQ+P#JZ{r8aZM`q~&^UqjNQq*Qhj1x$HIx9d;{P^Go-hY#405uOyfD*Flgnsyl7;q0+P_<Ox^R'
            'R9*<w&m%9C?UFZU%{ZJ!d2JI=0y1h)$0qVtz*^Tf8pm-6FpOOCO|5UoELA(-Ki1rc{d1|%m&`CgQGj*Jk|Bhy&bdh1-=tBuHW^q+'
            '>X@qvX{=d-<kopwzlwn5YMPiI4lC_3<9y*zRTQ%C8g-irFP(M0vvs|*$9hx#;`(n-Dtg<QjIF-bD^4SX_gAhlrsP6JbDfI{kvywu'
            'lRs7dbbd+`kYfW>0QCyclgiqd-Ii$9t{AiqbZ62dSDxXkt#Ty2j;f|z&82Z`{!EYMD#~1p^&f)?<(5Ob!!2GPLYoQ_E3?B%isFcW'
            '!HbVo6~VpKt*+>HWL4Rw8U)Z`w`Rzy!+upVw8;BpbIw(i*;cmj(Iys6Y_~4)95ryNjBc`P?Uh&cXH>68FTRQbBPWTI-JOzO!k@e1'
            '`EH$?FT=YEXjiO8SIRIOZL#EQ<z22ndF%LNd7Sm~<kwi=m#yvhtgD<B`*YV|2!x5>o`!rK_<Jn^WD+Kl$Z1R~q{5b{Ua)fV+^x)F'
            'D$PyrG2Vp@VaFHeLn_}e^3k6gwE(nE(K;xbn5<+ejbHfRX{3wCr4-3R8OpW$f5D;#>f0~<b6to_7G}w*3W`Z+xJH7hX%Oa5VM%32'
            'Wg5U+j+P3qkrjN&`e#_Q)m>#-<~2m@vaDNA#qkzShIf-`ll<kIH55|^GO<A<Tf%-!f=5dVHH%UidB)!*n71ko#(@AHGHVnwxN&fi'
            'MU82qB@LGve#T7=-`Lj^Vj9n>yX4{C8_{CxW3u08QDdz}-itrU0qVb`Gvt1vjeHQzQIG|2Tt3l)RBGJMqj;sepdw2z$e`6;-kJ{='
            'S{u2I7Lx+wEy{Kz23gZskgeh|nE==ow5)cgZ*ZFH%)5nCw=X1abGX$c4wsZ*m`2~O8h!ifVdoPJ{(z1y26#aZUbmQe<znXR7o({='
            't!KW`dM4g1n91<5;L!!~7v!4exTa%cP1(9NWtD5n<eI9<IKBGqcQd;m>W2R_e*bg8#6{XIVEyN(3?VVJAYnx+XhTrsEz;XK+t9r6'
            '8Kk?~I$p}|^jYTcdgH{S_*eI#M{E(Fxvh`Qm#r4-Pk!x8ZGCM2agFV<sy=mk@?&*<>NMoXD*M!Fip72f^MmeYJuiQ)U_i;gX1%y7'
            '-BBoWQAv?2+3i`JjMMCQ%9BO=B3VwVWy$YB&YN*=x0c9Ll~p8Ht`TM)5k_0WHruRAPg;n%mA9Yrw%yHcG3hP9x?6k4UNf#5LGKg}'
            'zV0gi_fe*Lq4IGq_Cr_fZGAD#XA8=1i`?gH=s#IaHkcAi36=A)%#!Q$yMH2^56Z(9B35gEYwV4@P*up#e%ZADbhfOJhgtQvN_OG`'
            'xC`dKb#~UWf~@7y5?#X^ss=iKx~Nz+>b3b&KW(V3nW9Yxi&mr>%bPibWfCty0~VB%C(_0?Pv+*V*=%VzUx&u_rvZFb->^UNojBy$'
            'h~zy*+N4Z^k4Y7TkxoVGbM;Oln;Y#aT-nj=xWSq*A8l2lzWpyl#u6Hv*!Wmy5adfF|Fe*k(7L`u8BNKuXS$S2qh7a)aX8iRr0B^i'
            'EYTT{YOK0N)la9I6r*Lh<f0~+e(IU?0-m5H1>?cs>l06?e7ke+W8*RFHo8~23R?R3*UII#_WoINdF{sJL}zPL;-RN+UU*kcTFNRd'
            'X_S)E4ZDVnR>~u)(WqeUl)Mf{m-$~n2BYld?6e)}XgxVuI-U3p!!V}3>mtMRPAB<`Bkk19&UcC*l53pubZc2225c;oD^OWQ(qJO1'
            '-LvBx%n57T%X|3|(2|sEoO%@nBiiY`w;(@pj}ie^8_!fmX*TW2Vm5NU81ubomeMo?sc*!yS9lTO-T^+7Z4C4DB_Uw70FXxF1~ksu'
            '*cTt;1dR-nWg>Kb?rmSpywwB~q6djnbOp*!9wsy?mmw@_f|P`*80cFpa;TPAYQ<<o?f9p%j#jp$i`W_mg^K!^R(5!HxndnWP~b{l'
            'h?JbK2#6`ZnNW#bAO!x$j#a2oTP8eooJ=Y0xW3CY%QBH~3>PUa-+$VcP#%l3s?(_ZkKLbk7V)CVjf;swHuh*yIh|>TOPW7DRpk?y'
            'B1!LER(IHkW#4q7-BkN|q=v#*RC&;h$PR_N&<?pSZ&X{J(gjhj6*k}N;tQ06K~2(%`|jk)pCm<>&n-={ly)7#)Qx8rjPjrP+NvSb'
            't$tjBll`Yzc>}#sC7ATL<iyF<sE{q2IUk4jt(Dj{oUOYOfYCV<+{EY1hSCT|eOg3d#Uk{m?k)6f=2hy?58ulC5>CfRA<A(&C8KyT'
            '*@x}PCcjoljl(zduyOQy%Se@p?3rGo9kQZ~J2#PO6K4L(&SkZ)Go;Axnk~pVEQ<f`?Th{&p6<z@Wk00MDpVr}(2uv(CH99`8DlIO'
            'hj$5S@eN;{S`W91N!xER7%EH4gHqibnRCy2tAefNhLevHO_G)-FG=!?9i(Qwh6fXSJ^?+r<J3?s_z^_Lh^p`~Zq%I_&+wqwfy6q6'
            '@D{#Pwun-`gM(xVO#MHfwa2$!l2UjVBbyqd#ax|45*-^xH*Ldr$Kqz`B@+|XICo18rtO;Lm<&|IPQ`>hGS`I>X2-3{VRnVWHXoXM'
            'EjXQ#T<^|~c;<YA3>arrRD&z{a@u<T11T0dXMHN3ihunUiR;O-O&?Y)he8jihDXj>ka%jr6O#-A{$jX50#aP66ZDv6cxrhB(>mFQ'
            ')8}!W3s)uhv)D)dK$ww)J4&9R*;Zc5>#wT%jx2IJ9QKdj4bQ)rA8y3VNdGkxUI3Y0PJ)b7WLY~1Gvd(^K%Qq*5W?5pZG`YPPYAw8'
            '@gO83VVDZ;4(1gIGfHj<;+rl4^Mhg}#!yP)-=?AH<XuacP}vp+%;_(n;SCP^=UfT<`i07Qr}8~k`k`R$G)nxj9KkZPCD$R3Ek$E|'
            '8wX>`PqIqIEvz<GfuiqGC%LY~fUYc!q$6RF3S=NwQvzZPgTch()Q3OFSuQBJR7g7-g$Q%B3?|ga8aV)J&MHlp_@(67RBcBV_%uq|'
            'nAzhB`oR=)Lk7{@UjdPmL*HQ7Td*%a;c)+w;2o_$3$f}EJd;(5jnSP4r{S1oM%4_O>YOzcEbT;grK#NbW0WgC3c0;BKJ71j7)LHl'
            'Mq=4NM2;gk6^<cRXB&ev^=~R3WW6_E=T(L(1OU~lNSTd+F4ga=EejrK<I^B8*?_Fdj)DM6{(2}3;c=X@8YtUWUKnmvYelMHt*XNC'
            'RPKo$7vdOlUs_%Io!aIVouk-Q>vPp2svJnyS9sPsPNTH{CKW(+b=~>jCcJ&vzN!d%kzk2}dB{pz_(JH?xlGuwAlN+MO44)hW2Zcb'
            'Q{}yCOLuFx);K#%tR%`7s||N$%l=<3250?4w@0<txQkT+PnOF1jCxw;TcfNNjmqV;=rX_>zP99+Zf8|AC;L=ZGa<Y7Ieq<7gMs;j'
            'L^eEDSEzvMJY`yC4x*|f@fj7OiRAKU@`79yC5be?7HdftDruF1nd?hP*HxabEGzxl^L)Gj^#(KFxHk3YgR#1b^(K?5!8NqM6?c72'
            ')3QfLQ)0bJm}=sgRWbW_ty@hA#DPL}HljtF06+!<tx4HMwnA-6DEctw^QsmUo$^%ynQ+Ul!k@Ep{_)^!JsST{r@ve+&Qz7EMeb5<'
            'K`u?Vzjw7&@l<I>csE-X!yV^pYpasCT0C{jvTrKhx@CEU7{`v_U_CM0miV#$U2KK#h(>NzaAx_-e5cv8tl-X|;LMA_xy|FzDD}&k'
            'b)(qX=H89lU_8=w2CL9pH1VA8NYbQJ+lU#EQVK$KUKCii8guO3B$_2-;mxfAFMg1!KHGI=kw(nn(9|lasw`hh%lbVfw{@|ir)L+~'
            'uFN5(E0&V0ajxZ#yeN`(r@5D6=(~=g=NhO|X*d9kN5h~5)Zmaw(jo@&W<^;Lea?8=ZtH9SoUyf#FkT{bgCa^f1EyZA!K*cQC7XM*'
            't}3MT9Ww$Yc`6`NfMh&Qf{_&lPA{9qnx&5igs#b`$_S}ypV=khbNhE=NNW8jz0}KXJd4{`55E~VS<?m?B%mF^pztpLFf=GGs?fH_'
            'H%Pi16QU7c5*EJa-3vk?v0PyAD2%+#gle)aGh;J~@jAEFDBCjetS+-qQ_``Bihax%go$&>GDKM<K`fRP9}d_ccc!XwQSm3U;!<u8'
            'm4+pkjBomLR5|uTmg{x+SIK~d4;gE2VcY0(BNx-O!I2WmmWI#mkqSZWRPd?>k?bk|(Y>ey5?A_gsANHsRSZNH{lF&+tsDVc+N(1V'
            'mD*O6=%(c<rbLY_^lZrE$L-2H$##6bl~(bOGGq##E_ygL$c!`BE>KuPkfQDU%m@&m9;HA5dA}_)*RZLEIFH5=^{!jk4P(;M=sflP'
            '336lF6zZmZt`#Q9!OCgWi(_=m{@qWqpFA$V6T^5=S0X<qZK^=9DH^|S^#X`~Hk{59&>(3R;>YsgkeJ(5Jj}ONC6C9F<TV>ydIn7>'
            '@6Jx&3|<a~{X_ig<<aTE+au5&xE#X4VAw<5uoRx4VQa0W`^lKnL!&0Oxht!|8#Q;Q!HsEFE#Kd1l;^}48<am!SHLUhE;0x#zy=i7'
            '$d+!^<K18|E~*F8uFc0=oL#M(CyO-gpD%HlT%r7$*Lcl0Nw`<nHI;|2b-nZuI3vCRj?Ja~@XD0Z1>$2Yzquo_c;PPOzcct(V;NtT'
            'bi_NE9xa|ugE6oopyYJp{8Zd5p&_O>F-J^q<7GG@lO);(gDh2dLt+n;z$xDaToYwx7U;a=3vZ%Jfe6g4jxt6(53;=SO_0juh9sHY'
            '!}wW@M$b5r1H$L@lsrnfCke>Zn}bAwxjkU+=ur@j!{r2H^DMFD@tAqj7Qy?$q8r{IJ>;br3fPh3J@>h+DCmriEK1<F-sB|*O4!KR'
            'X>>p?tx^VuHg0gChC%N;!AqCL66GHD-VIctqtplXg9MAegGr107u44LqG7Q94v;WJCl|{9Psz!MNTShPg>n%of#E#EiA2GmydDlP'
            'E<NUKp&)MJp+N*8osk}pYZ2QJZ3iRLMxj9&^m+h(K;Qh-u!jYWg8tQ8TA&@Kic6co<}VlU<>Fv?an?UX+M4f-vYSxZ_VSeU=<{F>'
            'vejxK-Jm?IdBC-ncR0CNZBL$8gJaOGXKD#K{V}<NocQZAbUcG;OMAkjQ?Id-e^R^fpbYlsROclBIuHZc-&qLww~CXFA?N4~0_w=;'
            ';qsG8v&Pj@D9O#km0A%I&LT8Boc*IoFD*%k-zx?}?v^fP7e$5+#H}ecY!_33erTCa<}LQlbf-}Zbg9(u5+xgZi|$YOBZ+PbRi4aZ'
            'PWPx#6)PX<vQN<MMV_G^Z+74biTo_y@Z8wN>_0I5Dolnl`19@5HB92#ywx>kU2TY=LNjPV1))&ZX34vNXhhjKtIWo*cU}5lAb|-W'
            'y-a^!91|wQKoZ}J<`eP#_rij!<KXk%&X3{;9IV45VYYVhL?2xq@6W=Bl2#kgq<4iHHt<B<r<2a>*WR|0GdgY~2n6F0JxV1@=Re%3'
            'M=5jF!-76u&)~JnrzC{0ccuIF?sUJ+rS9WiPC8dxeQr8)hPksZVkWR{HPn;*97<@YlecEXvWu`B?Q?@e>H<XS#xvtY)(k1#m0ab8'
            '_PNeo$udgyJvYiylp7euF1ErD<v_|aU52uRr!?qDG4WcSi6$R`mSH4CHb9d=3xVP**EnS~X0bR12WDwhQO3buh-+iZn`UQ&Pvf{$'
            'LT_>n+rDCU!ZWHYczn$z$qHq-Pf@6alcfHZWp9xSg2>5I$4jjBj75Bc2pz8Q9-Yg?2PvnE#Ea}yW~v+k_F)Ndz-qF(t0c%W_oR=-'
            'AZLcX3@sgsOtnQ%o+}|O`XCVu4u(g+iu3+oE;w5m{DC6FSsw(2lXE4IQ7Ho<BRz6qsV72@tJtF)3-d)8-@`SqA&v<>3t#?S5HA@Q'
            'KqXiP5z74>DbX^#!Z_VLxhiqyMz-Wly@-k!Dan~dxXcUVn~q){R-VM%e7JWcc^AqSD$c~53Rx8MuNGIH<Rtfbfk__nP!9=WDAqaP'
            'Q~p9OnpDzBMI?mE(^q-6bau_!56h~OJM8snmjJ~Hk<B7rh+s~(G1Q{S4O_~~WD2ZFqoIm}wj{U{S#(|&=fv%YKa_65G(4Z!v)#R&'
            'XS+|ecS_EMsMw1%&S@8~dmw5)l7SaA-_#uFnRaujgJnx!bEq;k1S@@+$RbKU=8qpf=rPr9)2d*gLe^Wx%@bh+atKY^jcKYQo9a`q'
            'ifO?&s6rT?^OL@IlfUyNAm!OP?;ZDTPgJ?>W!TCJ_sZS`A2=VLzB}*8M9XAJ_2uc^(dAqLGNTnn@yq`CctkT;Y1M%?G8<k7XK|a4'
            '-*!u`h?tf7*wh;(RP?)0G7P@sCz%_?ZgrwbY2Q(-5OpT?6H!8^V&2My2Yu&OeND%9U&H@deM0BLG169V<u&k>M_tK_Fn`sp^nbIn'
            '{b>EI%MKv9prn5p(vteoQi2ixW3$+5$EV7ttqQTITYgj+|Fis;jW9zgxFU11LJ?{%?-UB+9T!8eQMlSG=Ul6}uT#$Zr5hNdmAZ>5'
            '4TOb!qxHkocg}DBsyeH;72Va!^#&9?e;Pa#<$0tAWjLrEcF`{uUbDfi;(bm2w+0{D3IH1tP5JF3>F10fS3cJxACtL1-i~X`mYSh}'
            '5RF$TieehQ&Ti%XDifl0cc{g!v0h6HvpGBM)o)GdVavJ!d>muU+jDwjTEg*|T&h)&O$3zbB^e?)h8N&l)7*@cdwgky`{Ke)q_kzw'
            'aSQN9@`ZT(_O~=IbEB^Kx<8sU3pEdz;<m)oiY#;~RcbAz=7AS+c!$33B*5cUh<qfUXvyAIa!vi#Dw4*_&2J))IbA-6GeC>qy*~!#'
            'U>A;lG_v^~56;gAC$Bfz*G(#%e$$Ph>%ce-$Ya8C{{jhoPR92pVYTx}YS!G%O1D$)F<--BCde~}6>nh}CS`JarQK)B&XQGZNmO~+'
            '#IrKpttKH23RMVKuDnbm-&||}@6^=k<}3_T7tUHmg4Q}0rE+wt9m`Z5pIS)NWQVCb<TzUq7cWPy38KxFU-2C&Y5>TQzSP{Zzb#Ko'
            'HMh8`OS=0=FTHl1aHrtN#N?%5fw8g%<YIXBYrR9(V`T+o3q&F0-+C$IqK&xT&#r!B%%K!w&ZcY3yrB#q1p->KfK-Htv-;4jz4<N{'
            '4cD7W?`x7jl0E&LMmpn$Y*&}P$|JJ*#H@zE<`&b+f)PmhTbA^TjvU3=7So!QD8Kqo?i3Xv38t$O9WGCK?xz&u_SwYSKErkd=T^Ph'
            'v}K1Tsyo*_I?;|x{Bjvz8S=WeL9aZtan20PuyX<M-FjV)&@Z6aUzc)MQ*eT0PPFAt7io6>NA<W5&e~P;Oer6Xoa@@=rf7l&GXLIH'
            'evXZ`9ScGV{@CC{rf|}%2ixn5U-~CUgSP^Np2NYuw(e#AO#mZKfT(0aN4#F%`B50mC>{wugg3TcI`?KA65pu(z?8wXld1|C;*i9Q'
            '!P(&GxPLf63&ey%X5=P4RFAcOE2{OA^(<!HcwZ`(RpyP!MYO7emS=_OT@2r}YGqL8e~;k$GbPo^|A$A3RkImJ&CJ9kzDbspG~(97'
            '^aq#^g^a1rekmu4>{)+`8v6L87?6YMRxz?dfZpfLu!2oz?GZG+H*fuL!Mt``GA_^<x19&kTcQ7L@H<bU#zS3y<|?mI*7%=#h&8u-'
            'IXF6k!|ML!;&j;ipK)UAdw<qRmI(=q@;de`JzO(FNgqH%W%1d6H9P-jAw79%Dm0Yt$ZI8W=dU~t9?Bz%=17v(5F9B1_*jW5B>xyd'
            'CO8Y8>W-zbsLn#3-et{jljTQ0%z_0b#!kEp)UK5uVbxclW|2Ra0tJW?Mxf!E;WB@gcnMB{<&_(sC}VE?T1yW9m(qO=apk;!_Dlav'
            '2`t0GasTvU$d#zdO8cK`+$k!jQtn=<Ipcp-&QoP_z}l%>$kj<T5KuGRybO2ldxh>E@c-4YESr038HSw-@PDEAdC(i2{L(ub9J+vc'
            '?=FS}y$X}ipTypN&6i4TYhk7Drp0#~-2_wNb2`4|byzm0R<+4V3(S?Af-bhV3h5iA%Z8nKUz#{-a|)H>FTDFgVZ4pZ9F-;xSws5E'
            'E-0-P6cR3NX5w)E%gxI*1;ol1RJoWX|J+!X`f?fSOJt{qY8XBuE2(Dk((;OG3!#0mvXym(N$ktJuWVS6+}2JgWyw#)J6UG0WMZwu'
            '1LTu?m804Bx3nO6)q=jEr2r}8U77M|_o%M8^rR<Q5Us<{%l-D=mAaYqE)mzRuD0r=f#O{<i6t*o@MBFbiBI0;Htps1)dL7d9^Eu-'
            '=ZFg$Vk4icT0+^PNSCv5H1=l!;|4n3>pM)8%^F*eiPNyWt<6N@Mq}$+$X8!OuBwrzKAQ^tevQv2KV@cP5F?v0ZEd9OeIZ5qTx#@C'
            'zC8Us+gziTm7!z0$<%9on`mv*Mz)+=7x=PvaOx!i=JRQOGU(HG2}8E8oCm2cSs~z!aWKmQaq;Wn#o5Ktaqn=@X&8A3e-APjkoc?T'
            'L(9Eu+tSZYUJy|TF}c@%V1HEUMt9yn?wt$=2j|};Kbp_c!dzOFQzDYfFopFR35~F%J&?52aF_kJ^V1WVMRCVV5^qHgtnrj0N>zNZ'
            '^u$f$In^pk#eL$jZ(NWuS&dUls^N+2>ua2am}1dxrz6hrMlKbV?#Qaht*mmzo_peEiSD~bJO;V7$kpg9x$^IFo?LkaSoOVTV-%-c'
            '6k7(($OzHmXzZ6jWydiXsKc={&7=XVYyP~C3v_<|E&aaP`Qc{=o&jOwe0|XNT)X4Y0Hqv%^t6M)Jd5U@V|`K8{=*oMW<XpkAjOsD'
            'HibjoY<$AlTjTG`hC>&W0fwx#_nbV@B>bV>40&K|upyubNSVjzF$&|GCHm}0sQE1DdmngW3D4v~&rCkJ!m0fr1rrQ5f^L867K;_G'
            'ordxGnYZwwxi=3|wRg4y4;^@;k~YrdlD`EXzOuY?>ThM&&1TBe@m2|0z@=CKLI;chi{~hPBf?hS87uGlw8_$BD^C={Nin-k0Nwc3'
            'AJ6coV`N6<js|rRy;nnwLZ!g89Z6~O*V3#0+;Z?|-(Z?NdOTpJeAC~bUhj+RPYp~2L0*83#=|v>zNySVDx>tEKde0rF;8ZQ=Tuzt'
            'yss%GHw^suC7qjiN*$Y#hd$gOcTLDoS3<yH-{&}V;446fN~!J&IWra93i(;r@0Q9W$19Kg6#g%02`o{M_jVX0sHW3#U7V=vK1k%U'
            'GM5&1%ZmQvMl$A)N2?0)tDB?D#Vyc0Dg0R)VDbwZ^Iwt&BeL%fLj<)ta!D<&%z1DFBfpVZbDdstW;^LQpK%98ew;6!Gf>p>j1hW;'
            'jYse>rWh2l=ywv&V2Owp%@V@Ymt3#(tXRJSFM_j+-X}ARK8ZuI{RyZ0uuXSf0(esOxh|y7Dis^~ypL)D3O+@jx3i*1%~J+j5^I;|'
            'ocviHCm<#bs*4~hQP2vaRV7`N-Cy(*XEV!MVFhaX76{a2zJUbSOR@l+ILSVS%aeI*#oq2W%3*u!J^ArDgG(nT)+t{eOC}6cyA6X;'
            'hev6SFjK{?|6vl`fHKi^)}|V6QrPBbmHDajMLCBt1}Dj|j#f&hv8w!;YqA=XOaM#)aIV={X4CdhHli2SM!>NaX8~{u%1aZ2u518u'
            'WB6KOH2xAr@R9o<_Qcd>BRJSVIBtO)hQ6JfhVr~1y$V1rYjO_nec7@D8q%cs{Kp@kK8F*B+hCIbo`#B`rjhJGTYg_&oy^7|Z?+|J'
            'HdTNDsxd0V)%Pu7MdL2W!#d--8|#i9n=568i>jhyGQvhRc{C%3Y|fCb9FWz@%?q3fDF)}dXyvdv+mf3uN0{IOn68}?WGxsdbA@}m'
            '1On%J3)ELJRg@Y3D+A-rboKJ%>jFx>5=}vL;+RNQpW7<YNFH6uQIMc|p%q^TnJOiEhcJKx>~;{fchH2`mg&T#GcK}pPENuRmXu8%'
            '%cCr_+c*h6vfy|Nn2VNf&+rX|d0d<wwODmSk4|56RVMT-#8I4No*9k(Pf>J;Zc<Jf;)QrHR^lGL=Vb&f2clN`C|)~q;Z)3Jx{Nr|'
            'i3_m^A8Vg(^Uh#?E%+E(6(*d^!lf^5zo1+hF)xiQP%Ns_TvZJ3HF-q4i(uhny10(oZTZ7EY0h;a|C(}Wm0@UI6_19ZRml%dr_PgS'
            ')eMqQj?S~j<7|;!OV<p9MoMYEag4+o)=W7yvjkp)lQ&tQTbriYwslofC1Y!L%KIXKd1U_F|M&v70u$JPx>U;h2Z-Fxyb*{*fdCz9'
            'KJi4ec5fUM$S{bc6dIyuWQBnlhQ~4}^-Bh3eLz^bE59)VtU$3`#6gsiVNuax<9+_Hu0?goT21s>c}iazfL1q>YPM-CTHQFG8I)Et'
            'R=H`#coi{h6$hx;$XOHPMhZxI47-`*tc`+O(_O_(RmGSU&holYGxKlCCsIQ->ch`~4R{Qb#(rLh9E&Jf6~U_uDivbp%3n9%w=ky8'
            'Sr^||?qAH4w@LJ0d6b%_&IU1pj}OKz&9N*IOA1D~S%&OyHDS}8`J^0ETp3VV{)Q!tQ7KIOL2+1{(wj51zu*{St(~J&i0&*1X!>B4'
            'Us*>f@j<5y&^-yZ-G$EtW8B(TXZ^lA9Q4oJ!``rmmHq|?eWPD>oHv@Xw42kKJy(^%$k}Y--}nh#ad3UGVeifH;N);{)>4JRR(>`E'
            'N}kFj^AluEDt1$;M%7Yg9<C!<*JZ<Q!O662cBz@Ux+cpXaiKJM7q1_B<$zln2$z{wv@ao(-S?p;U6fiweZnZ!;u|RN@BE~lvs4rv'
            ';|L9!AbigOpp#T9^et56=2EeTo_#|gH@LE(3T~M7<@EgtnXe|Y-W~Dy4#=wh3bQ)kVM%<H)nPDlmmyM-*r_U%uMLH$h29vmOyDc+'
            'oy18qP<B_W0fwU#EAvI@b8XUZVu_$r?xM&c^S*J?Inpd^^EGai`f;sQsy63ip4qHht-O8zJp<UC-FwMRD#ce>W4T*X0E!od7ZBC&'
            '%PVzivk#WdGUX=S@-v-U&&g|RqRG83O^}s{4G$!a%Ih#qkDF&P>{etkEDS$wd@7Ik@LTj-^ohUxP)wPJA$!%iLu*n}@Cu6vRQTK8'
            '^a?aAKdM5g(i3~!9Mkl#R%N(3r@NzV>eS>2<y+BizCs1nuKLlE5bN?k5(|`36Ytm`G3(k!(Soy1b6rM76bS9H1X;GKM`M9NR;#mu'
            'ANQ*Q9i=heI+Rv@$!Zrlyd^bas#R&px^#1#Y0gf4rhd0dG(i=oTt;f7+c=xYbN0zVZCz3XmCg2h*>RHuVbdT2#sUMjc|^e@iYC`Z'
            '>$M6wHM=O_Ea%F|apsTx;LcBljL!x_kig&4W0vmRFvZ~{zE8!DNXLL^K(;(`h)>5Lucqpz9C?lkzD3n27R}U$2^<}rzIV@h?_CT{'
            '>$$^ISIbL<1g#ajRX`NXQ`z8k!f(0P72Oz_u=Q4+(OCAwsv@Xot>CpptHr)`p{G=8{tPe&)B*W+3Y|$7m-^k6gYIZjMszIPF*sSH'
            'xZMqduYF&OBn*8;$uKj}@5K_ipZSOyOY*Gj8*YNUj{V#{kZYO;5ny9<ligaIfY8N^*m4KdH!%0L_C|29F!-~_VP1mXW<Vb;!<j1s'
            'o@H{@-vRd9;O4e{7l+HaZ|1+k9CHfvFUtosh=4+cN|Z>@HzA4dg*_|i7G$Bn!zB{PE=oq&%)2o*xMiGUPT&{%UbI{Q6_whn_F|Up'
            'jcL-|H1LNz^aqZk&qR^>t*L<VW}9;tBpy(H;9$uj@w4B%JMiG(Y%rvU_MqrwL=NZpXG+cpf<A8McTyYZP~JG@E=2NlTI!7{Q_0Mm'
            '8*xi;saoD<fMJP&s>e%zAR8(yj^t_SRZ`0zKNDhAS%3MyfS*W^94PK`kxEdQ3Fl;T>by7A-4O#6OeQk(ohQzZfpdsxIUfO7aXaFL'
            '%s{CYp=*_3`Ezf@0!=bUU4{~md)Ox!Q(DDkl5n}jAd*?|?xpxOT4v-a#?B|6QIC-(@@u>tE=N8dmn0aIq#}VZ#Dkeq>NHII8>l>Q'
            '=UkZg;_N8h83(+JDJf6#vfAfckr{dl$mwWp^)Y^^MMJ=g{JK*o95b=0gcFshWO_5|!CkH&%s!dwioNu<Y1vjyhjXWH+`lggP&MPq'
            'w;PXZ<21X@TaT>P3Jmc6&iAr{BWdHcLPsyb0ORxCVgH~nwntcMJz4@<;H}lIHzM=P1P~(F#;fNgnq73(>%{UQwQy%fRvQW%v=Qhd'
            'd@dpk!hpR<tl$?G+w314_y1P#I<-AVP4~)*-7)<h@NkE^Y_|5bObuKmc9JR=%j+D9msg#9bj3DQ6@pjnW8|Ud<ut}qP6&OYuN1vN'
            'T)$S`k5zB6!#8mH%>iCVzxPMi0)3xS(Y(&T$;C}z8u$LFvrK{=$@${B7ev=mh*Z@^F0b-z+}cZYOm~=+mNQRPu9i}A@G>D+>-8Y('
            '%}8hi-}L=RRt&`AgU<R|A(qF>+Ik_HPwNsWW}%Ss#xzRBJJp;0^%dSM_X7DTU4z~H5^qO!$s5=qW1`p)y28Uk!5i4bycxX4+A~Q|'
            'u8M=x$F}e#{0CFl1DN+8ujR&BD*dSAO(sp~YL6^8ZB2+0&SoFKSrzhw*avT|>GQ)1_o4=)X|cV0>QsgXNfeXQtUc+PUamdqSUKz('
            'J}R^9t6tYe1s+cFjNCCzm*~q_#hWC$qZhOyl_0|fbE*|QqTH`znGzT?s#50Vdk<&~eb&#5lgkP4)!HQpC|4VlZ`%C8;1KwU)%qGv'
            'RVRmWwE(DUL<i*Lk4X1QwW!km%NGjQsFStJvVk%~i{bEG$x(;SK1xFUcRdVgm0Gtp!AgN0CxH7z9!=z!mk4Q5eceAEoD5tT(f#Fg'
            'aL_;3EU)o5e{s7?sJzc|Ji~tvZoP!$&Bj~2hJt@z`B4&3n<IbX;lHQA@9^JV?CUbSjRQcPAW3dx>eO3ial}9SpiPI~gu4v9g`e@y'
            'i}?sv98&A>Ch$j|8tM?X7&KMt3|tU&X4%M(f)N!Q@=_1}0*#5L<w1*yA5e!LAXyT5i5jaHQgjr)I!FBTXeF<`akBit>khF;0=ky$'
            '9wg+G@HgPuk&oY8jsi8>FIe7S9(cgXsrNac%=mW1pT<8zP`rHl#Q8GxrXXOnwwx`McwcV;2DuP?tyf3Alh+r$*Yq|^+QYNP)_42j'
            'Fdi@2)~;k`#0@qf?u#eS#O%$-me_mV9tC8u#SGNmy?A$UESm6p%;o<41UMvq<h*zM4oe#&Ds`c|ySuw<zIZ=49KOLf^zh(K@8qO^'
            'gebFDm<-N>?1&$fZHPZ`aC$QA{}onr`nG?<0oc&{=@a^pwC^h_M!4*f<DXqfEL#G4U($<~;a??>-#=4%w;)1=u2kQRAG-x!Y&t<w'
            'gAJLO&HA}3vXAoUSY4)~WnOfqG?}tcQ1+c<?r-}}R?H98+vm?Zqvy{iJ~>n3^Q!D_MDlpssY#3Gz4d6c1>zh9egM9C3iM16??`l('
            'F(8a)i}BpeV)x#=lPCrc_Ju+!4=JdN_VRAwT%b`Ky-dQiYo37`mpwVqeK5&xyTyaVVFdU^&57S9-XhgU6j0C;XM<O-cHZ}X>F1|a'
            '3f9JnmoZ4vOAbRDaDWyw<LeHr@)~o9$>N<O*Z`Eq7ooSp%VqXPl7$2)M#5i!ww+((u?H&KeUNZuATEH4bnV~=Ks82d6)M}(gOAn('
            'YV&kT)o-xEKujgVXb!e8jcp0gln_vZU3{<M3_ndU=0*QpFAQ!_qF&dMA%ghVKu9*k4{{xy1&ATd#-9d2ke@z>$7Hx2&nfC>)If+&'
            '=b69`c}3<z*Eeql@c#{@ur0i)RTHr=sFkm=8cU4Dw5$cddVX@O?~nLcS1v-&-||l8<^O>Au1h}#(<zJSZW?^#gQ!uTpX1dDyIdoV'
            '22n06o<k1~u;V*s1gel?{1oLSEK2mgcl35}@|uemkivk1oJ8CAZ+<01N>98R{I!25PG7w`?+@u!&Al(z!N~8(pFL~i<kaGT1f=4c'
            'l|~O$(<zo;c>0`5&0&E<(&g}K9GA;X0)rwLO`q+ypFLv??ueJmj6FgFX*<H(sL;QGkHT4rk*TsPctZ9XIhIeQ6M!K4z&Eb3^5ON)'
            '_2A^y>9s08%8om7l02{JbiVL!#3TmE4^&f{q^^rM4L)EI1YQY+V9tFH9oyMf_WtY?pv#cuH7;UcLa-?lS-^sEAGQOiWJ<;{ihRuC'
            'Sfq3G0st{Gg~NyzuND#nZ<+e?(_Cp>m!!(C{kRq|03xN3s*ZT?g)>>5k9Js9=p)d{P-L|M_OX{s1IgjZR-W8dKK()bgrRnb(8K|Z'
            'n`O4rK|g$P$?f-bw`X}!5pFeC&%|T0jEsX~bj3b-_n)qu%UvrjNq?`KAB;l!j{}e(7h0XcFfyeo#pp1PuI?UaKFerKj?)C*`7+Jq'
            'rWHm*=rLXfRA?j@-f2OI#1iXCvoZ?B7>6DsFtIt=bep@a9Fn3fn)z}aBQ6OR!l;U7NH`6XHW4IL^^l1~BA!fm4rA%<O|Bu{gr=KM'
            'Lyg{rJ^X<?{2*vH;$Ov{MVoeuk%lbs=@1FFN(T`$=<eR`lV|AKzF=R-H9We)P_0<!4j$g|4$fysos}J+D{Z{*g9gAj=~habnAtxV'
            'whh>UBwz0+u8_8bXrJ5;*cm^Zwj@K0d^~o*^`=u!@Qf#ixun#Y=`vMD@lxHNAS-Z2D$Ub{?`5|fF(RuO<DD6mEc8FXkwb?leA@|9'
            'h8&8d`u;L^xzfS|iad2W>KX(c7iFV#@BeS_TASN8vg~*L3IzEeG8QQ*Gx27mVWyZ79jhWsF3HZ8SIZ(n0-`v4E<nhpHT>^;U)|_N'
            'H>ii5nc1mLr4o^BH2U55aqhV<$%LVvH-@`sP#(Y!h>owOy$jVS_n4mJPFK1SR0%X|${oUf5I8jD4y)Q}{)z3|sE(1FULcRzScB_D'
            'F6ZJKP>QnN5{Z?ly!@~+!;yHnuhm$09U~vMI?F6{o47TP<Y8<ycM^fB{FlcFRHU-U<rOcoHFCu*rg=v69Wq)hOVfnoxhk@2-5knm'
            'bI=t1N+E2k9_ck=6$yTV+tn(k=C@R=RqV_n-=^^^+^M+c>8w5aV;4_QC4uShe>^(;UVcGw%YgZ)m)onhD*-MAgpUslg7;QRUp@L4'
            '=h|~0XbFxQ^!>rnA6|BgLy<t7a+7$)radK*8PD)6C{*EnX)#(&GuNbUeKr4{yI%1dgQBAtHhUYBBf}gl9P~q)2p$*s^wO=SwCAZ7'
            'Q$}yKyvMO+UN*}UQJ9j|)yqo}Xet3$%NYFqfb&#V>IXII#r{5CX?AsLOIXK7!NZ}xc6eme5V)roKCD}*Scv!xKqeE!$I-J*3Q^8V'
            'PHyA4;6I*oUKnQu#qP-42?r&ohp^ygV>3>-Dum}nfJP}MD&m_Dg!#K7=DCRoROpF16z}RA71g+$Vnzf8Qow_~)hesWM@`=$uH{>}'
            'IF(oVj|T?_t{wUcYs-R*^Ro}^p;4HDCW!xnOd;v~hJ^VQ3ZhYfnMtOJ(m@!+D*2gbGYZRsVUYxX2C9`zR4`rI=q&M`X7g26b(7{o'
            '4mu8k{d3Fco8$k%GF=@UOVcApY*q{Q40#s4LSd0UK1015nFuI=ze(d<OT*e<{@l85e)E@Z{lBvp!rM`|BKomyQfpn5T2&X(S-_Ij'
            'E#?i$g-fm<YM-)5OA(E)v5V?1sQ2ucz`Uc1!bp$IA2z|gdU6w%hr0A?^M-h^g$f5rC3jp%1gup)&)E%SE?Cg?dRhd%Ta1S~m#bs0'
            '-IljCnvZgf0~s30V0}j2Ep^P1)`m`?c${($Bb`<@bb*AAfoNpQB$_abqH^Zp2dXIMJ)y!Ik?<O=IJ6W=Y3|zt*jFkaCm|%PIYVQN'
            '$VirJohPS@#WJH@ZxUt7%{e{4Iz8#o9_{<!m3^INRP0+AQizG?g#vQ*5d?7%ow>F&X;6Pn)}RDYR17;-X9dIAB{3ccrm}`3qw=@K'
            'KqWG#?!h~VdNP`-+%2HRze1gXRqq7zQlh^Xq$Kpy=xNmEg#qRy#p`pPA&Vt8CZB_JlE~n{P8g-Hvbd6EP@i`4(jx`P*DRP6K8ko2'
            'J`N*cu_Tp)dq<Hb+&KvlDAG}<GOl)ZRHf0fQGvD`g8TimiWw@C>3BeBV`$AZxmjVXIwNQr4Nlm2OZEyC*~-LbLdmp8TJ9jZjgagv'
            '<(#K?oJ@2bkRVPr2OM)Yr9|%KFrG^|tsW|YU)@GA4-o2Q(_~gMOUS-gdYAfKw|;FdBRXwn6WEhA{*Wo!O>AFJSUCBy!6L@o*v2VS'
            'aj)RriiJH&hD2J1t5qgJu^x81BH5dpgv+CGl!Nk!UkZ9NjE?1Y9Hkb&v#*khEPa(xpzqpOzxFOK-oNWyT~lwmK96`g?2~G=lksiM'
            'dshS>k<crOr3xIP?+Vr-t@o*UpaWmIZ5<Ekmhj`75QA~%vt$iB-!RXEOoogsRPQ(G2ISSr=cZe8Jrt1rGEHvApK#51j(@dkP;A3U'
            'hmhnlU&+!k69F-l^K(fq6!Ec>A}3z<r$&27O-%%js>GoJt73pm>!DZ#+e=jym>{F_g7(N>8LEPj3im#>TJZl+A<X!S84s_+_WP65'
            'i?Hk47Rw|-3>Z}jW&LRZLhEwJ)op(EU+5t%0HbEuJqZS*ta{Eb`~}+B;Uj(6Y}{H)Uu}HhAch|mG-2ff-0Ed5<AaPskLL^`ha4U$'
            'sF8>VE?&$*cZzdEU)IY<L^<PKzQlXfz@29>1Sx}#yiQT^sn6F3-6wX<=oa)cbmMP|ns=TXapDjD!u4M-;X1v6C~{pqEfS@aeok|5'
            'e#0~nZuUpxjO~{5y4hunGuKnnVQ&34Xs;Ht@enQOm_o1dF~x}6Ia*`+uFvxHj$%`g79`4UDK~c%u<<)0HXW7_q&}AEL26`ylT)j+'
            '!S~}LpJGf)5)s;$>Jj-4r9th%&>my7!<C)LWeR4+(>Plyu|mrbME_2=^d~K5H2E{0>?mUtJdg5x69+&{do}%55l*tEK+9{oe4)tL'
            '(OYa_@!8#Nfu~}I(LlScU}>6*k5*Ya)VKitRbovw4_7ihn%`JuJr^H6QE?&aMW+`yVdxkaEg{6zNh7A5Hr$TOu`MI#KIIw9Z-c&P'
            '!28JT2lA@8_b6s4K<HdrSb>hU4^wF?-y&{3pjZ&?#VdRj6hc>U_plzMOjb>+8;)75W{@!wx>_}Y@GbY7x_oOsXOGNM*kuZ`a&5BJ'
            't3;y3drSo6;H(S5so?8<d)fKzwDX_GET9HNz6&QWqWBpeT_UxyVA0BMrTccR%y8Z9$!yChW~vE?&AQdshI6gWjz3mVs8Vw;l0T57'
            '|Ah4=4z?x|TMP`Mv&pD!1|nNDL3g0|scvAvvRl;Nak33>oH|%?gu_f~Gfw1d>nD#n%S)%TYf14~{l3(Q@@3|e2Mj0L2u22V8aC57'
            '>8<iP#uj5SYse;JVo*(8e4KkRo5}!_rxTc0)O$r%2x@(6SX#RT2Hfq}Rk$wfOwemZhf(T(VSXsoh~QGejF=I5nIK*03N0$^?Z;b|'
            'R;?IH=2if33<|CC&fvS_WT+f30ed(27A{QemYY?@|Fqw}y}bCXeRian#>y~8Y+k>#x%(QiKvoneyrVS-_F&6#fyfHH6Y=nbjRFP6'
            'H*1Ahkb$489>J4xFfngfUvoN<grG?-`e;<*AlR&cOrWe$SM~_ai9jQUQ?buDYD7Y_A!0m-Jw9gNzRpcRMbX%?TQo*&CN^c1GEqec'
            'KgnQ=<#WY{5%g_Njy?m7g5@;NRAkeRRt%`!{{SP19Kl<x>;Fuv2e<0~sM$jd7XmeEJ?ND29)5K(JYe)!?-}-oZ3Ruc!vnk*Ml<XG'
            'I`)|1V#*9$U%^_Us&dRFR&)(drjy$HIrfwXY%1o2zYw#r4|wdiuRPDj@&r4}b8RiZkG;iXbMbs-JIKDo{_+?b%vZ9**o*TC#uyYx'
            'rx5|#tXR5CvS6v;nJ+QgsCFr<%^(zawf?34m900PH|%^J-jIJ|gUvH+BZWPr+O+ZiRtwi^At9CumBk<HKEsAwsTHOT>%#pq3&%go'
            '0>FuPsx|-M3KpOdWQ{C$obqCvX34%vfr?D8{8b>CvxxCkCqf-1Lu9r14D)~~PmK!KNt{gZ1QDU$cpMY0|906vd(-}}e;EseFDRTT'
            'un+;EIfU_p&DHIy^xx?(O>XQ~;7|%zUdkQ<MdnR&wP{Uzs<ooV5Tq68A}Y9A6=DmB84Qyv{kc8;h!`PPeXPkE<>f`Ttv-)*f4}dj'
            '5&goC9&wGNz(+{yc=h^W)l1_Ps+B?(ZdoIY3Vl#(ptgdBi6)Ba+CtZCPz?Ji+6%UOT^$p&xAaw{7zSIRBWHR9zt;6GiG91>y0Hi>'
            'lF*i^i|lym*jwe^`A#E4gW}G&yv)`xTFekuIbtR+b-7O2VXYFiw6WaeT;ioPKE`taPGrp1oX8myBdY`ATH5FE-@kUwI&V5xr@_yi'
            'U(ZfITwIpZCWjZAdHQ+m(q^yyo%Yc9`ZlKLyS*Kkz*=Kncl>_$avO{vP4ZM0!N;-Z3BGl00$;vap|CTt_B~AV>oXhH1y&a}LX7+H'
            'RIc=m2VP)2>ATk5=Y6FtMrGg4?N;M0X4La7M*gCnZ^WN);s!JUtcJH_30q}NJ{#QgRxdNNPB_bumIQvyVQpu-!ER9=jY&PKI?1G6'
            'yTJ(Yx*Yrft+<7;OH5<7b$;urnpLBLukcN}e=x+xbbWOAva95%>Va04B<r(9{0yTLCwDzfhplr?m4YA2aSFAXe-ju&XUW6z2)~QT'
            'hP}vEy*!J1N(m_=Vu})s=5k^u!vD$f2w(Szv#)p9`@t>ZLC!qfEwc?Qw7O3z!VrsT-IOBBNq!v(($kVs37hGfuG*}pZrXyu*qX}C'
            'RzBdMGaDpvOnD^4xR0?%&!V<f2UO?H&z+N#)AL_?C#P>Z=U1l}=Wu!d$IBl*CEi?|bj|{1NrlbRNK|tg--kOGk?(#_4)XaT=;Iwv'
            'LErsjMoq%!V6;edfD&7VIQ`8|@lG;i!j*@yn^&6bIaK+Udf9k3o<=Dr4G1OGi%JAn{22T_V6jrMCk&@~wnAQ`QSn^yol~4)k2t%r'
            'unw57Bc?IYwErHc627O2Y@_;7`k9{FxcFf`tekr&)5&eboJNUEwS?>&p>2s?;O0kYD0;2fEwyEZY|lr7+Zk_i62eNEW}fBKDmotA'
            'jU!qu)Tjp0JDk4@RAvc|6<1<oLvgZ;bPkKU=w)U(xgtGj#2tr7erA3{pHjUU^xc)+7oWyz(a>4fGwb40Zz6FV_5jRBPWTRs-;hE$'
            'mmnA<@v(-^I(`>AmG+8CdEymtkK$xOU7-;Vu3onra2W{+RrHaiVVyCHCMag;E=4``OH)Q(lv08hipW3%G_Aa@C!JL$8S_XABp(EQ'
            'YbV>nTDJIIii!UH+%KcWEC;y*6lzS*Iwz8by-Ej5mCt0m!^P-Fx4!4~v$z#M!tY{XH7m|Yx{S)F0uh#~$I*9&u|wC|3Xi5LoSQml'
            'QAPfDPUtnup6qb&vO&{;JRN>XE%hTWm4>EwL?nr0;25asFY6XU@fikT<LGHsFh3cS?Eo2z-=1JDwgxNENqBum^g9rs7im_oRVl{~'
            'Bkf@lt6IR(VnwIu0q6~X@2833-=63V&<%jbPN1=Q$^LD`5pn%>G$h-z__SAgFDE0x>0q&j<M3WrzmY7b!M)LljT^jXw^@c>c`mLw'
            'Wblr+5CyYwP2GAqTuPp#eiDg>Ec7S16n5P1-CI>{g3&mQ#{ry<s$cTfjm!ftr?A42q*6-qL(la@Y)|2p<+GaKE4$)Rlx*43rAPTQ'
            'tQ+J>ZVd~jrj`a7I_<S+6)c~#RKEOW)e}^h|FnwmP@vu^0oB)5Mx3g1+O2{jZ?wstiWy0+Z86O;_|n(>etNRfv<r$JvP%_WW(~uX'
            'NwA|mab!vWk@l$JAK0u&?}Nd%OlzU@njl((0-t}z?7EXxL<i+!I*Z`2lqEh*(tvh-D(6kSptwO;cb|gqf;Eler@ep<%Fp8^H=K*2'
            'jMj^c(x13AktEe&`M{0*Ls4=dCD*!Ms&rtLq%q)XTfqU31V0NOof{Nxd@C0ww0$21T&mzZc~sF$%@=pcR4LO1by$x#@Se8t_VnRn'
            'PY=j>hd~2@8Zw`z?i7t86!ml^2>#3mQ$hgl7daZ`#u91!uz7&)rNxS@K2jTVk}^9DrMbco<~hg725}BjFbdDj;3R>qi4r$*Ra14N'
            '@tshpQVtn)`uZ|qm;36B?4gY8fODWCX^Fdt-7zGF8s2*Jr<_jqxLVtUV!nwJrFlTqBDF*O1Ba$TZ?A)N_)^Uw=7HCVgnN24vMtkz'
            'OznurT1-Llqv?eH95ny*y&LR&qUysKZ*lZR!*r2h3>GqikY!1c?y6`z^&&T6IcC5ZpiaVVsy8*?EwOmAO^*1H(jXeZ^5hD2d;2JF'
            ';d^CeP^hFmpQ;jzi_?0mIR(Wu(<VUF$|ELKT5nMZ(BnU~jIp&PR&JK&Y7MDL1QVgAg_cc%7Xj6Yb6w_BO=0KMq()>pUAH9p!0ae!'
            '%&yJ4x@w__ouW5bn|-6VPtlypOR-{Y{P=^rHhxKyBnIg(Hn*Ly`7>j=VH&A%NN58AzQ#KmNVpdlSsUaDMXHKAe)oZyEcc)oc8W8I'
            'l@-B;F!6f`#H3zX7B1a|pr&6QxekXY058TdCD5pxdZT04;S7b16s}0M5~%$y#h?&+klWmOwq9L-(Igx<uPjv?A{zz0pb;D*L=FVd'
            'eSRYij_@kr<>sl^UH<ByuJeta0*etv6UBjoJt=EH{75MxU*@RTJ!&XGEv5lP45DtLXxGwAd-GhI&kl_--EiN{CDE0wqfgZyygh;&'
            'vnDsEY!SvP%5Ubj`NI#MEAuUvQ}&yH!OWO5p#i57E_pm<`8>fU&66l?=qBh3Q*9Y(id;$dox$pbl&IG|_e_ni6;%WlhS3$dX4s6}'
            'h6`qtMt69(iyiq@{fL6Y(=lPfVOw6jdS#l(suQH4g)%B?)a3p>vpy&L12hZM3_d;)wet`5Znc2AMQdQY<)iave-*U7*+pwub^Zxv'
            'U6bqdM3O7o0Q<}L=e_p%yVGAT+rOQ@`_Q}WTy@%)e|@b+rO)X+C};zT3}L`kyg6gobyz#=M4ap^*CFU4t6AT%Tg(Qai~GqeM!BUU'
            '6a(zM#eHp|WCXh8s#tN3)F8@s3ie9WT^oX}C-43g1iUs1eYg*x{(gBT`@tZOM+sUhsU95b_PdMA55jgP_yrw_1pA1>R#5PWqxEhk'
            '2sjz_(^Y0*`vp<aH>a)-2d4wt@XD5I**Z2%+oP@wn!bHeZQ4|>Cr>WQ!s=<ZJM@++jGeWxYHcwF2E(t?ELy{@IIko5&pcUY$iALN'
            '2w<8;!9!WNL(s7W?1a$uz2J5{9E?*2wHJ`~W|>z0%K+`(VJ#!vze-SlSYDe@sQi^D^(Q4cC>0Ax$XV6O-xI6P!K0ImKkB$;yH7M6'
            'PXe3`rzwMcHB2%m)A;8_irRFXr<f@%12F<b{XxFYC%mJARX)MI$2hZDG^sl&#%15O{2kSaQMaa`{HZ0?{1Z2spmQGDUN@ogT1K}7'
            'C*V=S9R-71?tkl^`?Yy^SIPW9jZrpUFVf?G>kY-jVs4|a;d6L4>jNbu@;e=zgqeIC)DHL5(r;{E-5!U;R}0fuuXu!=)W9|PHxHRk'
            'cMNBNHlaQ`@1<j7wekiP`I4-~&<)h?x1q&al2F4hk_0_aD1|;)daDYSZo*!zTlEp(*V3VU(mC&Zxayn*ktqX4Ao}IUE?)ROqV#4c'
            'LcvTUwpiuG-YXx-gb;?~IG=G&Ck7VJlF$00&5|{o3hVJ4>GwW62XZwQ*PM6_&ouG4B?0*H&`ulHS-HTr;5@CXEE=H?KXj0r3f<|#'
            'HZe_tLTFZqOE%c@LBOJoKfHEupBCu6C{`AmORKdg=-mfqtDFMm7Cm-{{QjxNU!_UXz?}5V9?NK`V3bE`tdflXss7bdbyM<2L_54e'
            '{OJ27sLS!s2!>AL+GRdRI&zYx<kmniyle-A#iy%Yfjy<nfbbaozxbY2RWy?6^p+ZIGl|yoXu?TmL)th+hAz+`E|Z&_NSqf)S{PHp'
            'h2dhkPKmWG4yb<om7;rz-yO#fWHBU&#xLRVlr=ulnvhj;xuOtiQFG3a;W>EqifG>VgPRBqAB~T<;|vMc_Jg<UE6kO9Le^jSJRJ|x'
            'MTRl|nD?omtK)?=kEZL4&?2z^ju|adorrgj`QBI2Y>7A!e9t~1^LDg7R@fhPrWBn;Y_!0hGal2cbCu_zBq{sne8={VZp-=0y3kQ@'
            'Z_Ml_$cM01KmraX(P{?6Q9hbKy#MvAbB+n|C})GNtp}=Vy9e4Zrq+1@zkSFj7->HV@azUzF2m$h(K?Mrj{J=R`!E8z9r4?vBuysM'
            'ac~<=rl634RCzmI28(H&kMJ^BC2LTHla+Pki$z8+Ww>|e<N2sYOA^-Vk$DCfQo#EU?aR}nt(lPt71Rt~M>AgL`?#P5MXSfk2wLG`'
            '8_#iq5xqMm(9J`G^>YrY3lxU2;m2^`j9N*r**Y>Y`jenl@L5O1{4WWOPcQ'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
script_generation = _local_pipeline_modules['script_generation']
video_analysis = _local_pipeline_modules['video_analysis']
approval = _local_pipeline_modules['approval']
google_ai = _local_pipeline_modules['google_ai']



## `local_pipeline` kanonik modüllerini yükle

In [ ]:
HOT_MOMENTS_PATH = ''  # elle yol vermek istersen buraya yaz

CANDIDATE_HOT_MOMENTS_PATHS = [
    HOT_MOMENTS_PATH,
    str(RESULTS_DIR / 'hot_moments.json'),
    'results/hot_moments.json',
    '../results/hot_moments.json',
    '/content/results/hot_moments.json',
]

def find_hot_moments_path(candidates):
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate).resolve()
    return None

hot_moments_path = find_hot_moments_path(CANDIDATE_HOT_MOMENTS_PATHS)
if hot_moments_path is None:
    raise FileNotFoundError(
        'hot_moments.json bulunamadi. Aranan yollar: '
        + ', '.join(p for p in CANDIDATE_HOT_MOMENTS_PATHS if p)
        + ". Once Notebook 01'i calistir veya HOT_MOMENTS_PATH degiskenini elle ayarla."
    )

with open(hot_moments_path, 'r', encoding='utf-8') as f:
    hot_moments_data = json.load(f)

# Kanonik sema dogrulamasi (Notebook 06 ve master notebook ile ayni fonksiyon).
schema_errors = video_analysis.validate_hot_moments_schema(hot_moments_data)
if schema_errors:
    raise ValueError(
        f'hot_moments.json semasi gecersiz ({hot_moments_path}):\n- ' + '\n- '.join(schema_errors)
    )

print('Yuklendi:', hot_moments_path)
print('Uretim yontemi:', hot_moments_data.get('method'))
print('Kaynak durum:', hot_moments_data.get('status'), '-', hot_moments_data.get('status_reason', ''))
print(f"{len(hot_moments_data['hot_moments'])} aday an bulundu.")
if hot_moments_data.get('transcript_available') is False:
    print("UYARI: Notebook 01 bos transcript bildirdi (konusma tespit edilemedi); "
          "uretilecek prompt'ta diyalog metni olmayacak.")


## 1. Notebook 01 çıktısını yükle: `hot_moments.json`

Aşağıdaki sırayla aranır: `HOT_MOMENTS_PATH` (elle ayarlanmışsa) → `results/hot_moments.json` → `../results/hot_moments.json` → `/content/results/hot_moments.json`. Bulunamazsa veya şema geçersizse notebook **açık bir hata** ile durur (sessizce örnek veriye düşmez).

In [ ]:
HOT_MOMENTS_PATH = ''  # elle yol vermek istersen buraya yaz

CANDIDATE_HOT_MOMENTS_PATHS = [
    HOT_MOMENTS_PATH,
    str(RESULTS_DIR / 'hot_moments.json'),
    'results/hot_moments.json',
    '../results/hot_moments.json',
    '/content/results/hot_moments.json',
]

def find_hot_moments_path(candidates):
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate).resolve()
    return None

hot_moments_path = find_hot_moments_path(CANDIDATE_HOT_MOMENTS_PATHS)
if hot_moments_path is None:
    raise FileNotFoundError(
        'hot_moments.json bulunamadi. Aranan yollar: '
        + ', '.join(p for p in CANDIDATE_HOT_MOMENTS_PATHS if p)
        + ". Once Notebook 01'i calistir veya HOT_MOMENTS_PATH degiskenini elle ayarla."
    )

with open(hot_moments_path, 'r', encoding='utf-8') as f:
    hot_moments_data = json.load(f)

# Kanonik sema dogrulamasi (Notebook 06 ve master notebook ile ayni fonksiyon).
schema_errors = video_analysis.validate_hot_moments_schema(hot_moments_data)
if schema_errors:
    raise ValueError(
        f'hot_moments.json semasi gecersiz ({hot_moments_path}):\n- ' + '\n- '.join(schema_errors)
    )

print('Yuklendi:', hot_moments_path)
print('Uretim yontemi:', hot_moments_data.get('method'))
print('Kaynak durum:', hot_moments_data.get('status'), '-', hot_moments_data.get('status_reason', ''))
print(f"{len(hot_moments_data['hot_moments'])} aday an bulundu.")
if hot_moments_data.get('transcript_available') is False:
    print("UYARI: Notebook 01 bos transcript bildirdi (konusma tespit edilemedi); "
          "uretilecek prompt'ta diyalog metni olmayacak.")


## 2. Aday anları görüntüle

In [ ]:
moments_view = pd.DataFrame([
    {
        'rank': m.get('rank'),
        'source_start_sec': m.get('source_start_sec', m.get('aligned_start_sec')),
        'source_end_sec': m.get('source_end_sec', m.get('aligned_end_sec')),
        'score': m.get('score'),
        'reason': m.get('reason'),
        'transcript': (m.get('transcript') or '')[:200],
        'preview_path': m.get('preview_path'),
        'review_status': hot_moments_data.get('status', 'review_required'),
    }
    for m in hot_moments_data['hot_moments']
]).sort_values('rank')
display(moments_view)
print('Kalite politikasi hatirlatma: kaynak durum ->', hot_moments_data.get('status'))
print('Bu durum insan veya LLM onayi olmadan otomatik olarak \'approved\' sayilmaz.')

## 3. An seçimi

`SELECTED_MOMENT_RANK = None` bırakılırsa en yüksek skorlu (rank=1) an otomatik seçilir. İstersen belirli bir rank ver.

In [ ]:
SELECTED_MOMENT_RANK = None  # None = en yuksek skorlu an; istersen ornek: 2

moments_by_rank = {m['rank']: m for m in hot_moments_data['hot_moments']}
if SELECTED_MOMENT_RANK is None:
    SELECTED_MOMENT = min(hot_moments_data['hot_moments'], key=lambda m: m['rank'])
else:
    if SELECTED_MOMENT_RANK not in moments_by_rank:
        raise ValueError(f'SELECTED_MOMENT_RANK={SELECTED_MOMENT_RANK} hot_moments icinde bulunamadi. Mevcut ranklar: {sorted(moments_by_rank)}')
    SELECTED_MOMENT = moments_by_rank[SELECTED_MOMENT_RANK]

print('Secilen an (rank', SELECTED_MOMENT['rank'], '):')
print(' Zaman araligi:', SELECTED_MOMENT.get('source_start_sec', SELECTED_MOMENT.get('aligned_start_sec')), '-', SELECTED_MOMENT.get('source_end_sec', SELECTED_MOMENT.get('aligned_end_sec')), 'sn')
print(' Skor:', SELECTED_MOMENT.get('score'))
print(' Gerekce:', SELECTED_MOMENT.get('reason'))
print(' Transcript:', (SELECTED_MOMENT.get('transcript') or '')[:300])

## 4. İnsan onay kapısı (elle doldurulacak)

**Kalite politikası:** `hot_moments.json` durumu (`review_required`) tek başına yeterli değildir; script üretimi API çağrıları yalnızca `MOMENT_APPROVED = True` olduğunda çalışır. Bu hücreyi izleme/önizleme klibini kontrol ettikten sonra düzenle.

In [ ]:
CONTENT_MODE = 'spoiler-free-recap'  # 'spoiler-free-recap', 'dramatic-retelling', 'comedic-commentary'
TONE = 'gergin ve merak uyandirici'
TARGET_DURATION_SEC = 60

# Kanonik prompt olusturma (master notebook ile ayni fonksiyon).
USER_PROMPT_TEXT = script_generation.build_user_prompt(
    SELECTED_MOMENT, content_mode=CONTENT_MODE, tone=TONE, target_duration_sec=TARGET_DURATION_SEC,
)
print(USER_PROMPT_TEXT)


## 5. Kullanıcı promptunu oluştur

In [ ]:
scripts = {}
script_meta = {}
attempts = []  # her zaman tanimli olsun (disa aktarim hucresi bunu okur)

if not MOMENT_APPROVED:
    print('MOMENT_APPROVED=False oldugu icin API tabanli script uretimi ATLANDI. '
          'Asagidaki disa aktarim hucresi yine de prompt paketini kaydedecek.')
elif not (OPENAI_API_KEY or ANTHROPIC_API_KEY or GOOGLE_API_KEY):
    print('Hicbir API anahtari yok. Sahte/ornek script uretilmeyecek; sadece prompt paketi disa aktarilacak.')
else:
    # Bu notebook'un katkisi: AYNI kanonik call_*_script() fonksiyonlarini her
    # saglayicinin BIRDEN FAZLA modeliyle calistirip karsilastirmak.
    attempts = []
    if OPENAI_API_KEY:
        from openai import OpenAI
        openai_client = OpenAI(api_key=OPENAI_API_KEY)
        for model_name, price_in, price_out in script_generation.OPENAI_SCRIPT_MODELS:
            attempts.append(script_generation.call_openai_script(
                openai_client, model_name, price_in, price_out, USER_PROMPT_TEXT))
    else:
        print('OpenAI key yok, atlandi.')

    if ANTHROPIC_API_KEY:
        import anthropic
        anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        for model_name, price_in, price_out in script_generation.ANTHROPIC_SCRIPT_MODELS:
            attempts.append(script_generation.call_anthropic_script(
                anthropic_client, model_name, price_in, price_out, USER_PROMPT_TEXT))
    else:
        print('Anthropic key yok, atlandi.')

    if GOOGLE_API_KEY:
        # Karsilastirma hucresi bile kanonik router'i kullanir: istemci TEK bir
        # kimlik bilgisinden kurulur ve call_google_script() her modeli
        # local_pipeline/google_ai.py uzerinden cagirir (kota defteri, hata
        # siniflandirma ve sanitize edilmis hatalar dahil).
        google_client = google_ai.build_google_client(
            google_ai.resolve_secret('google_api', get_secret))
        for model_name, price_in, price_out in script_generation.GOOGLE_SCRIPT_MODELS:
            attempts.append(script_generation.call_google_script(
                google_client, model_name, price_in, price_out, USER_PROMPT_TEXT))
    else:
        print('Google key yok, atlandi.')

    for attempt in attempts:
        name = attempt['candidate']
        if 'script' in attempt:
            scripts[name] = attempt['script']
            script_meta[name] = {
                'provider': attempt['provider'], 'model': attempt['model'],
                'cost_usd': attempt['cost_usd'], 'latency_sec': attempt['latency_sec'],
            }
            log_result('script-generation', name, None, attempt['cost_usd'], attempt['latency_sec'],
                       f"{len(attempt['script'])} karakter", status='review_required')
        else:
            log_result('script-generation', name, None, 0.0, attempt.get('latency_sec', 0.0),
                       attempt['error'], status='failed')

print(f'{len(scripts)} script uretildi.')


## 6. Aday LLM'ler — Script Üretimi (sadece an onaylandıysa çalışır)

Yapılandırılmış JSON çıktı (OpenAI için `response_format=json_object`) ve sağlam JSON ayrıştırma kullanılır; ayrıştırma başarısız olursa aday atlanır ve neden loglanır (sessizce yanlış veri üretilmez).

In [ ]:
scripts = {}
script_meta = {}

if not MOMENT_APPROVED:
    print('MOMENT_APPROVED=False oldugu icin API tabanli script uretimi ATLANDI. '
          'Asagidaki disa aktarim hucresi yine de prompt paketini kaydedecek.')
elif not (OPENAI_API_KEY or ANTHROPIC_API_KEY or GOOGLE_API_KEY):
    print('Hicbir API anahtari yok. Sahte/ornek script uretilmeyecek; sadece prompt paketi disa aktarilacak.')
else:
    # Bu notebook'un katkisi: AYNI kanonik call_*_script() fonksiyonlarini her
    # saglayicinin BIRDEN FAZLA modeliyle calistirip karsilastirmak.
    attempts = []
    if OPENAI_API_KEY:
        from openai import OpenAI
        openai_client = OpenAI(api_key=OPENAI_API_KEY)
        for model_name, price_in, price_out in script_generation.OPENAI_SCRIPT_MODELS:
            attempts.append(script_generation.call_openai_script(
                openai_client, model_name, price_in, price_out, USER_PROMPT_TEXT))
    else:
        print('OpenAI key yok, atlandi.')

    if ANTHROPIC_API_KEY:
        import anthropic
        anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        for model_name, price_in, price_out in script_generation.ANTHROPIC_SCRIPT_MODELS:
            attempts.append(script_generation.call_anthropic_script(
                anthropic_client, model_name, price_in, price_out, USER_PROMPT_TEXT))
    else:
        print('Anthropic key yok, atlandi.')

    if GOOGLE_API_KEY:
        # Karsilastirma hucresi bile kanonik router'i kullanir: istemci TEK bir
        # kimlik bilgisinden kurulur ve call_google_script() her modeli
        # local_pipeline/google_ai.py uzerinden cagirir (kota defteri, hata
        # siniflandirma ve sanitize edilmis hatalar dahil).
        google_client = google_ai.build_google_client(
            google_ai.resolve_secret('google_api', get_secret))
        for model_name, price_in, price_out in script_generation.GOOGLE_SCRIPT_MODELS:
            attempts.append(script_generation.call_google_script(
                google_client, model_name, price_in, price_out, USER_PROMPT_TEXT))
    else:
        print('Google key yok, atlandi.')

    for attempt in attempts:
        name = attempt['candidate']
        if 'script' in attempt:
            scripts[name] = attempt['script']
            script_meta[name] = {
                'provider': attempt['provider'], 'model': attempt['model'],
                'cost_usd': attempt['cost_usd'], 'latency_sec': attempt['latency_sec'],
            }
            log_result('script-generation', name, None, attempt['cost_usd'], attempt['latency_sec'],
                       f"{len(attempt['script'])} karakter", status='review_required')
        else:
            log_result('script-generation', name, None, 0.0, attempt.get('latency_sec', 0.0),
                       attempt['error'], status='failed')

print(f'{len(scripts)} script uretildi.')


## 8. Final script seçimi

`SELECTED_SCRIPT_CANDIDATE = None` bırakılırsa en yüksek `quality_score_1_5` puanlı aday (yoksa ilk
üretilen) otomatik seçilir.

**Onay burada verilmez.** Dışa aktarım her zaman `review_required` yazar; onay, bölüm 10'daki ayrı
onay hücresinde, scripti okuyup SHA-256'sını doğruladıktan sonra verilir.

In [ ]:
SELECTED_SCRIPT_CANDIDATE = None  # None = otomatik en iyi aday

if scripts:
    if SELECTED_SCRIPT_CANDIDATE is None:
        scored = [r for r in results if r['task'] == 'script-generation'
                  and r['candidate'] in scripts and r['quality_score_1_5'] is not None]
        if scored:
            SELECTED_SCRIPT_CANDIDATE = max(scored, key=lambda r: r['quality_score_1_5'])['candidate']
        else:
            SELECTED_SCRIPT_CANDIDATE = next(iter(scripts))
    if SELECTED_SCRIPT_CANDIDATE not in scripts:
        raise ValueError(
            f"SELECTED_SCRIPT_CANDIDATE='{SELECTED_SCRIPT_CANDIDATE}' scripts icinde yok. "
            f'Mevcutlar: {list(scripts)}'
        )
    print('Secilen script adayi:', SELECTED_SCRIPT_CANDIDATE)
    print()
    print(scripts[SELECTED_SCRIPT_CANDIDATE])
else:
    print('Uretilmis script yok (an onaylanmadi ya da API anahtari yok). '
          'Sadece prompt paketi disa aktarilacak.')


## 8. Final script seçimi ve onayı

`SELECTED_SCRIPT_CANDIDATE = None` bırakılırsa en yüksek `quality_score_1_5` puanlı aday (yoksa ilk üretilen) otomatik seçilir. **Bu script'i yayına/TTS'e vermeden önce `SCRIPT_APPROVED = True` yapman gerekir.**

In [ ]:
SCRIPTS_DIR = RESULTS_DIR / 'scripts'
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
for name, text in scripts.items():
    (SCRIPTS_DIR / f'{name}.txt').write_text(text, encoding='utf-8')

df = pd.DataFrame(results)
out_path = RESULTS_DIR / '02_script_generation_sonuclar.csv'
if not df.empty:
    display(df)
    df.to_csv(out_path, index=False)
else:
    pd.DataFrame(columns=['task', 'candidate', 'quality_score_1_5', 'cost_usd_estimate',
                           'latency_sec', 'notes', 'status']).to_csv(out_path, index=False)
print('Kaydedildi:', out_path)

# Kanonik artefakt olusturma (master notebook ile ayni fonksiyon). Bu fonksiyon
# onay argumani ALMAZ ve asla 'approved' uretemez -- yeni uretilen script her
# zaman 'review_required' olur. Onay bolum 10'daki ayri hucrede verilir.
benchmark_attempts = [
    {'provider': a.get('provider'), 'model': a.get('model'),
     'ok': 'script' in a, 'error': a.get('error')}
    for a in attempts
]

if scripts and SELECTED_SCRIPT_CANDIDATE:
    meta = script_meta[SELECTED_SCRIPT_CANDIDATE]
    quality_score = next((r['quality_score_1_5'] for r in results
                          if r['candidate'] == SELECTED_SCRIPT_CANDIDATE), None)
    selected_script = script_generation.build_selected_script_artifact(
        scripts[SELECTED_SCRIPT_CANDIDATE], meta['provider'], meta['model'],
        meta['cost_usd'], meta['latency_sec'], SELECTED_MOMENT, moment_review_record,
        hot_moments_data.get('source') or {}, hot_moments_path,
        quality_score_1_5=quality_score, attempts=benchmark_attempts,
    )
else:
    selected_script = script_generation.build_selected_script_artifact(
        None, None, None, 0.0, 0.0, SELECTED_MOMENT, moment_review_record,
        hot_moments_data.get('source') or {}, hot_moments_path,
        prompt_package={'user_prompt': USER_PROMPT_TEXT, 'content_mode': CONTENT_MODE,
                         'tone': TONE, 'target_duration_sec': TARGET_DURATION_SEC},
        status_reason_override=(
            'moment not approved (MOMENT_APPROVED=False)' if not MOMENT_APPROVED
            else 'no API keys available or all attempts failed; prompt package exported'
        ),
        attempts=benchmark_attempts,
    )

selected_script_path = RESULTS_DIR / 'selected_script.json'
with open(selected_script_path, 'w', encoding='utf-8') as f:
    json.dump(selected_script, f, ensure_ascii=False, indent=2)
print('Kaydedildi:', selected_script_path, '(status:', selected_script['status'], ')')

zip_path = RESULTS_DIR / '02_script_generation_paylasim_paketi.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(selected_script_path, arcname='selected_script.json')
    if out_path.exists():
        archive.write(out_path, arcname=out_path.name)
    for script_file in SCRIPTS_DIR.glob('*.txt'):
        archive.write(script_file, arcname=f'scripts/{script_file.name}')
print('Paylasim paketi hazir:', zip_path)


## 10. İnsan onayı (ayrı adım, üretimden sonra)

Yukarıdaki scripti **okuduktan sonra** inceleme çıktısının yazdırdığı SHA-256'yı
`SCRIPT_REVIEWED_SHA256` alanına yapıştır. Hash eşleşmezse onay reddedilir; script sonradan
değişirse onay otomatik geçersizleşir.

In [ ]:
if selected_script.get('script_text'):
    script_review_info = approval.describe_for_review('script', selected_script)
    print('Inceledigin scriptin SHA-256 degeri (asagiya yapistir):')
    print('   ', script_review_info['content_sha256'])
else:
    print('Onaylanacak bir script yok (sadece prompt paketi).')

SCRIPT_REVIEWED_SHA256 = ''   # yukaridaki SHA-256'yi buraya yapistir
SCRIPT_APPROVED_BY = ''       # ornek: 'insan: birkan'
SCRIPT_APPROVAL_NOTES = ''

if not selected_script.get('script_text'):
    pass
elif not SCRIPT_REVIEWED_SHA256 or not SCRIPT_APPROVED_BY:
    print('Onay ATLANDI: SCRIPT_REVIEWED_SHA256 ve SCRIPT_APPROVED_BY doldurulmali.')
    print('status olarak kaliyor:', selected_script['status'])
else:
    try:
        selected_script = approval.approve_artifact(
            selected_script, 'script', SCRIPT_REVIEWED_SHA256,
            SCRIPT_APPROVED_BY, SCRIPT_APPROVAL_NOTES,
        )
        with open(selected_script_path, 'w', encoding='utf-8') as f:
            json.dump(selected_script, f, ensure_ascii=False, indent=2)
        print('Script ONAYLANDI ->', selected_script_path, '(status:', selected_script['status'], ')')
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('status olarak kaliyor:', selected_script['status'])


## 9. Dışa aktarım